# Model B V3 — Directional, Gradient-Enhanced Endpoint Network

**Objective.** Upgrade the V2 TCN–Transformer so it learns relationships between market conditions and the future ZigZag pivot endpoint `tp_pct`, instead of predicting nearly the same ATR multiple for every candle.

> **Status (built 1 Nov 2026): not yet trained.** This notebook implements the V3 plan. It contains no V3 performance result. V2 scored **14.10%** strict accuracy (1,428 / 10,130) against **14.24%** for a fixed 2.1246 × ATR rule on 1 Jul–14 Oct 2026, and its predicted Z spanned only p1 2.14 – p99 2.36 (verdict `effectively_constant`). Fixed ATR remains the benchmark to beat.

**Findings so far (Nov 2026).** The V3 network does not beat a fixed ATR multiple: at the 98% safe-TP goal (window dev_1) it gave a mean TP of 0.096% vs 0.095% for fixed ATR and 0.145% for the gradient-boosted quantile model. The causal ZigZag live side agrees with the label direction on only 49% of candles. The limit is information at candle close, not network capacity, so the expensive V3 selection and refit stages are now **off by default**. See `README.md`.

**Unchanged contract.** Original future-ZigZag endpoint `tp_pct`; ATR-normalized target \(Z = tp\_pct / (100\,ATR_{14}/C)\); strict hit \(0.75y \le \hat y \le y\) (zero target accepts only zero); one prediction for every eligible candle, no confidence filtering. Acceptance: **≥ 80% strict accuracy** at 100% coverage and **≥ 98% execution TP reach**, measured separately. These are acceptance requirements, not outcomes that architecture changes can guarantee.

| Stage | What this notebook does | Acceptance evidence it writes |
|---|---|---|
| **V3.0** Diagnose | A: prediction collapse on seen vs unseen data, ordering, accuracy by direction / magnitude / leg age, leg dependence. B: per-component gradient flow before clipping. C: 64–128-anchor memorization test. | `v3_0_diagnostics/` |
| **V3.1** Inputs | Live side (causal ZigZag, never hindsight) + 16 causal leg-state features; FiLM \(h' = (1+\gamma)\odot h + \beta\) and gated late fusion, both identity at initialization. | prefix-replay and future-perturbation audits; conditioning on/off stage |
| **V3.2** Learning | Auxiliary head on \(\log(1+Z)\) with Huber loss, \(\lambda \in \{0, 0.03, 0.1, 0.3\}\); per-module AdamW learning rates; gradient and loss-contribution logging. | `gradients_epoch_*.csv`, `loss_gradient_probe_*.csv` |
| **V3.3** Head | V2 interval head vs a coherent 96-bin log-Z distribution with a separate zero probability and an exact interval-mass decoder. | head stage in `selection_table.csv` |
| **V3.4** Overfitting | Leg-capped sampler vs uniform; dropout 0.20 / 0.15, weight decay 0.005, 40 epochs, patience 5 on validation loss; three chronological development windows; regularization stage only if a generalization gap appears. | `v3_frozen_plan.json` |
| **V3.5** Validate | Refit, full-coverage evaluation vs fixed ATR, fixed TP, causal-pivot rule and frozen V2 on identical candles; week- and leg-block bootstrap; subgroup reports; side ablation; separate execution-reach simulator; saved-model replay. | `main/`, `reach/`, `milestone_gates.csv` |

Run order: V3.0 → V3.1 → V3.2 → V3.3 → V3.4 → V3.5. The selection stages are attributable: each changes one factor and keeps the incumbent unless an option wins under a rule declared before training.

**Safeguard.** If the network starts producing visibly different predictions but still cannot beat fixed ATR on unseen windows, investigate the information available at prediction time — not more network complexity. The gate report at the end prints this automatically.

## Run in Google Colab

1. Select **Runtime → Change runtime type → GPU**. Run every cell in order; switches in the configuration cell decide what executes.
2. **First session — V3.0 diagnostics (done 1 Nov 2026).** Their results are in `v3_0_diagnostics/`, and the switches now default to off.
2b. **Information tests (now on by default).** These answer whether the 80% target is reachable with the information available at candle close. Diagnostic A showed V2 is constant even on training data, and the live side agrees with the label direction on only 49% of candles. The direction study and gradient-boosted test run on CPU in minutes; the V3 screen trains two short GPU runs on one window. Read the summary they print before enabling selection.
2c. **Cross-asset study (on by default).** This downloads Binance 15-minute klines for ETH, SOL, BNB, XRP and DOGE (cached on Drive) and adds them as BTC inputs. It then scores a combined direction-plus-TP system on strict accuracy and reach together, and draws the frontier. It runs on CPU; it uses the GPU only if you enable the V3 screen.
2d. **Option A — safe TP (on by default).** The model outputs a TP for every candle; it is correct when the TP is at or below the real endpoint. Accuracy is calibrated to `SAFE_TP_GOAL` (98%), and models compete on TP size against a fixed ATR rule. The tree-model part runs on CPU; `RUN_SAFE_TP_NEURAL` adds one short V3 network run per fold on the GPU. The cross-asset study is now off: it added nothing.
3. **Second session — selection (off by default; only if an information test shows a real out-of-sample lift).** Set `RUN_V3_SELECTION=True`. With the default stages this trains 7–8 configurations × 3 windows (up to 40 epochs each, early stopping on validation loss). Every run checkpoints to Drive and resumes; cached candidates are not retrained.
4. **Third session — refit and evaluation.** Set `RUN_V3_REFIT_EVALUATION=True`. The plan is reloaded from `selection/v3_frozen_plan.json`. Reach simulation, replay audits and the milestone report follow.
5. `EXECUTION_TEST=True` runs the whole notebook with a reduced network and sampled anchors. It is a code check; its scores are not evidence.

**Data.** The supplied file is BTC-only; V3 trains on the primary symbol. The `signal` column is the hindsight label direction: it defines the target and the training-only leg grouping, and it is never a model input. **Label maturity** still uses `tp_confirm_time` as an unverified proxy, because the original ZigZag generator is not available here.

In [1]:
# Install only missing runtime packages. Preserve Colab's existing CUDA PyTorch.
import importlib.util
import importlib.metadata
import subprocess
import sys

requirements = {
    'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<3',
    'torch': 'torch>=2.6,<3', 'matplotlib': 'matplotlib>=3.8,<4',
    'tqdm': 'tqdm>=4.66,<5',
}
missing = [requirement for module, requirement in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
version = importlib.metadata.version('torch').split('+')[0].split('.')
if tuple(map(int, version[:2])) < (2, 6):
    raise RuntimeError('PyTorch 2.6 or later is required. Update your runtime and restart it.')
print('Python', sys.version.split()[0], '| PyTorch', importlib.metadata.version('torch'))

Python 3.13.16 | PyTorch 2.11.0+cu130


### Imports and configuration types

`Config` keeps every V2 backbone field and adds the V3 fields. V2 field names are retained so that a saved V2 manifest still loads for diagnostics. The acceptance threshold is now 0.80 (V2 recorded an outdated 0.50).

In [2]:
from __future__ import annotations
import copy
import hashlib
import importlib.metadata
import json
import math
import os
import random
import time
from contextlib import nullcontext
from dataclasses import dataclass, field, asdict, replace, fields
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# V2 identifiers are kept so a saved V2 bundle can be loaded for diagnostics.
V2_FEATURE_VERSION = "atr_endpoint_81_v1"
V2_IMPLEMENTATION_VERSION = "model_b_interval_v1_rowwise_decode"
FEATURE_VERSION = "atr_endpoint_81_v1+zigzag_state_v3"
IMPLEMENTATION_VERSION = "model_b_v3_directional_multitask_v1"
BAR = pd.Timedelta(minutes=15)
HORIZONS = (1, 2, 4, 8, 16, 32, 64, 96, 192)
RAW_COLUMNS = (
    "open", "high", "low", "close", "volume", "quote_volume",
    "num_trades", "taker_base_volume", "taker_quote_volume",
)
STRICT_BAND_LOG = math.log(4/3)  # log width of the 75%-100% acceptance window

@dataclass
class Config:
    # ---- Preserved V2 backbone (do not shrink by default) ----
    primary_symbol: str = "BTCUSDT"
    context: int = 1024
    width: int = 192
    kernel: int = 5
    dilations: tuple = (1, 2, 4, 8, 16, 32, 64)
    transformer_layers: int = 2
    heads: int = 8
    ff_multiplier: int = 4
    dropout: float = 0.20            # V2 0.15
    asset_dim: int = 16
    adapter_dim: int = 48
    unknown_asset_probability: float = 0.10
    fractal_depth: int = 12
    use_structure: bool = True
    lower_ratio: float = 0.75
    target_mode: str = "z_atr"

    # ---- V3.1 direction and ZigZag leg state ----
    direction_source: str = "causal_zigzag"  # or "column" (needs a verified live column)
    side_column: str = "signal_live"
    zigzag_depth: int = 12
    zigzag_min_deviation_atr: float = 0.0
    use_state_features: bool = True     # 16 causal leg-state/directional features
    state_in_sequence: bool = True      # also feed state history to the encoder
    use_film: bool = True               # h' = (1 + gamma) * h + beta
    use_state_fusion: bool = True       # gated residual late fusion
    side_embedding_dim: int = 8
    conditioning_hidden: int = 64

    # ---- V3.2 / V3.3 heads and losses ----
    head_type: str = "distribution"     # "distribution" or "interval" (V2 head)
    aux_lambda: float = 0.10            # weight of Huber(log(1+Z)) auxiliary loss
    magnitude_head: bool = True
    magnitude_hidden: int = 128
    huber_delta: float = 1.0
    head_dropout: float = 0.15          # V2 0.10
    dist_bins: int = 96
    dist_core_quantiles: tuple = (0.005, 0.995)
    dist_label_sigma_bins: float = 0.75  # Gaussian label smoothing width, in core-bin widths; 0 = hard labels
    # V2 interval-head settings
    query_rbf_count: int = 32
    query_dim: int = 32
    grid_intervals_per_window: int = 32
    grid_chunk: int = 256
    refine_peaks: int = 5
    refinement_points: int = 16
    checkpoint_head: bool = True

    # ---- V3.2 optimizer (per-module learning rates) ----
    lr_input: float = 1e-4
    lr_tcn: float = 1e-4
    lr_transformer: float = 1e-4
    lr_adapter: float = 3e-4
    lr_film: float = 3e-4
    lr_fusion: float = 3e-4
    lr_heads: float = 3e-4
    lr_aux: float = 3e-4
    weight_decay: float = 0.005         # V2 0.0005; matrices only
    gradient_clip: float = 1.0
    warmup_fraction: float = 0.05
    ema_decay: float = 0.995
    batch_size: int = 16
    eval_batch_size: int = 8
    gradient_accumulation: int = 4
    max_epochs: int = 40                # V2 80
    patience: int = 5                   # V2 8; on validation LOSS
    grad_log_every: int = 25            # per-component gradient snapshot interval (updates)
    loss_grad_probe_every: int = 200    # primary-vs-auxiliary gradient probe interval (updates)
    train_probe_anchors: int = 2048     # fixed training subset scored every epoch (train vs val spread)

    # ---- V3.4 sampling, validation and selection ----
    sampler: str = "uniform"            # "uniform", "leg_capped", "leg_inv_sqrt"
    leg_cap: int = 8
    max_train_anchors_per_epoch: int | None = None
    anchor_gap_bars: int = 128          # embargo between training data and the next period
    dev_validation_days: int = 60
    selection_tie_pp: float = 0.25
    scaler_samples_per_asset: int = 20000
    seeds: tuple = (7,)
    amp: bool = True
    workers: int = 0
    checkpoint_every_updates: int = 500
    bootstrap_block_days: int = 7
    bootstrap_repetitions: int = 1000
    rowwise_final_decode: bool = True

    # ---- Acceptance contract ----
    minimum_accuracy: float = 0.80
    minimum_execution_reach: float = 0.98
    run_mode: str = "development"
    protocol_frozen_at: str | None = None
    resume: bool = True
    legacy_v2: bool = False             # True only when loading a saved V2 bundle for diagnostics
    execution_test_max_train_anchors: int | None = None
    execution_test_max_eval_anchors: int | None = None

    # ---- V2 fields kept only so V2 manifests load ----
    pooled_lr: float = 3e-4
    btc_encoder_lr: float = 3e-5
    btc_head_lr: float = 1e-4
    pooled_epochs: int = 80
    btc_epochs: int = 40
    inner_validation_days: int = 60

    def validate(self):
        assert self.context >= 2 and self.width % self.heads == 0
        assert self.lower_ratio == 0.75, "This notebook implements the agreed 75%-100% rule."
        assert self.target_mode in ("z_atr", "raw_y")
        assert self.run_mode in ("development", "final")
        assert self.head_type in ("distribution", "interval")
        assert self.direction_source in ("causal_zigzag", "column")
        assert self.sampler in ("uniform", "leg_capped", "leg_inv_sqrt")
        assert self.batch_size > 0 and self.gradient_accumulation > 0
        assert self.grid_intervals_per_window >= 8 and self.dist_bins >= 8
        assert self.aux_lambda >= 0 and (self.aux_lambda == 0 or self.magnitude_head)
        if not self.legacy_v2:
            assert self.minimum_accuracy >= 0.80, "The acceptance contract is >= 80% strict accuracy."
            assert self.minimum_execution_reach >= 0.98
        if self.run_mode == "final":
            assert self.target_mode == "z_atr" and not self.legacy_v2
            assert self.protocol_frozen_at, "Final mode requires a recorded protocol freeze."
            assert self.execution_test_max_train_anchors is None
            assert self.execution_test_max_eval_anchors is None
        return self

    @property
    def conditioned(self):
        return self.use_state_features and (self.use_film or self.use_state_fusion)

def config_from_dict(d, **overrides):
    """Build a Config from a saved manifest, ignoring unknown keys and restoring tuples."""
    names = {f.name for f in fields(Config)}
    values = {k: (tuple(v) if isinstance(v, list) else v) for k, v in d.items() if k in names}
    values.update(overrides)
    return Config(**values)

def utc(value):
    return pd.Timestamp(value).tz_localize("UTC") if pd.Timestamp(value).tzinfo is None else pd.Timestamp(value).tz_convert("UTC")

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def json_safe(x):
    if isinstance(x, dict):
        return {str(k): json_safe(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [json_safe(v) for v in x]
    if isinstance(x, np.ndarray):
        return json_safe(x.tolist())
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating, float)):
        return float(x) if np.isfinite(x) else None
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, (pd.Timestamp, Path)):
        return str(x)
    return x

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(json_safe(obj), indent=2, allow_nan=False))
    os.replace(temp, path)

def atomic_torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, temp)
    os.replace(temp, path)

def package_versions():
    result = {}
    for name in ("torch", "numpy", "pandas", "matplotlib", "tqdm"):
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            pass
    return result

## Configuration — edit this cell

`SELECTION_BASE` starts from the V2 head with no auxiliary loss, so the first stage isolates the V3.1 input change. Later stages change one factor each, in plan order. Options that alter data preparation (context, bins, embargo, side source) are rejected inside selection, because every candidate must see identical windows.

`direction_source='causal_zigzag'` derives the live side from confirmed pivots: after a confirmed low the leg is LONG, after a confirmed high it is SHORT. To use a strategy column instead, set `direction_source='column'`, `side_column`, and `live_side_verified=True` only with evidence that the column is produced at candle close. For out-of-memory errors, reduce `batch_size`/`eval_batch_size` and raise `gradient_accumulation`, then use a new `RUN_NAME`.

In [3]:
CFG = Config()  # Full V2 backbone + V3 conditioning, distribution head, auxiliary loss, V3 regularization.

# Path to the labelled BTC CSV. Override without editing the notebook by setting the MODEL_B_DATA environment variable.
DATA_FILES = {
    'BTCUSDT': os.environ.get('MODEL_B_DATA', '/content/drive/MyDrive/data_raw_with_indicators_tp_cleaned_continuous.csv'),
}
# Set DATA_FILES = {} to use automatic local-file discovery/upload instead.
DATA_METADATA = {
    'BTCUSDT': {'confirmation_timestamp_is_bar_open': True,
                'maturity_verified': False, 'label_rule_id': None,
                # Set True ONLY if CFG.side_column is produced by the live strategy at candle close.
                'live_side_verified': False},
}

# Jul-Oct 2026 was inspected during V2. It remains DEVELOPMENT history for V3, never a final test.
EVAL_START = '2026-07-01'
EVAL_END = '2026-10-15'  # Exclusive, UTC bar opening time.
RUN_NAME = 'model_b_v3_directional_multitask'
V2_RUN_DIR = 'model_b_atr_interval_v1'  # saved V2 run: a name under OUTPUT_ROOT, or an absolute path
MOUNT_GOOGLE_DRIVE = True
# A saved checkpoint whose config/data/scaler/bins differ from this run: 'archive' moves it to
# <dir>__stale_<timestamp> and retrains (nothing is deleted); 'raise' stops with an error instead.
ON_CHECKPOINT_MISMATCH = 'archive'
OUTPUT_ROOT = '/content/model_b_runs' if Path('/content').exists() else str(Path.cwd()/'model_b_runs')

# ---------------- Stage switches (run in this order) ----------------
RUN_CAUSALITY_AUDITS = True
# V3.0 ran on 1 Nov 2026 (results in v3_0_diagnostics/; M1 passed). Set True only to repeat it.
RUN_V30_COLLAPSE_DIAGNOSTICS = False # A: reads the saved V2 run (+ V2 bundle if present). CPU is fine.
RUN_V30_GRADIENT_PROBE = False       # B: per-component gradient flow on the V2 configuration. GPU.
RUN_V30_MEMORIZATION_TEST = False    # C: can the network fit 64-128 anchors? GPU.
# Information tests ran on 1 Nov 2026 (direction usable at ~69%; TP size information-limited). Results in information_tests/.
RUN_DIRECTION_STUDY = False          # Info 1 (CPU): is the label direction predictable at candle close?
RUN_INFORMATION_TEST = False         # Info 2 (CPU): gradient-boosted log-Z test, with/without ORACLE direction
RUN_V3_SCREEN = False                # Info 3 (GPU): V3 on the latest window, conditioning on vs off, short budget
# Off by default: the information tests and safe-TP study show the network does not beat fixed ATR.
RUN_V3_SELECTION = False            # V3.1-V3.4 staged selection on development windows. Expensive (7-8 configs x 3 windows).
RUN_V3_REFIT_EVALUATION = False     # Refit selected plan before EVAL_START; needs selection or FROZEN_PLAN_PATH.
RUN_REACH_SIMULATION = True          # Needs refit predictions (V3) and/or saved V2 predictions.
EXPORT_MODEL_ZIP = True
DOWNLOAD_MODEL_ZIP = False
FROZEN_PLAN_PATH = ''                # Final mode only: a frozen v3_frozen_plan.json.

# ---------------- V3.0 diagnostics ----------------
COLLAPSE_TRAIN_SAMPLE = 5000         # training anchors scored with the saved V2 bundle (seen data)
GRADIENT_PROBE_UPDATES = 300
MEMORIZATION_ANCHORS = 128           # 64-128, one per original leg where possible
MEMORIZATION_UPDATES = 600
MEMORIZATION_PASS_ACCURACY = 0.90

# ---------------- Information tests ----------------
INFO_ZIGZAG_DEPTHS = (3, 5, 8, 12)   # causal confirmed-pivot sides compared with the label direction
INFO_GBDT_BINS = 32                  # log-Z bins for the distributional GBDT (same exact decoder as V3)
INFO_MAX_TRAIN_ROWS = 40000          # evenly thinned training anchors per fold (effective sample is ~9k anyway)
INFO_GBDT_ROUNDS = 600               # maximum trees; early stopping on the last 10% of training time
INFO_V3_EPOCHS = 12                  # V3 screen budget (early stopping patience 3)
INFO_V3_ANCHORS_PER_EPOCH = 20000

# ---------------- Cross-asset study and strict/reach frontier ----------------
# Ran 1 Nov 2026: cross-asset inputs added nothing (strict 14.3% vs 14.0%); no rule reaches 98% reach.
RUN_CROSS_ASSET_STUDY = False        # CPU + download: other symbols as BTC inputs; combined direction + TP system
CROSS_SYMBOLS = ('ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'XRPUSDT', 'DOGEUSDT')
CROSS_ASSET_FILES = {}               # optional {'ETHUSDT': '/content/drive/MyDrive/ethusdt_15m.csv'} with the raw columns
CROSS_ASSET_BASE_URL = 'https://data.binance.vision/data/spot'
FRONTIER_QUANTILES = (0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60)
FRONTIER_ATR_MULTIPLES = tuple(np.round(np.geomspace(0.25, 4.0, 17), 3))

# ---------------- Option A: safe TP (TP at or below the real endpoint) ----------------
RUN_SAFE_TP_STUDY = True             # CPU: calibrated quantile TP models vs fixed ATR at equal accuracy
RUN_SAFE_TP_NEURAL = False           # GPU: also the V3 network's distribution head. Off: matched fixed ATR (0.096% vs 0.095%)
SAFE_TP_GOAL = 0.98
SAFE_TP_TARGETS = (0.90, 0.95, 0.98)
SAFE_TP_NEURAL_EPOCHS = 15
SAFE_TP_NEURAL_ANCHORS_PER_EPOCH = 20000

# ---------------- V3.4 chronological development windows ----------------
DEV_WINDOW_STARTS = ['2025-09-01', '2026-01-01', '2026-04-01']  # each validates CFG.dev_validation_days
REG_GAP_TRIGGER_PP = 5.0  # regularization stage runs only if train-probe accuracy exceeds validation by this much
# Staged, attributable selection. Each stage keeps the incumbent unless an option wins under the
# predeclared rule (pooled strict accuracy, then min per-window lift over fixed ATR, then val loss).
SELECTION_BASE = dict(head_type='interval', aux_lambda=0.0, magnitude_head=False)
SELECTION_STAGES = [
    ('V3.1_conditioning', [
        dict(use_state_features=True, use_film=True, use_state_fusion=True, state_in_sequence=True),
        dict(use_state_features=False, use_film=False, use_state_fusion=False, state_in_sequence=False)]),
    ('V3.2_aux_lambda', [
        dict(aux_lambda=0.0, magnitude_head=False), dict(aux_lambda=0.03, magnitude_head=True),
        dict(aux_lambda=0.10, magnitude_head=True), dict(aux_lambda=0.30, magnitude_head=True)]),
    ('V3.3_head', [dict(head_type='interval'), dict(head_type='distribution')]),
    ('V3.4_sampler', [dict(sampler='uniform'), dict(sampler='leg_capped')]),
    ('V3.4_regularization_if_gap', [
        dict(), dict(dropout=0.25, head_dropout=0.20, weight_decay=0.01)]),
]

# ---------------- V3.5 execution reach (separate from strict accuracy) ----------------
REACH_SETTINGS = dict(
    entry='signal_close',        # 'signal_close' or 'next_open'
    max_hold_bars=192,           # 2 days of 15-minute bars
    stop_atr_multiple=None,      # e.g. 3.0 for a stop at 3 x ATR14 from entry; None = no stop
    spread_bps=1.0,              # adverse price buffer a TP must clear
    fee_bps_per_side=5.0,        # used for net return reporting
    same_bar_policy='stop_first',
)

# Confirmed-pivot ATR comparator (V2 rule): bins are training-set quantiles per pivot side.
PIVOT_RULE_AGE_BINS = 3
PIVOT_RULE_DISTANCE_BINS = 3
PIVOT_RULE_MIN_BUCKET = 500

EXECUTION_TEST = False
if EXECUTION_TEST:
    CFG = replace(CFG, context=128, width=64, heads=4, transformer_layers=1, dilations=(1, 2, 4),
                  batch_size=4, eval_batch_size=4, gradient_accumulation=2, max_epochs=2, patience=1,
                  dist_bins=24, grid_intervals_per_window=8, bootstrap_repetitions=100,
                  train_probe_anchors=32, grad_log_every=2, loss_grad_probe_every=3,
                  execution_test_max_train_anchors=256, execution_test_max_eval_anchors=64)
    GRADIENT_PROBE_UPDATES, MEMORIZATION_ANCHORS, MEMORIZATION_UPDATES, COLLAPSE_TRAIN_SAMPLE = 6, 16, 30, 64
    INFO_ZIGZAG_DEPTHS, INFO_MAX_TRAIN_ROWS, INFO_GBDT_ROUNDS, INFO_V3_EPOCHS, INFO_V3_ANCHORS_PER_EPOCH = (5, 12), 2000, 30, 1, 128
    SAFE_TP_NEURAL_EPOCHS, SAFE_TP_NEURAL_ANCHORS_PER_EPOCH = 1, 128
    RUN_NAME += '_EXECUTION_TEST'
    print('EXECUTION TEST: reduced network and sampled anchors; no performance claim.')

CFG.validate()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| model:', CFG.width, 'wide /', CFG.context, 'candles | head:', CFG.head_type)
if DEVICE.type == 'cpu' and not EXECUTION_TEST:
    print('Full training will run on CPU. Select a GPU runtime for the intended workload.')

Device: cuda | model: 192 wide / 1024 candles | head: distribution


In [4]:
# Optional packages are installed only when the corresponding study is enabled.
import importlib.util, subprocess, sys
if (RUN_DIRECTION_STUDY or RUN_INFORMATION_TEST or RUN_CROSS_ASSET_STUDY or RUN_SAFE_TP_STUDY) and importlib.util.find_spec('lightgbm') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm>=4,<5'])
try:
    from IPython.display import display, FileLink
except ImportError:
    display = print

## Implementation

| Input group | Count | Notes |
|---|---:|---|
| Frozen V2 causal features | 81 | Returns, ATR displacements, candle geometry, structure, volatility, activity, momentum, depth-12 fractals, calendar |
| ZigZag leg state | 12 | Has-state mask, side, pivot-confirmation age, leg age, leg extension, MFE and MAE since the live signal, distance from the favorable extreme, previous leg, median of the last 5 legs, extension/median ratio, leg-history fraction |
| Volatility and direction | 4 | ATR14/ATR96; side × 16-bar displacement, side × MACD histogram, side × HMA slope |
| Live side | embedding | LONG / SHORT / unknown, into the FiLM and fusion conditioner |

All state is computed online from bars ≤ t. A fractal pivot enters the state only after its 12 right-hand candles close. Favorable and adverse extremes are the values observed so far; the still-unconfirmed final endpoint is never an input. The ZigZag is a **causal reconstruction**, not the original generator, so `audit_label_prefixes` remains the route to verify label finality once that generator is supplied.

### Source data contract, label availability and live-direction provenance

Validation is unchanged from V2. New: `label_side` (hindsight, from `signal`) and a training-only `leg_id` built from `tp_pivot_time`, so anchors sharing one future endpoint form one original leg. `direction_provenance_report` measures how often the live side agrees with the label direction; low agreement is not leakage, but it bounds what the side input can contribute. A `signal_live` column that matches the hindsight direction on more than 99.5% of rows is flagged.

In [5]:
def validate_raw_frame(frame):
    d = frame.copy()
    missing = set(("time",) + RAW_COLUMNS) - set(d)
    if missing:
        raise ValueError(f"Missing raw input columns: {sorted(missing)}")
    d["time"] = pd.to_datetime(d["time"], utc=True, errors="raise")
    if d["time"].duplicated().any() or not d["time"].is_monotonic_increasing:
        raise ValueError("Timestamps must already be unique and sorted. No silent sorting.")
    if len(d) < 2 or not d["time"].diff().iloc[1:].eq(BAR).all():
        raise ValueError("Expected consecutive 15-minute bars; repair/segment raw gaps explicitly.")
    if not d["time"].dt.floor("15min").eq(d["time"]).all():
        raise ValueError("Bar opening timestamps are not on the 15-minute grid.")
    for col in RAW_COLUMNS:
        d[col] = pd.to_numeric(d[col], errors="raise").astype(np.float64)
    a = d[list(RAW_COLUMNS)].to_numpy()
    if not np.isfinite(a).all():
        raise ValueError("Nonfinite raw market inputs; do not drop interior rows.")
    if (d[["open", "high", "low", "close"]] <= 0).any().any():
        raise ValueError("Prices must be strictly positive.")
    if (d.high < d[["open", "low", "close"]].max(axis=1)).any() or (d.low > d[["open", "high", "close"]].min(axis=1)).any():
        raise ValueError("OHLC ordering is invalid.")
    volumes = ["volume", "quote_volume", "num_trades", "taker_base_volume", "taker_quote_volume"]
    if (d[volumes] < 0).any().any():
        raise ValueError("Negative volume or trade counts.")
    if ((d.taker_base_volume > d.volume + 1e-8) | (d.taker_quote_volume > d.quote_volume + 1e-5)).any():
        raise ValueError("Taker volume exceeds total volume.")
    d["available_at"] = d.time + BAR
    if "close_time" in d:
        ct = pd.to_datetime(d.close_time, utc=True, errors="raise")
        if not ct.eq(d.available_at - pd.Timedelta(milliseconds=1)).all():
            raise ValueError("close_time does not match a completed 15-minute candle.")
    return d.reset_index(drop=True)

def parse_side(values):
    """long/buy/+1 -> +1, short/sell/-1 -> -1, anything else -> 0 (unknown)."""
    s = pd.Series(values)
    if pd.api.types.is_numeric_dtype(s):
        v = pd.to_numeric(s, errors="coerce").to_numpy(float)
        return np.where(v > 0, 1, np.where(v < 0, -1, 0)).astype(np.int8)
    t = s.astype(str).str.strip().str.lower()
    return np.where(t.isin(["long", "buy", "1", "+1", "1.0"]), 1,
                    np.where(t.isin(["short", "sell", "-1", "-1.0"]), -1, 0)).astype(np.int8)

def load_labeled_csv(path, metadata=None):
    metadata = dict(metadata or {})
    d = validate_raw_frame(pd.read_csv(path))
    if "tp_pct" not in d:
        raise ValueError("Training needs original tp_pct labels; no replacement label is invented.")
    d["tp_pct"] = pd.to_numeric(d.tp_pct, errors="raise")
    y = d.tp_pct.to_numpy(dtype=np.float64)
    if np.isinf(y).any() or np.any(y[np.isfinite(y)] < 0):
        raise ValueError("tp_pct must be nonnegative or pending (NaN).")
    for c in ("tp_pivot_time", "tp_confirm_time", "label_available_time"):
        if c in d:
            d[c] = pd.to_datetime(d[c], utc=True, errors="coerce")
    if "label_available_time" in d and metadata.get("maturity_verified", False):
        maturity = d.label_available_time
        maturity_status = "verified_by_supplied_provenance"
    elif "tp_confirm_time" in d:
        offset = BAR if metadata.get("confirmation_timestamp_is_bar_open", True) else pd.Timedelta(0)
        maturity = d.tp_confirm_time + offset
        maturity_status = "provided_confirmation_proxy_UNVERIFIED"
    else:
        raise ValueError("Provide tp_confirm_time or a verified label_available_time.")
    d["label_available_at"] = maturity
    if (d.label_available_at.notna() & (d.label_available_at < d.available_at)).any():
        raise ValueError("A future endpoint label is claimed available before its prediction.")
    if "tp_pivot_time" in d:
        valid = d.tp_pct.notna() & d.tp_pivot_time.notna()
        if (d.loc[valid, "tp_pivot_time"] <= d.loc[valid, "time"]).any():
            raise ValueError("This task expects strictly future endpoint pivots.")
        if (d.loc[valid, "label_available_at"] < d.loc[valid, "tp_pivot_time"] + BAR).any():
            raise ValueError("Label availability precedes the close of its endpoint bar.")
    if {"signal", "tp_price"}.issubset(d.columns):
        s = d.signal.map({"long": 1.0, "short": -1.0}).to_numpy()
        signed = 100 * s * (d.tp_price.to_numpy() - d.close.to_numpy()) / d.close.to_numpy()
        valid = np.isfinite(y)
        if not np.isfinite(signed[valid]).all() or not np.allclose(np.maximum(signed[valid], 0), y[valid], rtol=1e-9, atol=1e-10):
            raise ValueError("Supplied targets do not reproduce the documented original endpoint formula.")
    # Label-defining direction: HINDSIGHT. Used only for diagnostics and training-only leg IDs.
    d["label_side"] = parse_side(d["signal"]) if "signal" in d else np.zeros(len(d), np.int8)
    # Training-only original leg identifier: anchors that share the same future endpoint pivot.
    if "tp_pivot_time" in d:
        key = d.tp_pivot_time.astype(str) + "|" + d.label_side.astype(str)
        codes, _ = pd.factorize(key)
        d["leg_id"] = np.where(d.tp_pivot_time.notna() & d.tp_pct.notna(), codes, -1).astype(np.int64)
        leg_source = "tp_pivot_time + signal (training-only grouping; never an input)"
    else:
        d["leg_id"] = -1
        leg_source = "unavailable: leg-aware samplers fall back to uniform"
    info = {
        "source_path": str(path), "source_sha256": file_sha256(path),
        "rows": len(d), "start": str(d.time.iloc[0]), "end": str(d.time.iloc[-1]),
        "pending_labels": int(d.tp_pct.isna().sum()),
        "zero_targets": int(d.tp_pct.eq(0).sum()),
        "maturity_status": maturity_status,
        "label_rule_id": metadata.get("label_rule_id"),
        "leg_id_source": leg_source,
        "unique_legs": int(len(np.unique(d.leg_id[d.leg_id >= 0]))),
        "metadata": metadata,
    }
    return d, info

def audit_label_prefixes(raw, label_generator: Callable, cut_positions, atol=1e-10):
    """Audit a user's original generator; never manufacture a substitute rule.

    The callback receives raw market columns only and returns time, tp_pct,
    tp_pivot_time, and optionally label_available_time. It must recompute its
    own indicators/labels; the supplied future-label columns are never passed.
    """
    if label_generator is None:
        raise ValueError("Supply the original label generator to audit label finality.")
    reference = raw.set_index("time")
    reports = []
    for end in cut_positions:
        cutoff = raw.available_at.iloc[int(end)]
        input_columns = ["time", *RAW_COLUMNS]
        if "close_time" in raw:
            input_columns.append("close_time")
        generated = label_generator(raw.iloc[:int(end) + 1][input_columns].copy())
        generated = generated.copy()
        generated["time"] = pd.to_datetime(generated.time, utc=True)
        generated = generated.set_index("time")
        expected = reference.loc[reference.label_available_at <= cutoff]
        matches = generated.reindex(expected.index)
        mismatch = ~np.isclose(
            expected.tp_pct.to_numpy(float),
            pd.to_numeric(matches.tp_pct, errors="coerce").to_numpy(float),
            atol=atol, rtol=1e-9, equal_nan=False,
        )
        if "tp_pivot_time" in expected and "tp_pivot_time" in matches:
            mismatch |= pd.to_datetime(expected.tp_pivot_time, utc=True).ne(
                pd.to_datetime(matches.tp_pivot_time, utc=True)).to_numpy()
        reports.append({"prefix_close": str(cutoff), "claimed_mature_rows": len(expected), "mismatches": int(mismatch.sum())})
    report = pd.DataFrame(reports)
    if report.mismatches.sum():
        raise AssertionError(f"Label maturity audit failed:\n{report}")
    return report

def direction_provenance_report(asset, cfg, start=None, end=None):
    """How often does the live side used as model input agree with the hindsight label direction?

    The model is conditioned on asset.side (live). The target is still defined by the label direction.
    A low agreement rate is not leakage, but it bounds how useful the side input can be.
    """
    raw = asset.raw
    m = np.isfinite(asset.y) & asset.finite_features
    if start is not None:
        m &= (raw.time >= utc(start)).to_numpy()
    if end is not None:
        m &= (raw.time < utc(end)).to_numpy()
    live, label = asset.side[m], raw.label_side.to_numpy()[m]
    known = (live != 0) & (label != 0)
    report = {
        "rows": int(m.sum()), "side_source": asset.side_source,
        "live_side_unknown_rows": int((live == 0).sum()),
        "live_long_share": float(np.mean(live[live != 0] > 0)) if (live != 0).any() else None,
        "agreement_live_vs_label_direction": float(np.mean(live[known] == label[known])) if known.any() else None,
    }
    if "signal_live" in raw:
        sl = parse_side(raw.signal_live)[m]
        k2 = (sl != 0) & (label != 0)
        report["agreement_signal_live_vs_label_direction"] = float(np.mean(sl[k2] == label[k2])) if k2.any() else None
        k3 = (sl != 0) & (live != 0)
        report["agreement_signal_live_vs_causal_zigzag_side"] = float(np.mean(sl[k3] == live[k3])) if k3.any() else None
        if report["agreement_signal_live_vs_label_direction"] is not None and report["agreement_signal_live_vs_label_direction"] > 0.995:
            report["warning"] = ("signal_live matches the hindsight label direction on >99.5% of rows. "
                                 "Treat it as hindsight unless its live generation is proven.")
    return report

### Feature pipeline: frozen V2 features + causal ZigZag state

`causal_features` is the V2 function, unchanged. `causal_zigzag_state` alternates confirmed pivots: a more extreme same-side pivot extends the leg; an opposite pivot starts a new one (optionally only beyond `zigzag_min_deviation_atr`). `build_features` is used identically by training, evaluation and the saved-model predictor.

In [6]:
def safe_ratio(num, den, neutral=0.0):
    num, den = np.asarray(num, dtype=np.float64), np.asarray(den, dtype=np.float64)
    out = np.full(np.broadcast_shapes(num.shape, den.shape), neutral, dtype=np.float64)
    np.divide(num, den, out=out, where=(den != 0))
    out[~np.isfinite(num) | ~np.isfinite(den)] = np.nan
    return out

def wma(s, n):
    weights = np.arange(1, n + 1, dtype=np.float64)
    return s.rolling(n, min_periods=n).apply(lambda a: np.dot(a, weights) / weights.sum(), raw=True)

def wilder_rsi(close, n=14):
    diff = close.diff().to_numpy()
    gain, loss = np.maximum(diff, 0), np.maximum(-diff, 0)
    ag, al = np.full(len(close), np.nan), np.full(len(close), np.nan)
    if len(close) > n:
        ag[n], al[n] = np.mean(gain[1:n+1]), np.mean(loss[1:n+1])
        for i in range(n + 1, len(close)):
            ag[i] = ((n-1) * ag[i-1] + gain[i]) / n
            al[i] = ((n-1) * al[i-1] + loss[i]) / n
    result = np.full(len(close), np.nan)
    active = np.isfinite(ag)
    result[active & (al == 0) & (ag == 0)] = 50
    result[active & (al == 0) & (ag > 0)] = 100
    nz = active & (al > 0)
    result[nz] = 100 - 100 / (1 + ag[nz] / al[nz])
    return pd.Series(result, index=close.index)

def causal_features(raw, fractal_depth=12, use_structure=True):
    """FROZEN V2 81-feature pipeline. Uses raw OHLC/activity only; all TP/signal columns are ignored."""
    d = raw
    c, h, l, o = (d[k].astype(np.float64) for k in ("close", "high", "low", "open"))
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    atrs = {n: tr.ewm(alpha=1/n, adjust=False).mean() for n in (14, 56, 192)}
    atr = atrs[14]
    a_raw = 100 * atr / c
    f = {}
    for k in HORIZONS:
        f[f"logret_{k}"] = np.log(c/c.shift(k))
    for k in HORIZONS:
        f[f"displacement_atr_{k}"] = (c-c.shift(k))/atr
    f["body_atr"] = (c-o)/atr
    f["upper_wick_atr"] = (h-np.maximum(o, c))/atr
    f["lower_wick_atr"] = (np.minimum(o, c)-l)/atr
    f["range_atr"] = (h-l)/atr
    f["close_position"] = safe_ratio(c-l, h-l, neutral=0.5)-0.5
    for w in (16, 64, 192):
        hh, ll = h.rolling(w).max(), l.rolling(w).min()
        f[f"distance_high_atr_{w}"] = (hh-c)/atr
        f[f"distance_low_atr_{w}"] = (c-ll)/atr
        f[f"range_position_{w}"] = safe_ratio(c-ll, hh-ll, neutral=0.5)-0.5
        f[f"ma_distance_atr_{w}"] = (c-c.rolling(w).mean())/atr
    for n in (14, 56, 192):
        f[f"atr_pct_{n}"] = 100*atrs[n]/c
    f["atr_ratio_14_56"] = safe_ratio(atrs[14], atrs[56], 1)
    f["atr_ratio_14_192"] = safe_ratio(atrs[14], atrs[192], 1)
    r = np.log(c/c.shift())
    rv = {w: 100*r.rolling(w).std(ddof=0) for w in (16, 64, 192)}
    for w in (16, 64, 192):
        f[f"rv_{w}"] = rv[w]
    f["rv_ratio_16_64"] = safe_ratio(rv[16], rv[64], 1)
    f["rv_ratio_64_192"] = safe_ratio(rv[64], rv[192], 1)
    f["true_range_atr"] = tr/atr

    def z96(v):
        v = pd.Series(v, index=c.index)
        return safe_ratio(v-v.rolling(96).mean(), v.rolling(96).std(ddof=0), 0)
    v, q, ntr, taker = (d[k].astype(float) for k in ("volume", "quote_volume", "num_trades", "taker_base_volume"))
    f["volume_z96"] = z96(np.log1p(v))
    f["trades_z96"] = z96(np.log1p(ntr))
    f["average_trade_value_z96"] = z96(np.log1p(safe_ratio(q, ntr, 0)))
    imbalance = pd.Series(safe_ratio(2*taker-v, v, 0), index=c.index)
    f["taker_imbalance"] = imbalance
    for w in (4, 16, 64):
        f[f"taker_imbalance_mean_{w}"] = imbalance.rolling(w).mean()
    f["volume_ratio_16_96"] = safe_ratio(v.rolling(16).mean(), v.rolling(96).mean(), 1)
    f["trades_ratio_16_96"] = safe_ratio(ntr.rolling(16).mean(), ntr.rolling(96).mean(), 1)
    vwap = pd.Series(safe_ratio(q, v, 0), index=c.index).where(v > 0, c)
    f["vwap_distance_atr"] = (c-vwap)/atr
    for w in (16, 64):
        f[f"volume_weighted_imbalance_{w}"] = safe_ratio((2*taker-v).rolling(w).sum(), v.rolling(w).sum(), 0)
    rsi = wilder_rsi(c)
    stoch = pd.Series(safe_ratio(rsi-rsi.rolling(14).min(), rsi.rolling(14).max()-rsi.rolling(14).min(), 0.5)*100, index=c.index)
    k, kd = stoch.rolling(4).mean(), stoch.rolling(4).mean().rolling(4).mean()
    line = c.ewm(span=8, adjust=False).mean()-c.ewm(span=21, adjust=False).mean()
    hist = line-line.ewm(span=5, adjust=False).mean()
    hma = wma(2*wma(c, 17)-wma(c, 34), 5)
    f["rsi14"] = rsi/100-0.5
    f["rsi14_ma14"] = rsi.rolling(14).mean()/100-0.5
    f["stochrsi_k"] = k/100-0.5
    f["stochrsi_d"] = kd/100-0.5
    f["macd_line_atr"] = line/atr
    f["macd_hist_atr"] = hist/atr
    f["rsi_change"] = rsi.diff()/100
    f["macd_hist_change_atr"] = hist.diff()/atr
    f["hma_distance_atr"] = (c-hma)/atr
    f["hma_slope_atr"] = hma.diff()/atr

    high, low, close, av = h.to_numpy(), l.to_numpy(), c.to_numpy(), atr.to_numpy()
    structure = np.zeros((len(d), 9), dtype=np.float64)
    hi_i = lo_i = None
    D = fractal_depth
    for t in range(len(d)):
        u = t-D
        if u >= D:
            wh, wl = high[u-D:u+D+1], low[u-D:u+D+1]
            if high[u] == wh.max() and np.count_nonzero(wh == high[u]) == 1:
                hi_i = u
            if low[u] == wl.min() and np.count_nonzero(wl == low[u]) == 1:
                lo_i = u
        if hi_i is not None:
            structure[t, 0] = (close[t]-high[hi_i])/av[t]
            structure[t, 2] = np.log1p(t-hi_i)
            structure[t, 7] = 1
        if lo_i is not None:
            structure[t, 1] = (close[t]-low[lo_i])/av[t]
            structure[t, 3] = np.log1p(t-lo_i)
            structure[t, 8] = 1
        if hi_i is not None and lo_i is not None:
            left, right = sorted([high[hi_i], low[lo_i]])
            structure[t, 4] = np.sign(lo_i-hi_i)
            structure[t, 5] = (right-left)/av[t]
            structure[t, 6] = (close[t]-left)/(right-left)-0.5 if right > left else 0
    names = ["swing_high_distance_atr", "swing_low_distance_atr", "swing_high_age", "swing_low_age",
             "swing_direction", "swing_span_atr", "swing_position", "has_swing_high", "has_swing_low"]
    for j, name in enumerate(names):
        f[name] = structure[:, j] if use_structure else np.zeros(len(d))
    times = pd.to_datetime(d.time, utc=True)
    hours = times.dt.hour + times.dt.minute/60
    f["hour_sin"] = np.sin(2*np.pi*hours/24)
    f["hour_cos"] = np.cos(2*np.pi*hours/24)
    f["weekday_sin"] = np.sin(2*np.pi*times.dt.dayofweek/7)
    f["weekday_cos"] = np.cos(2*np.pi*times.dt.dayofweek/7)
    features = pd.DataFrame(f, index=d.index).astype(np.float64)
    assert features.shape[1] == 81, features.columns.tolist()
    return features, a_raw.to_numpy(dtype=np.float64)

BASE_FEATURE_COUNT = 81
STATE_COLUMNS = [
    "zz_has_state", "zz_side", "zz_pivot_confirm_age_log", "zz_leg_age_log",
    "zz_leg_extension_atr", "zz_mfe_atr", "zz_mae_atr", "zz_dist_from_favorable_extreme_atr",
    "zz_prev_leg_atr_log", "zz_median5_leg_atr_log", "zz_extension_to_median_ratio", "zz_leg_history_fraction",
    "atr_ratio_14_96", "dir_momentum_16_atr", "dir_macd_hist_atr", "dir_hma_slope_atr",
]
PASSTHROUGH_COLUMNS = ("has_swing_high", "has_swing_low", "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
                       "zz_has_state", "zz_side", "zz_leg_history_fraction")

def causal_zigzag_state(high, low, close, atr, depth=12, min_deviation_atr=0.0):
    """Online ZigZag state from confirmed depth-D fractals. CAUSAL RECONSTRUCTION, not the original generator.

    A fractal at bar u is known only at bar t = u + depth, after its right-hand bars closed. Pivots alternate:
    a new same-side pivot that is more extreme replaces the previous one (the leg extends); an opposite pivot
    is appended when it is at least min_deviation_atr x ATR away. The still-unconfirmed final endpoint is
    never used. Every quantity at bar t depends only on bars <= t, so prefix replay is exact.
    Returns (state array [n, 12], live side [n], leg start index [n]).
    """
    n, D = len(close), int(depth)
    out = np.zeros((n, 12), dtype=np.float64)
    side_live = np.zeros(n, dtype=np.int8)
    leg_start = np.full(n, -1, dtype=np.int64)
    pivots = []           # [pivot_side (+1 high / -1 low), bar index, price, confirmation bar]
    leg_pcts = []         # completed leg sizes between consecutive confirmed pivots, percent of price
    fav = hi_since = lo_since = entry = np.nan

    def add(side, idx, price, t):
        if not pivots:
            pivots.append([side, idx, price, t]); return True
        last = pivots[-1]
        if side == last[0]:
            if (side > 0 and price > last[2]) or (side < 0 and price < last[2]):
                pivots[-1] = [side, idx, price, t]
                if len(pivots) >= 2:
                    leg_pcts[-1] = 100*abs(price-pivots[-2][2])/pivots[-2][2]
                return True
            return False
        if idx <= last[1] or abs(price-last[2]) < min_deviation_atr*atr[t]:
            return False
        pivots.append([side, idx, price, t])
        leg_pcts.append(100*abs(price-last[2])/last[2])
        return True

    for t in range(n):
        changed = False
        u = t-D
        if u >= D:
            wh, wl = high[u-D:u+D+1], low[u-D:u+D+1]
            cands = []
            if high[u] == wh.max() and np.count_nonzero(wh == high[u]) == 1:
                cands.append((1, u, high[u]))
            if low[u] == wl.min() and np.count_nonzero(wl == low[u]) == 1:
                cands.append((-1, u, low[u]))
            if len(cands) == 2 and pivots and pivots[-1][0] > 0:
                cands.reverse()   # alternate away from the last pivot first
            for c_side, c_idx, c_price in cands:
                changed |= add(c_side, c_idx, c_price, t)
        if not pivots:
            continue
        p_side, p_idx, p_price, p_conf = pivots[-1]
        s = -p_side   # after a confirmed low the live leg is LONG (+1); after a high it is SHORT (-1)
        if changed:
            fav = high[p_idx:t+1].max() if s > 0 else low[p_idx:t+1].min()
            entry = close[p_conf]
            hi_since = lo_since = entry
        else:
            fav = max(fav, high[t]) if s > 0 else min(fav, low[t])
            hi_since, lo_since = max(hi_since, high[t]), min(lo_since, low[t])
        a = atr[t]
        atr_pct_t = 100*a/close[t]
        ext_pct = 100*s*(close[t]-p_price)/p_price
        out[t, 0] = 1.0
        out[t, 1] = s
        out[t, 2] = np.log1p(t-p_conf)
        out[t, 3] = np.log1p(t-p_idx)
        out[t, 4] = s*(close[t]-p_price)/a
        out[t, 5] = ((hi_since-entry) if s > 0 else (entry-lo_since))/a
        out[t, 6] = ((entry-lo_since) if s > 0 else (hi_since-entry))/a
        out[t, 7] = ((fav-close[t]) if s > 0 else (close[t]-fav))/a
        if leg_pcts:
            med = float(np.median(leg_pcts[-5:]))
            out[t, 8] = np.log1p(leg_pcts[-1]/atr_pct_t)
            out[t, 9] = np.log1p(med/atr_pct_t)
            out[t, 10] = ext_pct/med if med > 0 else 0.0
        out[t, 11] = min(len(leg_pcts), 5)/5
        side_live[t] = s
        leg_start[t] = p_idx
    return out, side_live, leg_start

def build_features(raw, cfg):
    """V2 81 features + 16 causal leg-state/directional features + live side. Same code for training and inference."""
    base, atr_pct = causal_features(raw, cfg.fractal_depth, cfg.use_structure)
    c = raw.close.to_numpy(np.float64)
    h, l = raw.high.to_numpy(np.float64), raw.low.to_numpy(np.float64)
    atr14 = atr_pct*c/100
    state, zz_side, leg_start = causal_zigzag_state(h, l, c, atr14, cfg.zigzag_depth, cfg.zigzag_min_deviation_atr)
    if cfg.direction_source == "column":
        if cfg.side_column not in raw:
            raise ValueError(f"direction_source='column' needs the live column {cfg.side_column!r} in every prefix.")
        side = parse_side(raw[cfg.side_column])
        source = f"column:{cfg.side_column}"
    else:
        side, source = zz_side.copy(), f"causal_zigzag(depth={cfg.zigzag_depth}, min_dev_atr={cfg.zigzag_min_deviation_atr})"
    cs = pd.Series(c)
    hs, ls = pd.Series(h), pd.Series(l)
    tr = pd.concat([hs-ls, (hs-cs.shift()).abs(), (ls-cs.shift()).abs()], axis=1).max(axis=1)
    atr96 = tr.ewm(alpha=1/96, adjust=False).mean().to_numpy()
    s = side.astype(np.float64)
    extra = {name: state[:, j] for j, name in enumerate(STATE_COLUMNS[:12])}
    extra["zz_side"] = s   # the side actually supplied to the model
    extra["atr_ratio_14_96"] = safe_ratio(atr14, atr96, 1)
    extra["dir_momentum_16_atr"] = s*base["displacement_atr_16"].to_numpy()
    extra["dir_macd_hist_atr"] = s*base["macd_hist_atr"].to_numpy()
    extra["dir_hma_slope_atr"] = s*base["hma_slope_atr"].to_numpy()
    state_frame = pd.DataFrame(extra, index=base.index)[STATE_COLUMNS]
    features = pd.concat([base, state_frame], axis=1).astype(np.float64)
    assert features.shape[1] == BASE_FEATURE_COUNT + len(STATE_COLUMNS)
    return features, atr_pct, side, source, leg_start

@dataclass
class AssetData:
    symbol: str
    raw: pd.DataFrame
    features: pd.DataFrame
    atr_pct: np.ndarray
    y: np.ndarray
    z: np.ndarray
    finite_features: np.ndarray
    side: np.ndarray
    side_source: str
    leg_start: np.ndarray
    info: dict

def prepare_assets(data_files, metadata, cfg):
    cfg.validate()
    if cfg.primary_symbol not in data_files:
        raise ValueError(f"Primary symbol {cfg.primary_symbol} is missing.")
    if len(data_files) > 1:
        raise ValueError("V3 trains on the primary symbol only. Multi-asset pooling is deferred until "
                         "identically generated labels and live-side semantics are verified for each asset.")
    result = {}
    for symbol, path in data_files.items():
        meta = metadata.get(symbol, {})
        if cfg.direction_source == "column" and not meta.get("live_side_verified", False):
            raise ValueError(f"{symbol}: direction_source='column' requires live_side_verified=True provenance.")
        raw, info = load_labeled_csv(path, meta)
        feat, atr_pct, side, source, leg_start = build_features(raw, cfg)
        if np.any(~np.isfinite(atr_pct) | (atr_pct <= 0)):
            raise ValueError(f"{symbol}: nonpositive/nonfinite target ATR.")
        y = raw.tp_pct.to_numpy(dtype=np.float64)
        z = y/atr_pct
        finite = np.isfinite(feat.to_numpy()).all(axis=1)
        if not np.allclose(z[np.isfinite(y)]*atr_pct[np.isfinite(y)], y[np.isfinite(y)], rtol=1e-12, atol=1e-12):
            raise AssertionError("Target inverse failed.")
        info = {**info, "side_source": source}
        result[symbol] = AssetData(symbol, raw, feat, atr_pct, y, z, finite, side, source, leg_start, info)
    return result

### Train-only scaling, target supports, leg-aware sampling and sequence datasets

Scaling, the V2 candidate grid and the distribution bins are fitted on pre-cutoff training anchors only. Training anchors need a label resolved before the cutoff (purge) and inputs that end 128 bars before it (embargo).

**Distribution support.** \(U = \log Z\): 96 uniform core bins between the 0.5% and 99.5% training quantiles, plus a lower and an upper tail bin that extend to half the smallest and twice the largest training target (always at least one core width). Density is uniform in log space inside every bin.

**Samplers** change only which training anchors appear in an epoch. `leg_capped` keeps up to 8 anchors per original leg, one from each early-to-late segment, and rotates them each epoch; `leg_inv_sqrt` draws with probability ∝ 1/√(anchors in leg). Evaluation always counts every eligible candle once. `leg_statistics` reports anchors per leg, the intra-leg correlation of log Z and the resulting effective sample size.

In [7]:
def n_sequence_features(cfg):
    return BASE_FEATURE_COUNT + (len(STATE_COLUMNS) if cfg.use_state_features and cfg.state_in_sequence else 0)

def n_condition_features(cfg):
    return len(STATE_COLUMNS) if cfg.conditioned else 0

def eligible_indices(asset, cfg, start=None, end=None, fit_cutoff=None, labels_mature_before=None):
    """Complete finite causal window. With fit_cutoff: training anchors whose label resolved before the cutoff
    (data-driven purge) and whose inputs end anchor_gap_bars before it (embargo)."""
    good = asset.finite_features.astype(np.int32)
    complete = pd.Series(good).rolling(cfg.context, min_periods=cfg.context).sum().to_numpy() == cfg.context
    mask = complete
    if start is not None:
        mask &= (asset.raw.time >= utc(start)).to_numpy()
    if end is not None:
        mask &= (asset.raw.time < utc(end)).to_numpy()
    if fit_cutoff is not None:
        cutoff = utc(fit_cutoff)
        mask &= (asset.raw.available_at < cutoff-cfg.anchor_gap_bars*BAR).to_numpy()
        mask &= (asset.raw.label_available_at < cutoff).to_numpy()
        mask &= np.isfinite(asset.y)
    if labels_mature_before is not None:
        mask &= (asset.raw.label_available_at < utc(labels_mature_before)).to_numpy() & np.isfinite(asset.y)
    idx = np.flatnonzero(mask)
    cap = cfg.execution_test_max_train_anchors if fit_cutoff is not None else cfg.execution_test_max_eval_anchors
    if cap is not None and len(idx) > cap:
        idx = idx[np.linspace(0, len(idx)-1, cap).astype(int)]  # engineering tests only
    return idx

@dataclass
class FeatureScaler:
    columns: list
    center: np.ndarray
    scale: np.ndarray
    passthrough: np.ndarray

    @classmethod
    def fit(cls, assets, cutoff, cfg):
        samples, counts, available = [], [], {}
        rng = np.random.default_rng(20261030)
        for sym, asset in assets.items():
            mask = asset.finite_features & (asset.raw.available_at < utc(cutoff)-cfg.anchor_gap_bars*BAR).to_numpy()
            available[sym] = np.flatnonzero(mask)
            counts.append(len(available[sym]))
        if not counts or min(counts) == 0:
            raise ValueError("Every asset needs finite pre-cutoff feature rows.")
        n = min(cfg.scaler_samples_per_asset, min(counts))
        for sym, asset in assets.items():
            chosen = rng.choice(available[sym], n, replace=False)
            samples.append(asset.features.iloc[chosen].to_numpy())
        values = np.concatenate(samples)
        columns = next(iter(assets.values())).features.columns.tolist()
        center = np.median(values, axis=0)
        scale = np.maximum(np.percentile(values, 75, axis=0)-np.percentile(values, 25, axis=0), 1e-6)
        keep = np.array([c in PASSTHROUGH_COLUMNS for c in columns])
        center[keep], scale[keep] = 0, 1
        return cls(columns, center, scale, keep)

    def transform(self, features):
        if list(features.columns) != self.columns:
            raise ValueError("Feature order/version mismatch.")
        x = features.to_numpy(dtype=np.float64)
        result = np.clip((x-self.center)/self.scale, -8, 8)
        result[:, self.passthrough] = x[:, self.passthrough]
        return result.astype(np.float32)

    def to_dict(self):
        return {"columns": self.columns, "center": self.center.tolist(),
                "scale": self.scale.tolist(), "passthrough": self.passthrough.tolist()}

    @classmethod
    def from_dict(cls, d):
        return cls(d["columns"], np.array(d["center"]), np.array(d["scale"]), np.array(d["passthrough"], bool))

@dataclass
class CandidateDomain:
    """V2 interval-head candidate grid (log spacing log(4/3)/32)."""
    log_min: float
    log_max: float
    count: int
    lower_ratio: float

    @classmethod
    def fit(cls, values, cfg):
        positive = np.asarray(values, float)
        positive = positive[np.isfinite(positive) & (positive > 0)]
        if not len(positive):
            raise ValueError("At least one positive training target is required.")
        low, high = float(positive.min()*0.5), float(positive.max()*2)
        lo, hi = math.log(low), math.log(high)
        step = math.log(1/cfg.lower_ratio)/cfg.grid_intervals_per_window
        return cls(lo, hi, int(math.ceil((hi-lo)/step))+1, cfg.lower_ratio)

    def log_grid(self):
        return np.linspace(self.log_min, self.log_max, self.count, dtype=np.float64)

    def grid(self):
        return np.exp(self.log_grid())

    def coverage(self, targets):
        target = np.asarray(targets, dtype=np.float64)
        a = self.grid()
        idx = np.searchsorted(a, self.lower_ratio*target, side="left")
        safe = np.minimum(idx, len(a)-1)
        covered = (idx < len(a)) & (a[safe] <= target)
        return covered | (target == 0)

@dataclass
class LogZBins:
    """Histogram support for U = log Z (Z > 0): train-fitted uniform core bins plus explicit tail bins.

    edges has dist_bins + 3 entries: [lower tail edge, core edges..., upper tail edge].
    Density is piecewise uniform in log space inside every bin, including the two tail bins.
    Values outside the outer edges are clamped into the tail bins for the loss (documented support limit).
    """
    edges: list

    @classmethod
    def fit(cls, z_train, cfg):
        z = np.asarray(z_train, float)
        u = np.log(z[np.isfinite(z) & (z > 0)])
        if len(u) < 10:
            raise ValueError("Too few positive training targets for distribution bins.")
        lo, hi = np.quantile(u, cfg.dist_core_quantiles)
        if not hi > lo:
            raise ValueError("Degenerate positive target support.")
        core = np.linspace(lo, hi, cfg.dist_bins+1)
        w = (hi-lo)/cfg.dist_bins
        under = min(lo-w, u.min()-math.log(2))
        over = max(hi+w, u.max()+math.log(2))
        return cls([float(under), *map(float, core), float(over)])

    @property
    def count(self):
        return len(self.edges)-1

    def core_width(self):
        return (self.edges[-2]-self.edges[1])/(self.count-2)

    def frequencies(self, z_train, smoothing=1.0):
        z = np.asarray(z_train, float)
        u = np.log(z[np.isfinite(z) & (z > 0)])
        e = np.asarray(self.edges)
        k = np.clip(np.searchsorted(e, np.clip(u, e[0], e[-1]), side="right")-1, 0, self.count-1)
        counts = np.bincount(k, minlength=self.count)+smoothing
        return counts/counts.sum()

    def soft_targets(self, u, sigma):
        """Gaussian (HL-Gauss) label mass per bin for log targets u [N]; sigma=0 gives one-hot labels."""
        e = torch.tensor(self.edges, dtype=torch.float64, device=u.device)
        u = u.double().clamp(e[0].item(), e[-1].item())
        if sigma <= 0:
            k = (torch.searchsorted(e, u.contiguous(), right=True)-1).clamp(0, self.count-1)
            return F.one_hot(k, self.count).double()
        cdf = torch.special.ndtr((e[None, :]-u[:, None])/sigma)
        mass = (cdf[:, 1:]-cdf[:, :-1]).clamp_min(0)
        return mass/mass.sum(dim=1, keepdim=True).clamp_min(1e-300)

def model_target(asset, cfg):
    return asset.z if cfg.target_mode == "z_atr" else asset.y

@dataclass
class Prepared:
    assets: dict
    scaled: dict
    train_idx: dict
    scaler: FeatureScaler
    domain: CandidateDomain
    bins: LogZBins
    asset_to_id: dict
    fit_cutoff: str

def prepare_fit(assets, cutoff, cfg, asset_to_id=None, scaler=None):
    train_idx = {s: eligible_indices(a, cfg, fit_cutoff=cutoff) for s, a in assets.items()}
    if any(len(v) == 0 for v in train_idx.values()):
        raise ValueError(f"No training anchors for some assets: { {s: len(v) for s, v in train_idx.items()} }")
    scaler = scaler or FeatureScaler.fit(assets, cutoff, cfg)
    scaled = {s: scaler.transform(a.features) for s, a in assets.items()}
    values = np.concatenate([model_target(a, cfg)[train_idx[s]] for s, a in assets.items()])
    domain = CandidateDomain.fit(values, cfg)
    bins = LogZBins.fit(values, cfg)
    vocab = asset_to_id or {s: i+1 for i, s in enumerate(sorted(assets))}
    return Prepared(assets, scaled, train_idx, scaler, domain, bins, vocab, str(utc(cutoff)))

# ---------------- Leg dependence and leg-aware sampling (training only) ----------------

def leg_ids_for(asset, idx):
    legs = asset.raw.leg_id.to_numpy()[idx].copy()
    singles = legs < 0
    legs[singles] = -(np.flatnonzero(singles)+2)   # unknown leg -> its own singleton group
    return legs

def leg_statistics(asset, idx, cap=8):
    """Anchors per original leg, intra-leg target correlation (ICC) and a design-effect sample size."""
    idx = np.asarray(idx)
    legs = leg_ids_for(asset, idx)
    z = asset.z[idx]
    frame = pd.DataFrame({"leg": legs, "z": z, "logz": np.log(np.where(z > 0, z, np.nan))})
    sizes = frame.groupby("leg").size()
    g = frame.dropna().groupby("leg").logz
    grand = frame.logz.mean()
    k = g.ngroups
    n = int(g.size().sum())
    if k > 1 and n > k:
        ss_between = float((g.size()*(g.mean()-grand)**2).sum())
        ss_within = float(((frame.dropna().logz-g.transform("mean"))**2).sum())
        ms_b, ms_w = ss_between/(k-1), ss_within/max(1, n-k)
        n0 = (n-(g.size()**2).sum()/n)/(k-1)
        icc = float((ms_b-ms_w)/(ms_b+(n0-1)*ms_w)) if ms_b+(n0-1)*ms_w > 0 else float("nan")
    else:
        icc = float("nan")
    m = float(sizes.mean())
    design_effect = 1+(m-1)*max(icc, 0) if np.isfinite(icc) else float("nan")
    z_series = pd.Series(z)
    return {
        "anchors": int(len(idx)), "unique_legs": int(len(sizes)),
        "known_leg_anchors": int((asset.raw.leg_id.to_numpy()[idx] >= 0).sum()),
        "anchors_per_leg_mean": m, "anchors_per_leg_median": float(sizes.median()),
        "anchors_per_leg_p90": float(sizes.quantile(.9)), "anchors_per_leg_max": int(sizes.max()),
        "share_anchors_in_legs_over_cap": float(sizes[sizes > cap].sum()/len(idx)), "cap": cap,
        "intra_leg_icc_logz": icc, "design_effect": design_effect,
        "effective_sample_size": float(len(idx)/design_effect) if np.isfinite(design_effect) else None,
        "lag1_autocorrelation_z": float(z_series.autocorr(1)) if len(z) > 2 else None,
        "anchors_per_leg_table": sizes.value_counts().sort_index().head(40).to_dict(),
    }

def training_refs(prepared, cfg, seed, epoch, symbol=None):
    """Per-epoch anchor order. Evaluation weights are never changed: every eligible candle counts once."""
    symbol = symbol or cfg.primary_symbol
    rng = np.random.default_rng(seed+100003*epoch)
    idx = np.asarray(prepared.train_idx[symbol])
    asset = prepared.assets[symbol]
    if cfg.sampler == "uniform":
        chosen = rng.permutation(idx)
    else:
        legs = leg_ids_for(asset, idx)
        order = np.lexsort((idx, legs))
        legs_sorted, idx_sorted = legs[order], idx[order]
        starts = np.r_[0, np.flatnonzero(np.diff(legs_sorted))+1]
        sizes = np.diff(np.r_[starts, len(idx_sorted)])
        size_of = np.repeat(sizes, sizes)
        if cfg.sampler == "leg_capped":
            # Up to leg_cap anchors per leg: one random anchor per equal early/middle/late segment, rotating by epoch.
            position = np.arange(len(idx_sorted))-np.repeat(starts, sizes)
            segment = np.minimum(position*cfg.leg_cap//size_of, cfg.leg_cap-1)
            key = rng.random(len(idx_sorted))
            o2 = np.lexsort((key, segment, legs_sorted))
            first = np.r_[True, (np.diff(legs_sorted[o2]) != 0) | (np.diff(segment[o2]) != 0)]
            chosen = rng.permutation(idx_sorted[o2][first])
        else:  # leg_inv_sqrt: draw with probability proportional to 1/sqrt(anchors in leg)
            w = 1/np.sqrt(size_of)
            chosen = rng.choice(idx_sorted, size=len(idx_sorted), replace=True, p=w/w.sum())
    if cfg.max_train_anchors_per_epoch and len(chosen) > cfg.max_train_anchors_per_epoch:
        chosen = chosen[:cfg.max_train_anchors_per_epoch]
    return [(symbol, int(i)) for i in chosen]

class WindowDataset(Dataset):
    def __init__(self, prepared, refs, cfg):
        self.prepared, self.refs, self.cfg = prepared, refs, cfg
        self.n_seq, self.n_cond = n_sequence_features(cfg), n_condition_features(cfg)
    def __len__(self):
        return len(self.refs)
    def __getitem__(self, i):
        symbol, end = self.refs[i]
        p, cfg = self.prepared, self.cfg
        window = p.scaled[symbol][end-cfg.context+1:end+1]
        x = np.ascontiguousarray(window[:, :self.n_seq])
        if x.shape != (cfg.context, self.n_seq) or not np.isfinite(x).all():
            raise ValueError("A sequence violates the complete causal-window contract.")
        cond = np.ascontiguousarray(window[-1, BASE_FEATURE_COUNT:BASE_FEATURE_COUNT+self.n_cond])
        asset = p.assets[symbol]
        side_index = int(asset.side[end])+1   # 0 short, 1 unknown, 2 long
        return (torch.from_numpy(x), torch.from_numpy(cond), side_index,
                p.asset_to_id.get(symbol, 0), np.float64(model_target(asset, cfg)[end]), end)

def make_loader(prepared, refs, cfg, train=False, seed=0):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(WindowDataset(prepared, refs, cfg),
        batch_size=cfg.batch_size if train else cfg.eval_batch_size,
        shuffle=False, num_workers=cfg.workers, pin_memory=torch.cuda.is_available(),
        generator=generator, persistent_workers=cfg.workers > 0)

### Preserved backbone, direction-conditioned representation and V3 heads

The input projection, seven causal TCN blocks (kernel 5, dilations 1–64), two masked Transformer layers (width 192, 8 heads) and the asset adapter are V2's modules under V2's names.

**Conditioning** (V3.1): \(c_t = \mathrm{MLP}([e_{side}, s_t])\) from the side embedding and the 16 scaled state features at the anchor. FiLM: \(h' = (1+\gamma(c_t)) \odot h_t + \beta(c_t)\). Late fusion: \(h'' = h' + \sigma(W_g[h', c_t]) \odot \mathrm{MLP}([h', c_t])\). Both output layers start at zero, so the conditioned network equals the unconditioned one at initialization.

**Heads.** `zero_head` gives \(p_0 = P(Z=0)\). The V2 interval head is kept for comparison. The distribution head outputs a softmax over the log-Z bins. The auxiliary magnitude head (Linear 192→128, GELU, Linear 128→1) predicts \(\log(1+Z)\); it shapes the representation and never makes the decision.

With conditioning off, `head_type='interval'` and no magnitude head, the module tree is exactly V2's, so `load_v2_bundle` loads saved V2 weights with `strict=True`.

In [8]:
class CausalConv(nn.Module):
    def __init__(self, channels, kernel, dilation):
        super().__init__()
        self.pad = (kernel-1)*dilation
        self.conv = nn.Conv1d(channels, channels, kernel, dilation=dilation)
    def forward(self, x):
        return self.conv(F.pad(x, (self.pad, 0)))

class TCNBlock(nn.Module):
    def __init__(self, width, kernel, dilation, dropout):
        super().__init__()
        self.conv1 = CausalConv(width, kernel, dilation)
        self.conv2 = CausalConv(width, kernel, dilation)
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        y = self.conv1(x).transpose(1, 2)
        y = self.drop(F.gelu(self.norm1(y))).transpose(1, 2)
        y = self.conv2(y).transpose(1, 2)
        y = self.drop(F.gelu(self.norm2(y))).transpose(1, 2)
        return x+y

class EndpointModelV3(nn.Module):
    """V2 TCN-Transformer backbone (module names unchanged) + live-side/leg-state conditioning + V3 heads.

    With use_state_features=False, head_type='interval' and magnitude_head=False the module tree and
    parameter names are identical to V2, so a saved V2 bundle loads with strict=True.
    """
    def __init__(self, cfg, n_assets, domain=None, bins=None):
        super().__init__()
        self.cfg = cfg
        H = cfg.width
        self.proj = nn.Sequential(nn.Linear(n_sequence_features(cfg), H), nn.LayerNorm(H), nn.GELU(), nn.Dropout(.10))
        self.tcn = nn.Sequential(*[TCNBlock(H, cfg.kernel, d, cfg.dropout) for d in cfg.dilations])
        self.transformer = nn.ModuleList([
            nn.TransformerEncoderLayer(H, cfg.heads, H*cfg.ff_multiplier,
                dropout=cfg.dropout, activation="gelu", batch_first=True, norm_first=True)
            for _ in range(cfg.transformer_layers)])
        position = torch.arange(cfg.context, dtype=torch.float32).unsqueeze(1)
        frequency = torch.exp(torch.arange(0, H, 2, dtype=torch.float32)*(-math.log(10000)/H))
        pe = torch.zeros(cfg.context, H)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(position*frequency), torch.cos(position*frequency)
        self.register_buffer("position_encoding", pe, persistent=False)
        self.register_buffer("causal_mask", torch.triu(torch.ones(cfg.context, cfg.context, dtype=torch.bool), diagonal=1), persistent=False)
        self.asset_embedding = nn.Embedding(n_assets, cfg.asset_dim)
        self.adapter_norm = nn.LayerNorm(H)
        self.adapter_down = nn.Linear(H+cfg.asset_dim, cfg.adapter_dim)
        self.adapter_up = nn.Linear(cfg.adapter_dim, H)
        nn.init.zeros_(self.adapter_up.weight)
        nn.init.zeros_(self.adapter_up.bias)
        # ---- V3.1 live side + leg-state conditioning (identity at initialization) ----
        if cfg.conditioned:
            C = cfg.conditioning_hidden
            self.side_embedding = nn.Embedding(3, cfg.side_embedding_dim)
            self.conditioner = nn.Sequential(
                nn.Linear(cfg.side_embedding_dim+n_condition_features(cfg), C), nn.GELU(),
                nn.Linear(C, C), nn.GELU())
            if cfg.use_film:
                self.film = nn.Linear(C, 2*H)
                nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)
            if cfg.use_state_fusion:
                self.fusion_gate = nn.Linear(H+C, H)
                self.fusion = nn.Sequential(nn.LayerNorm(H+C), nn.Linear(H+C, H), nn.GELU(),
                                            nn.Dropout(cfg.head_dropout), nn.Linear(H, H))
                nn.init.zeros_(self.fusion[-1].weight); nn.init.zeros_(self.fusion[-1].bias)
        self.zero_head = nn.Sequential(nn.Linear(H, 64), nn.GELU(), nn.Linear(64, 1))
        # ---- V2 interval head (improved only through shared supervision) ----
        if cfg.head_type == "interval":
            if domain is None:
                raise ValueError("The interval head needs a candidate domain.")
            self.query_embedding = nn.Sequential(nn.Linear(cfg.query_rbf_count+1, 64), nn.GELU(), nn.Linear(64, cfg.query_dim), nn.GELU())
            self.interval_head = nn.Sequential(
                nn.Linear(H+cfg.query_dim, 256), nn.GELU(), nn.Dropout(cfg.head_dropout),
                nn.Linear(256, 128), nn.GELU(), nn.Dropout(cfg.head_dropout), nn.Linear(128, 1))
            self.register_buffer("query_centers", torch.linspace(-1, 1, cfg.query_rbf_count))
            self.register_buffer("log_center", torch.tensor((domain.log_min+domain.log_max)/2, dtype=torch.float32))
            self.register_buffer("log_half_range", torch.tensor((domain.log_max-domain.log_min)/2, dtype=torch.float32))
        # ---- V3.3 coherent distribution head over log Z (+ separate zero probability) ----
        if cfg.head_type == "distribution":
            if bins is None:
                raise ValueError("The distribution head needs fitted log-Z bins.")
            self.dist_head = nn.Sequential(nn.Linear(H, 256), nn.GELU(), nn.Dropout(cfg.head_dropout), nn.Linear(256, bins.count))
            self.register_buffer("bin_edges", torch.tensor(bins.edges, dtype=torch.float64))
        # ---- V3.2 auxiliary magnitude head: log(1 + Z) ----
        if cfg.magnitude_head:
            self.magnitude_head = nn.Sequential(nn.Linear(H, cfg.magnitude_hidden), nn.GELU(), nn.Linear(cfg.magnitude_hidden, 1))

    def encode_sequence(self, x):
        length = x.shape[1]
        if length > self.cfg.context:
            raise ValueError("Window exceeds trained context length.")
        h = self.tcn(self.proj(x).transpose(1, 2)).transpose(1, 2)
        h = h+self.position_encoding[:length]
        for layer in self.transformer:
            h = layer(h, src_mask=self.causal_mask[:length, :length], is_causal=True)
        return h

    def market_state(self, x, asset_ids):
        h = self.encode_sequence(x)[:, -1]
        if self.training and self.cfg.unknown_asset_probability > 0:
            asset_ids = asset_ids.masked_fill(torch.rand_like(asset_ids, dtype=torch.float32) < self.cfg.unknown_asset_probability, 0)
        e = self.asset_embedding(asset_ids)
        return h+self.adapter_up(F.gelu(self.adapter_down(torch.cat([self.adapter_norm(h), e], dim=-1))))

    def condition(self, h, side_index, cond):
        if not self.cfg.conditioned:
            return h
        c = self.conditioner(torch.cat([self.side_embedding(side_index), cond.to(h.dtype)], dim=-1))
        if self.cfg.use_film:
            gamma, beta = self.film(c).chunk(2, dim=-1)
            h = (1+gamma)*h+beta
        if self.cfg.use_state_fusion:
            joint = torch.cat([h, c], dim=-1)
            h = h+torch.sigmoid(self.fusion_gate(joint))*self.fusion(joint)
        return h

    def encode(self, x, asset_ids, side_index=None, cond=None):
        h = self.market_state(x, asset_ids)
        if self.cfg.conditioned:
            h = self.condition(h, side_index, cond)
        return h

    def interval_logits(self, h, log_queries):
        u = (log_queries.float()-self.log_center)/self.log_half_range
        if u.ndim == 1:
            u = u.unsqueeze(0).expand(h.shape[0], -1)
        bandwidth = 4/(self.cfg.query_rbf_count-1)
        rbf = torch.exp(-.5*((u.unsqueeze(-1)-self.query_centers)/bandwidth)**2)
        query = self.query_embedding(torch.cat([u.unsqueeze(-1), rbf], dim=-1))
        state = h.unsqueeze(1).expand(-1, u.shape[1], -1)
        return self.interval_head(torch.cat([state, query], dim=-1)).squeeze(-1)

def build_model(cfg, prepared_like, n_assets=None):
    n_assets = n_assets or len(prepared_like.asset_to_id)+1
    return EndpointModelV3(cfg, n_assets, prepared_like.domain, getattr(prepared_like, "bins", None))

def initialize_output_priors(model, prepared, cfg):
    """Start every head at the training marginal; learning must then produce conditional deviations."""
    values = np.concatenate([model_target(prepared.assets[s], cfg)[idx] for s, idx in prepared.train_idx.items()])
    p0 = np.clip(np.mean(values == 0), 1e-5, 1-1e-5)
    with torch.no_grad():
        model.zero_head[-1].bias.fill_(math.log(p0/(1-p0)))
        if cfg.head_type == "interval":
            positive = np.sort(values[values > 0])
            a = prepared.domain.grid()
            counts = np.searchsorted(positive, a/cfg.lower_ratio, side="right")-np.searchsorted(positive, a, side="left")
            rate = np.clip(np.mean(counts)/len(positive), 1e-5, 1-1e-5)
            model.interval_head[-1].bias.fill_(math.log(rate/(1-rate)))
        if cfg.head_type == "distribution":
            freq = prepared.bins.frequencies(values)
            model.dist_head[-1].bias.copy_(torch.tensor(np.log(freq), dtype=model.dist_head[-1].bias.dtype))
        if cfg.magnitude_head:
            model.magnitude_head[-1].bias.fill_(float(np.mean(np.log1p(values[np.isfinite(values)]))))

def parameter_groups(model):
    """Component name for each parameter: used by the optimizer and gradient monitor."""
    out = {}
    for name, p in model.named_parameters():
        if name.startswith("proj."):
            g = "input_projection"
        elif name.startswith("tcn."):
            g = f"tcn_block_{name.split('.')[1]}"
        elif name.startswith("transformer."):
            g = f"transformer_{name.split('.')[1]}"
        elif name.startswith(("asset_embedding", "adapter_")):
            g = "asset_adapter"
        elif name.startswith(("side_embedding", "conditioner", "film")):
            g = "film_conditioning"
        elif name.startswith(("fusion", "fusion_gate")):
            g = "state_fusion"
        elif name.startswith("zero_head"):
            g = "zero_head"
        elif name.startswith(("query_embedding", "interval_head")):
            g = "interval_head"
        elif name.startswith("dist_head"):
            g = "distribution_head"
        elif name.startswith("magnitude_head"):
            g = "magnitude_head"
        else:
            g = "other"
        out[name] = g
    return out

def load_v2_bundle(path, device=None):
    """Load a saved V2 bundle into the V3 class with a V2-identical module tree (strict=True)."""
    path = Path(path)
    meta = json.loads((path/"manifest.json").read_text())
    if meta.get("feature_version") != V2_FEATURE_VERSION:
        raise ValueError("Not a V2 feature bundle.")
    cfg = config_from_dict(meta["config"], legacy_v2=True, head_type="interval", magnitude_head=False,
                           aux_lambda=0.0, use_state_features=False, use_film=False, use_state_fusion=False,
                           state_in_sequence=False).validate()
    domain = CandidateDomain(**meta["domain"])
    scaler = FeatureScaler.from_dict(meta["scaler"])
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    models = []
    for j in range(meta["ensemble_member_count"]):
        model = EndpointModelV3(cfg, len(meta["asset_to_id"])+1, domain).to(device)
        model.load_state_dict(torch.load(path/f"member_{j}.pt", map_location=device, weights_only=True), strict=True)
        models.append(model.eval())
    return models, cfg, domain, scaler, meta, device

def v2_reference_config(cfg):
    """The retired V2 setup expressed in V3 code: V2 inputs, interval head, no auxiliary loss,
    single learning rate 3e-4, V2 regularization. Used for V3.0 diagnostics and parameter-count checks."""
    return replace(cfg, head_type="interval", aux_lambda=0.0, magnitude_head=False, use_state_features=False,
                   use_film=False, use_state_fusion=False, state_in_sequence=False, dropout=0.15, head_dropout=0.10,
                   weight_decay=5e-4, lr_input=3e-4, lr_tcn=3e-4, lr_transformer=3e-4, lr_adapter=3e-4,
                   lr_film=3e-4, lr_fusion=3e-4, lr_heads=3e-4, lr_aux=3e-4, sampler="uniform")

### Losses and decoders

**Training objective:** \(L = L_{primary} + \lambda\,L_{magnitude}\), with \(L_{magnitude} = \mathrm{Huber}(\hat m, \log(1+Z))\).

- Interval head: V2's exact-event BCE, computed per anchor.
- Distribution head: joint negative log-likelihood, \(-\log p_0\) for a zero target and \(-\log(1-p_0) - \sum_k q_k \log p_k\) otherwise. During training, \(q\) is a Gaussian-smoothed label (σ = 0.75 core-bin widths). Validation loss always uses hard labels.

**Exact distribution decoder.** For a positive candidate \(a\), with \(\Delta = \log\frac43\),
\[P(\text{hit}\mid a) = (1-p_0)\,[F(\log a + \Delta) - F(\log a)].\]
\(F\) is piecewise linear, so \(G(u) = F(u+\Delta) - F(u)\) is piecewise linear with breakpoints only at bin edges and at bin edges minus \(\Delta\). Its maximum is therefore at one of those points, and the decoder evaluates all of them exactly: no grid and no refinement. Zero is chosen when \(p_0\) is at least the best positive mass. Exact ties choose the smaller candidate. Ensemble members are averaged as probability mixtures. Then \(\widehat{tp} = \hat Z \times ATR\%_{14}\).

In [9]:
def amp_context(device, cfg):
    enabled = cfg.amp and device.type == "cuda" and torch.cuda.is_bf16_supported()
    return torch.autocast("cuda", dtype=torch.bfloat16) if enabled else nullcontext()

def strict_hits(actual, predicted, lower_ratio=.75):
    actual, predicted = np.asarray(actual, float), np.asarray(predicted, float)
    finite = np.isfinite(actual) & np.isfinite(predicted)
    tol = 8*np.finfo(np.float64).eps*np.maximum(np.abs(actual), np.abs(predicted))
    return finite & (predicted >= lower_ratio*actual-tol) & (predicted <= actual+tol)

# ---------------- Losses (per anchor, FP32) ----------------

def zero_bce_rows(model, h, targets):
    zero_logit = model.zero_head(h).squeeze(-1).float()
    return F.binary_cross_entropy_with_logits(zero_logit, (targets == 0).float(), reduction="none")

def interval_loss_rows(model, h, targets, log_grid, cfg):
    """V2 exact-event BCE, returned per anchor: zero BCE + mean BCE over the candidate grid (positives)."""
    rows = zero_bce_rows(model, h, targets)
    positive = targets > 0
    if not positive.any():
        return rows
    where = positive.nonzero().squeeze(1)
    hp, yp = h[positive], targets[positive].double()
    acc = torch.zeros(len(yp), device=h.device, dtype=torch.float32)
    for start in range(0, len(log_grid), cfg.grid_chunk):
        u = log_grid[start:start+cfg.grid_chunk]
        if cfg.checkpoint_head and model.training:
            logits = checkpoint(model.interval_logits, hp, u, use_reentrant=False)
        else:
            logits = model.interval_logits(hp, u)
        candidates = torch.exp(u.double())[None, :]
        labels = ((candidates >= cfg.lower_ratio*yp[:, None]) & (candidates <= yp[:, None])).float()
        acc = acc+F.binary_cross_entropy_with_logits(logits.float(), labels, reduction="none").sum(1)
    return rows.index_put((where,), rows[where]+acc/len(log_grid))

def distribution_loss_rows(model, h, targets, bins, cfg, sigma_bins=None):
    """Joint NLL of the coherent distribution: -log p0 (zero) or -log(1-p0) - sum q_k log p_k (positive).

    q is a Gaussian-smoothed (HL-Gauss) label over log-Z bins during training; sigma 0 gives hard-label NLL."""
    rows = zero_bce_rows(model, h, targets)
    positive = targets > 0
    if not positive.any():
        return rows
    where = positive.nonzero().squeeze(1)
    logp = F.log_softmax(model.dist_head(h[positive]).float(), dim=-1)
    sigma = (cfg.dist_label_sigma_bins if sigma_bins is None else sigma_bins)*bins.core_width()
    q = bins.soft_targets(torch.log(targets[positive].double()), sigma).float()
    ce = -(q*logp).sum(-1)
    return rows.index_put((where,), rows[where]+ce)

def magnitude_loss_rows(model, h, targets, cfg):
    m_hat = model.magnitude_head(h).squeeze(-1).float()
    return F.huber_loss(m_hat, torch.log1p(targets.float()), reduction="none", delta=cfg.huber_delta)

def loss_rows(model, h, targets, prepared, log_grid, cfg, training=True):
    """(primary per-anchor loss, auxiliary per-anchor loss or None). Validation uses hard labels."""
    if cfg.head_type == "interval":
        primary = interval_loss_rows(model, h, targets, log_grid, cfg)
    else:
        primary = distribution_loss_rows(model, h, targets, prepared.bins, cfg, None if training else 0.0)
    aux = magnitude_loss_rows(model, h, targets, cfg) if cfg.magnitude_head else None
    return primary, aux

def total_loss(primary, aux, cfg):
    loss = primary.mean()
    if aux is not None and cfg.aux_lambda > 0:
        loss = loss+cfg.aux_lambda*aux.mean()
    return loss

# ---------------- Exact decoder for the distribution head ----------------

def cdf_at(cdf_edges, edges, u):
    """Piecewise-linear CDF (piecewise-uniform density in log space). cdf_edges [B, K+1], edges [K+1], u [M]."""
    K = len(edges)-1
    j = (torch.searchsorted(edges, u.contiguous(), right=True)-1).clamp(0, K-1)
    left, right = edges[j], edges[j+1]
    frac = ((u-left)/(right-left)).clamp(0, 1)
    c0, c1 = cdf_edges[:, j], cdf_edges[:, j+1]
    value = c0+(c1-c0)*frac[None, :]
    value = torch.where(u[None, :] <= edges[0], torch.zeros_like(value), value)
    return torch.where(u[None, :] >= edges[-1], torch.ones_like(value), value)

def distribution_candidates(edges, lower_ratio):
    """Every breakpoint of G(u) = F(u + delta) - F(u): bin edges and bin edges shifted down by delta."""
    delta = math.log(1/lower_ratio)
    return torch.unique(torch.cat([edges, edges-delta]))  # sorted ascending

def decode_distribution(p0, probs, edges, lower_ratio=.75):
    """argmax_a P(0.75 Y <= a <= Y). Exact: G is piecewise linear, so its maximum is at a breakpoint.

    p0 [B] float64, probs [B, K] conditional positive-bin probabilities (float64), edges [K+1].
    Exact ties choose the smallest candidate, including zero.
    """
    delta = math.log(1/lower_ratio)
    cdf_edges = torch.cat([torch.zeros_like(probs[:, :1]), probs.cumsum(1)], dim=1)
    cdf_edges = (cdf_edges/cdf_edges[:, -1:]).clamp(0, 1)
    u = distribution_candidates(edges, lower_ratio)
    g = (cdf_at(cdf_edges, edges, u+delta)-cdf_at(cdf_edges, edges, u)).clamp_min(0)
    positive_scores = (1-p0)[:, None]*g
    best_pos, best_j = positive_scores.max(dim=1)          # first maximum = smallest u
    choose_zero = p0 >= best_pos
    selected = torch.where(choose_zero, torch.zeros_like(p0), torch.exp(u[best_j]))
    score = torch.where(choose_zero, p0, best_pos)
    return selected, score, cdf_edges

def distribution_summaries(cdf_edges, edges, actual_z):
    """Conditional median of Z | Z>0, PIT F(log z) for positive actuals, and entropy proxy are added by caller."""
    target = torch.full((cdf_edges.shape[0],), 0.5, dtype=cdf_edges.dtype, device=cdf_edges.device)
    j = (torch.searchsorted(cdf_edges.contiguous(), target[:, None].contiguous(), right=False).squeeze(1)-1).clamp(0, len(edges)-2)
    c0 = cdf_edges.gather(1, j[:, None]).squeeze(1)
    c1 = cdf_edges.gather(1, (j+1)[:, None]).squeeze(1)
    frac = ((target-c0)/(c1-c0).clamp_min(1e-300)).clamp(0, 1)
    median_u = edges[j]+(edges[j+1]-edges[j])*frac
    pit = torch.full_like(target, float("nan"))
    az = torch.as_tensor(actual_z, dtype=cdf_edges.dtype, device=cdf_edges.device)
    pos = torch.isfinite(az) & (az > 0)
    if pos.any():
        lu = torch.log(az[pos])
        rows = cdf_edges[pos]
        jj = (torch.searchsorted(edges, lu.contiguous(), right=True)-1).clamp(0, len(edges)-2)
        fr = ((lu-edges[jj])/(edges[jj+1]-edges[jj])).clamp(0, 1)
        v = rows.gather(1, jj[:, None]).squeeze(1)+(rows.gather(1, (jj+1)[:, None]).squeeze(1)-rows.gather(1, jj[:, None]).squeeze(1))*fr
        v = torch.where(lu <= edges[0], torch.zeros_like(v), torch.where(lu >= edges[-1], torch.ones_like(v), v))
        pit[pos] = v
    return torch.exp(median_u), pit

# ---------------- V2 interval decoder (grid scan + local refinement) ----------------

def decode_interval(models, states, pzeros, domain, cfg, device):
    u = torch.tensor(domain.log_grid(), dtype=torch.float64, device=device)
    B = states[0].shape[0]
    def score(queries):
        parts = []
        for start in range(0, queries.shape[-1], cfg.grid_chunk):
            qpart = queries[..., start:start+cfg.grid_chunk]
            member = [(1-pz[:, None])*torch.sigmoid(m.interval_logits(h, qpart).float()).double()
                      for m, h, pz in zip(models, states, pzeros)]
            parts.append(torch.stack(member).mean(0))
        return torch.cat(parts, dim=1)
    p0 = torch.stack(pzeros).mean(0)
    scores = score(u)
    valid = torch.isfinite(scores).all(dim=1) & torch.isfinite(p0)
    clean = torch.nan_to_num(scores, nan=-float("inf"), posinf=-float("inf"), neginf=-float("inf"))
    local = torch.ones_like(clean, dtype=torch.bool)
    if domain.count > 1:
        local[:, 1:] &= clean[:, 1:] >= clean[:, :-1]
        local[:, :-1] &= clean[:, :-1] >= clean[:, 1:]
    peak_scores = clean.masked_fill(~local, -float("inf"))
    indices = peak_scores.topk(min(cfg.refine_peaks, domain.count), dim=1).indices
    left, right = u[(indices-1).clamp(min=0)], u[(indices+1).clamp(max=len(u)-1)]
    frac = torch.linspace(0, 1, cfg.refinement_points, dtype=torch.float64, device=device)
    refined_u = (left[..., None]+(right-left)[..., None]*frac).reshape(B, -1)
    refined_scores = score(refined_u)
    valid &= torch.isfinite(refined_scores).all(dim=1)
    candidate_values = torch.cat([torch.zeros((B, 1), dtype=torch.float64, device=device),
                                  torch.exp(u).expand(B, -1), torch.exp(refined_u)], dim=1)
    combined = torch.cat([p0[:, None], clean, refined_scores], dim=1)
    max_score = combined.max(dim=1).values
    selected = candidate_values.masked_fill(combined != max_score[:, None], float("inf")).min(dim=1).values
    selected[~valid], max_score[~valid] = float("nan"), float("nan")
    return selected, max_score

# ---------------- Batch inference shared by validation, evaluation and live prediction ----------------

@torch.no_grad()
def infer_batch(models, x, cond, side, ids, target, prepared_like, cfg, device, with_loss=False):
    """One decision per anchor. Returns numpy arrays. Losses use member 0 and hard labels."""
    states, pzeros, joints, mags = [], [], [], []
    for model in models:
        model.eval()
        with amp_context(device, cfg):
            h = model.encode(x, ids, side, cond)
        h = h.float()
        states.append(h)
        pzeros.append(torch.sigmoid(model.zero_head(h).squeeze(-1).float()).double())
        if cfg.head_type == "distribution":
            probs = torch.softmax(model.dist_head(h).double(), dim=-1)
            joints.append((1-pzeros[-1])[:, None]*probs)
        if cfg.magnitude_head:
            mags.append(model.magnitude_head(h).squeeze(-1).double())
    out = {}
    p0 = torch.stack(pzeros).mean(0)
    out["p_zero"] = p0
    if cfg.head_type == "distribution":
        edges = models[0].bin_edges.to(device=device, dtype=torch.float64)
        joint = torch.stack(joints).mean(0)
        probs = joint/(1-p0)[:, None].clamp_min(1e-300)
        selected, score, cdf_edges = decode_distribution(p0, probs, edges, cfg.lower_ratio)
        median_z, pit = distribution_summaries(cdf_edges, edges, target.double())
        out["dist_median_z"], out["pit"] = median_z, pit
        out["dist_entropy"] = -(probs*torch.log(probs.clamp_min(1e-300))).sum(1)
    else:
        selected, score = decode_interval(models, states, pzeros, prepared_like.domain, cfg, device)
    out["selected"], out["score"] = selected, score
    if mags:
        out["aux_predicted_z"] = torch.expm1(torch.stack(mags).mean(0))
    if with_loss:
        log_grid = torch.tensor(prepared_like.domain.log_grid(), dtype=torch.float64, device=device) if cfg.head_type == "interval" else None
        primary, aux = loss_rows(models[0], states[0], target, prepared_like, log_grid, cfg, training=False)
        out["primary_loss"] = primary
        if aux is not None:
            out["aux_loss"] = aux
    return {k: v.detach().cpu().numpy().astype(np.float64) for k, v in out.items()}

def decodable_coverage(prepared_like, cfg, z):
    """Is any decodable candidate inside [0.75 z, z]? A feasibility ceiling, not achieved accuracy."""
    z = np.asarray(z, float)
    if cfg.head_type == "interval":
        return prepared_like.domain.coverage(z)
    cands = np.sort(np.exp(distribution_candidates(torch.tensor(prepared_like.bins.edges, dtype=torch.float64), cfg.lower_ratio).numpy()))
    idx = np.searchsorted(cands, cfg.lower_ratio*z, side="left")
    safe = np.minimum(idx, len(cands)-1)
    return ((idx < len(cands)) & (cands[safe] <= z)) | (z == 0)

def predict_rows(models, prepared, symbol, indices, cfg, device, rowwise=False, with_loss=False, progress=False):
    """Row-level predictions. rowwise=True evaluates each anchor alone (identical kernel shapes to live use)."""
    indices = np.asarray(indices, dtype=np.int64)
    refs = [(symbol, int(i)) for i in indices]
    loader = make_loader(prepared, refs, cfg, train=False)
    parts = []
    for x, cond, side, ids, target, rows in tqdm(loader, disable=not progress, desc=f"Predict {symbol}"):
        x, cond, side, ids, target = x.to(device), cond.to(device), side.to(device), ids.to(device), target.to(device)
        if rowwise and len(x) > 1:
            for i in range(len(x)):
                parts.append(infer_batch(models, x[i:i+1].contiguous(), cond[i:i+1].contiguous(), side[i:i+1].contiguous(),
                                         ids[i:i+1].contiguous(), target[i:i+1], prepared, cfg, device, with_loss))
        else:
            parts.append(infer_batch(models, x, cond, side, ids, target, prepared, cfg, device, with_loss))
    out = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]} if parts else {}
    a = prepared.assets[symbol]
    pred = out.get("selected", np.empty(0))
    inverse_scale = a.atr_pct[indices] if cfg.target_mode == "z_atr" else np.ones(len(indices))
    raw = a.raw
    result = pd.DataFrame({
        "symbol": symbol, "time": raw.time.iloc[indices].to_numpy(),
        "prediction_available_at": raw.available_at.iloc[indices].to_numpy(),
        "label_available_at": raw.label_available_at.iloc[indices].to_numpy(),
        "side": a.side[indices].astype(int), "label_side": raw.label_side.to_numpy()[indices].astype(int),
        "actual_tp_pct": a.y[indices], "actual_z": a.z[indices], "atr_pct_points": a.atr_pct[indices],
        "predicted_model_target": pred,
        "predicted_z": pred if cfg.target_mode == "z_atr" else pred/a.atr_pct[indices],
        "predicted_tp_pct": pred*inverse_scale, "estimated_strict_hit_probability": out.get("score", np.empty(0)),
        "p_zero": out.get("p_zero", np.empty(0)),
        "leg_age_bars": np.where(a.leg_start[indices] >= 0, indices-a.leg_start[indices], -1),
    })
    for key in ("aux_predicted_z", "dist_median_z", "pit", "dist_entropy", "primary_loss", "aux_loss"):
        if key in out:
            result[key] = out[key]
    if "tp_pivot_time" in raw:
        result["bars_to_endpoint"] = ((raw.tp_pivot_time.iloc[indices]-raw.time.iloc[indices])/BAR).to_numpy()
    result["candidate_covered"] = decodable_coverage(prepared, cfg, model_target(a, cfg)[indices])
    result["strict_hit"] = strict_hits(result.actual_tp_pct, result.predicted_tp_pct, cfg.lower_ratio)
    if cfg.head_type == "interval":
        lo, hi = np.exp(prepared.domain.log_min), np.exp(prepared.domain.log_max)
    else:
        lo, hi = np.exp(prepared.bins.edges[0]-math.log(1/cfg.lower_ratio)), np.exp(prepared.bins.edges[-1])
    result["search_boundary"] = np.isclose(pred, lo, rtol=1e-8, atol=0) | np.isclose(pred, hi, rtol=1e-8, atol=0)
    return result

### Evaluation, prediction collapse statistics, baselines and paired comparisons

Every summary reports strict accuracy on all eligible candles plus collapse statistics: the standard deviation and p1–p99 span of predicted log Z, measured in strict bands of log(4/3), and the Spearman correlation with actual Z. Group reports cover month, target size, ATR regime, live side, label direction, side/label agreement, causal leg age and (diagnostic only) bars to the endpoint. Paired comparisons with fixed ATR use week blocks and, new in V3, original-leg blocks. The V2 confirmed-pivot rule is kept unchanged as a comparator.

In [10]:
DIAG_EPOCH = pd.Timestamp("1970-01-01", tz="UTC")

def spearman(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 3:
        return float("nan")
    ra, rb = pd.Series(a[m]).rank().to_numpy(), pd.Series(b[m]).rank().to_numpy()
    if ra.std() == 0 or rb.std() == 0:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])

def spread_summary(pred_z, actual_z=None):
    """Prediction-collapse statistics in log space; one strict band is log(4/3) = 0.288."""
    z = np.asarray(pred_z, float)
    lz = np.log(z[np.isfinite(z) & (z > 0)])
    out = {"predicted_z_std": float(np.nanstd(z)) if len(z) else None,
           "predicted_log_z_std": float(lz.std()) if len(lz) > 1 else None,
           "predicted_log_p1_p99_span": float(np.subtract(*np.percentile(lz, [99, 1]))) if len(lz) > 1 else None,
           "predicted_z_p1": float(np.percentile(z[np.isfinite(z)], 1)) if np.isfinite(z).any() else None,
           "predicted_z_p99": float(np.percentile(z[np.isfinite(z)], 99)) if np.isfinite(z).any() else None,
           "zero_prediction_rate": float(np.mean(z == 0)) if len(z) else None}
    if out["predicted_log_p1_p99_span"] is not None:
        out["span_in_strict_bands"] = out["predicted_log_p1_p99_span"]/STRICT_BAND_LOG
    if actual_z is not None:
        az = np.asarray(actual_z, float)
        out["actual_z_std"] = float(np.nanstd(az))
        out["spearman_predicted_vs_actual_z"] = spearman(z, az)
    return out

def summarize_predictions(predictions, cfg, bootstrap=False):
    p = predictions
    n = len(p)
    if n == 0:
        raise ValueError("Empty evaluation population.")
    y, pred = p.actual_tp_pct.to_numpy(float), p.predicted_tp_pct.to_numpy(float)
    known, issued = np.isfinite(y), np.isfinite(pred)
    hits = strict_hits(y, pred, cfg.lower_ratio)
    complete = bool(known.all())
    tol = 8*np.finfo(float).eps*np.maximum(np.abs(y), np.abs(pred))
    result = {
        "eligible_rows": n, "known_labels": int(known.sum()), "pending_labels": int((~known).sum()),
        "predictions_produced": int(issued.sum()), "prediction_failures": int((~issued).sum()),
        "coverage": float(issued.mean()), "strict_hits": int(hits.sum()),
        "evaluation_complete": complete,
        "strict_accuracy": float(hits.mean()) if complete else None,
        "observed_label_strict_accuracy": float(hits[known].mean()) if known.any() else None,
        "accuracy_lower_bound_pending": float(hits.sum()/n),
        "accuracy_upper_bound_pending": float((hits.sum()+(~known & issued).sum())/n),
        "underprediction_rate": float(np.mean(known & issued & (pred < cfg.lower_ratio*y-tol))),
        "overprediction_rate": float(np.mean(known & issued & (pred > y+tol))),
        "zero_targets": int((known & (y == 0)).sum()),
        "zero_target_accuracy": float(hits[known & (y == 0)].mean()) if np.any(known & (y == 0)) else None,
        "mae_pct_points": float(np.mean(np.abs(pred[known & issued]-y[known & issued]))) if np.any(known & issued) else None,
        "non_overshoot_rate_not_execution_reach": float(np.mean(known & issued & (pred <= y+tol))) if complete else None,
        "candidate_coverage_ceiling": float(p.candidate_covered[known].mean()) if "candidate_covered" in p and known.any() else None,
        "search_boundary_rate": float(p.search_boundary.mean()) if "search_boundary" in p else None,
        "execution_test_only": cfg.execution_test_max_eval_anchors is not None or cfg.execution_test_max_train_anchors is not None,
    }
    if "predicted_z" in p and "actual_z" in p:
        result.update(spread_summary(p.predicted_z, p.actual_z))
    if "aux_predicted_z" in p:
        result["spearman_aux_magnitude_vs_actual_z"] = spearman(p.aux_predicted_z, p.actual_z)
    score = p.estimated_strict_hit_probability.to_numpy(float) if "estimated_strict_hit_probability" in p else np.full(n, np.nan)
    valid_score = known & issued & np.isfinite(score)
    result["selected_probability_brier"] = float(np.mean((score[valid_score]-hits[valid_score])**2)) if valid_score.any() else None
    result["mean_selected_probability"] = float(np.mean(score[valid_score])) if valid_score.any() else None
    if "primary_loss" in p:
        result["validation_primary_loss"] = float(p.primary_loss.mean())
    if "aux_loss" in p:
        result["validation_aux_loss"] = float(p.aux_loss.mean())
    if "pit" in p and p.pit.notna().any():
        hist = np.histogram(p.pit.dropna(), bins=10, range=(0, 1))[0]/p.pit.notna().sum()
        result["pit_max_decile_deviation"] = float(np.max(np.abs(hist-.1)))
    result["strict_accuracy_95_block_ci"] = block_bootstrap_ci(p, cfg) if bootstrap and complete else None
    return result

def block_bootstrap_ci(predictions, cfg):
    times = pd.to_datetime(predictions.time, utc=True)
    block = ((times-DIAG_EPOCH) // pd.Timedelta(days=cfg.bootstrap_block_days)).to_numpy()
    hits = strict_hits(predictions.actual_tp_pct, predictions.predicted_tp_pct, cfg.lower_ratio)
    groups = pd.DataFrame({"block": block, "hit": hits.astype(int)}).groupby("block").hit.agg(["sum", "count"]).to_numpy()
    if len(groups) < 4:
        return {"low": None, "high": None, "calendar_blocks": len(groups), "reason": "Fewer than four calendar blocks"}
    rng = np.random.default_rng(12345)
    sampled = groups[rng.integers(len(groups), size=(cfg.bootstrap_repetitions, len(groups)))].sum(axis=1)
    rates = sampled[:, 0]/sampled[:, 1]
    return {"low": float(np.percentile(rates, 2.5)), "high": float(np.percentile(rates, 97.5)),
            "calendar_blocks": len(groups), "block_days": cfg.bootstrap_block_days}

def add_report_groups(predictions):
    p = predictions.copy()
    dates = pd.to_datetime(p.time, utc=True).dt.tz_localize(None)
    p["month"] = dates.dt.to_period("M").astype(str)
    p["quarter"] = dates.dt.to_period("Q").astype(str)
    p["target_band"] = pd.cut(p.actual_tp_pct, [-np.inf, 0, .1, .25, .5, 1, 2, 5, np.inf],
        labels=["zero", "(0,0.1]", "(0.1,0.25]", "(0.25,0.5]", "(0.5,1]", "(1,2]", "(2,5]", ">5"])
    p["tp_size"] = pd.qcut(p.actual_tp_pct.rank(method="first"), 3, labels=["small", "medium", "large"])
    p["volatility_regime"] = pd.qcut(p.atr_pct_points.rank(method="first"), 5,
                                     labels=["atr_q1_low", "atr_q2", "atr_q3", "atr_q4", "atr_q5_high"])
    if "side" in p:
        p["live_side"] = p.side.map({1: "long", -1: "short", 0: "unknown"})
    if "label_side" in p:
        p["label_direction"] = p.label_side.map({1: "long", -1: "short", 0: "unknown"})
        if "side" in p:
            p["live_side_matches_label"] = np.where(p.side == p.label_side, "match", "mismatch")
    if "leg_age_bars" in p:
        p["causal_leg_age"] = pd.cut(p.leg_age_bars, [-np.inf, -1, 12, 24, 48, 96, 192, np.inf],
            labels=["no_state", "0-12", "13-24", "25-48", "49-96", "97-192", ">192"])
    if "bars_to_endpoint" in p:
        p["bars_to_endpoint_label_diag"] = pd.cut(p.bars_to_endpoint, [0, 4, 16, 48, 96, 192, np.inf],
            labels=["1-4", "5-16", "17-48", "49-96", "97-192", ">192"])
    return p

REPORT_KEYS = ("month", "quarter", "target_band", "tp_size", "volatility_regime", "live_side", "label_direction",
               "live_side_matches_label", "causal_leg_age", "bars_to_endpoint_label_diag")

def grouped_reports(predictions, cfg, keys=REPORT_KEYS):
    p = add_report_groups(predictions)
    reports = {}
    for key in keys:
        if key not in p:
            continue
        rows = [{key: str(group), **summarize_predictions(frame, cfg)} for group, frame in p.groupby(key, observed=True)]
        reports[key] = pd.DataFrame(rows)
    return reports

def best_constant(values, lower_ratio=.75):
    a = np.asarray(values, dtype=np.float64)
    a = a[np.isfinite(a)]
    if not len(a):
        raise ValueError("No labels to fit the constant.")
    zero_count = int((a == 0).sum())
    positive = np.sort(a[a > 0])
    if not len(positive):
        return 0.0
    candidates = np.unique(positive)
    counts = np.searchsorted(positive, candidates/lower_ratio, side="right")-np.searchsorted(positive, candidates, side="left")
    best = int(np.argmax(counts))
    return 0.0 if zero_count >= counts[best] else float(candidates[best])

def constant_baseline_rows(prepared, symbol, indices, cfg):
    a = prepared.assets[symbol]
    idx = prepared.train_idx[symbol]
    constants = {"constant_raw_tp": best_constant(a.y[idx], cfg.lower_ratio),
                 "constant_atr_units": best_constant(a.z[idx], cfg.lower_ratio)}
    result = {}
    for name, value in constants.items():
        pred = np.full(len(indices), value)
        if name == "constant_atr_units":
            pred = pred*a.atr_pct[indices]
        frame = pd.DataFrame({
            "symbol": symbol, "time": a.raw.time.iloc[indices].to_numpy(),
            "side": a.side[indices].astype(int), "label_side": a.raw.label_side.to_numpy()[indices].astype(int),
            "actual_tp_pct": a.y[indices], "actual_z": a.z[indices], "atr_pct_points": a.atr_pct[indices],
            "predicted_tp_pct": pred, "predicted_z": pred/a.atr_pct[indices],
            "estimated_strict_hit_probability": np.nan,
        })
        frame["strict_hit"] = strict_hits(frame.actual_tp_pct, frame.predicted_tp_pct, cfg.lower_ratio)
        result[name] = {"fitted_constant": value, "metrics": summarize_predictions(frame, cfg, bootstrap=True), "predictions": frame}
    return result

# ---------------- Paired comparisons (from V2 diagnostics) ----------------

def hit_breakdown(actual, pred, lower_ratio=.75):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    hits = strict_hits(actual, pred, lower_ratio)
    tol = 8*np.finfo(float).eps*np.maximum(np.abs(actual), np.abs(pred))
    return {"strict_hits": int(hits.sum()), "strict_accuracy": float(hits.mean()),
            "below_band_rate": float(np.mean(pred < lower_ratio*actual-tol)),
            "above_actual_rate": float(np.mean(pred > actual+tol)),
            "median_pred_over_actual": float(np.median(pred[actual > 0]/actual[actual > 0]))}

def week_blocks(times, cfg):
    t = pd.to_datetime(pd.Series(times), utc=True)
    return ((t-DIAG_EPOCH).dt.total_seconds()//(86400*cfg.bootstrap_block_days)).astype(np.int64).to_numpy()

def accuracy_block_ci(times, hits, cfg):
    g = pd.DataFrame({"b": week_blocks(times, cfg), "h": np.asarray(hits, int)}).groupby("b").h.agg(["sum", "count"]).to_numpy()
    rng = np.random.default_rng(12345)
    s = g[rng.integers(len(g), size=(cfg.bootstrap_repetitions, len(g)))].sum(axis=1)
    r = s[:, 0]/s[:, 1]
    return float(np.percentile(r, 2.5)), float(np.percentile(r, 97.5))

def paired_block_bootstrap(times, hits_a, hits_b, cfg, blocks=None):
    """Accuracy difference A-B in percentage points, resampling whole calendar weeks (or supplied blocks, e.g. legs)."""
    b = week_blocks(times, cfg) if blocks is None else np.asarray(blocks)
    d = pd.DataFrame({"b": b, "a": np.asarray(hits_a, int), "c": np.asarray(hits_b, int)})
    g = d.groupby("b").agg(a=("a", "sum"), c=("c", "sum"), n=("a", "size")).to_numpy()
    rng = np.random.default_rng(12345)
    s = g[rng.integers(len(g), size=(cfg.bootstrap_repetitions, len(g)))].sum(axis=1)
    diff = 100*(s[:, 0]-s[:, 1])/s[:, 2]
    return {"difference_pp": float(100*(d.a.sum()-d.c.sum())/len(d)),
            "ci95_low_pp": float(np.percentile(diff, 2.5)), "ci95_high_pp": float(np.percentile(diff, 97.5)),
            "bootstrap_share_not_better": float(np.mean(diff <= 0)), "blocks": int(len(g))}

def mcnemar_exact_p(b, c):
    """Two-sided exact McNemar test on discordant pairs. Ignores serial dependence; read with the block CI."""
    n, k = b+c, min(b, c)
    if n == 0:
        return 1.0
    logs = np.array([math.lgamma(n+1)-math.lgamma(i+1)-math.lgamma(n-i+1)-n*math.log(2) for i in range(k+1)])
    m = logs.max()
    return float(min(1.0, 2*math.exp(m)*np.exp(logs-m).sum()))

def paired_report(name_a, hits_a, name_b, hits_b, times, cfg, legs=None):
    a, b = np.asarray(hits_a, bool), np.asarray(hits_b, bool)
    only_a, only_b = int((a & ~b).sum()), int((~a & b).sum())
    out = {"model": name_a, "reference": name_b,
           "both_hit": int((a & b).sum()), "only_model_hit": only_a, "only_reference_hit": only_b,
           "neither_hit": int((~a & ~b).sum()), "hit_difference": only_a-only_b,
           "mcnemar_exact_p_iid": mcnemar_exact_p(only_a, only_b),
           **paired_block_bootstrap(times, a, b, cfg)}
    if legs is not None:
        leg_ci = paired_block_bootstrap(times, a, b, cfg, blocks=legs)
        out["leg_block_ci95_low_pp"], out["leg_block_ci95_high_pp"] = leg_ci["ci95_low_pp"], leg_ci["ci95_high_pp"]
    return out

def prediction_spread_report(p, fixed_multiple, cfg):
    z = p.predicted_z.to_numpy(float)
    az = p.actual_z.to_numpy(float)
    band = math.log(1/cfg.lower_ratio)
    lz, laz = np.log(z[z > 0]), np.log(az[az > 0])
    pct = [0, 1, 5, 25, 50, 75, 95, 99, 100]
    def log_width(v, hi, lo):
        return float(np.subtract(*np.percentile(v, [hi, lo]))) if len(v) >= 2 else float("nan")
    log_iqr, log_span = log_width(lz, 75, 25), log_width(lz, 99, 1)
    rel = np.abs(np.log(np.where(z > 0, z, np.nan)/fixed_multiple))
    report = {
        "rows": int(len(z)), "zero_predictions": int((z == 0).sum()),
        "distinct_values_rounded_1e-4": int(len(np.unique(np.round(z, 4)))),
        "predicted_z_quantiles": {f"p{q}": float(v) for q, v in zip(pct, np.percentile(z, pct))},
        "actual_z_quantiles": {f"p{q}": float(v) for q, v in zip(pct, np.percentile(az, pct))},
        "predicted_z_std": float(z.std()), "actual_z_std": float(az.std()),
        "predicted_log_iqr": log_iqr, "actual_log_iqr": log_width(laz, 75, 25),
        "predicted_log_p1_p99_span": log_span, "strict_band_log_width": band,
        "log_iqr_as_share_of_band": log_iqr/band, "fixed_multiple": float(fixed_multiple),
        "share_within_5pct_of_fixed_multiple": float(np.nanmean(rel <= math.log(1.05))),
        "share_within_one_band_of_fixed_multiple": float(np.nanmean(rel <= band)),
        "spearman_predicted_vs_actual_z": spearman(z, az),
    }
    if len(lz) < 2:
        report["verdict"] = "fewer_than_two_positive_predictions"
    else:
        report["verdict"] = ("effectively_constant" if log_span < band and log_iqr < .25*band
                             else "varies_less_than_one_band" if log_span < band else "varies")
    return report

def decile_table(p, key, hits_model, hits_reference, cfg, bins=10):
    """Rows grouped by a model output. The per-decile best constant is an IN-SAMPLE ceiling, not a result."""
    d = p.assign(_m=np.asarray(hits_model, bool), _r=np.asarray(hits_reference, bool))
    d["decile"] = pd.qcut(d[key].rank(method="first"), bins, labels=False)+1
    rows = []
    for k, g in d.groupby("decile"):
        c = best_constant(g.actual_z.to_numpy(float), cfg.lower_ratio)
        ceiling = strict_hits(g.actual_z, np.full(len(g), c), cfg.lower_ratio)
        rows.append({"decile": int(k), "rows": len(g), f"{key}_min": float(g[key].min()), f"{key}_max": float(g[key].max()),
                     "mean_predicted_z": float(g.predicted_z.mean()), "median_actual_z": float(g.actual_z.median()),
                     "model_accuracy": float(g._m.mean()), "fixed_atr_accuracy": float(g._r.mean()),
                     "model_minus_fixed_hits": int(g._m.sum()-g._r.sum()),
                     "in_sample_best_z": float(c), "in_sample_ceiling_accuracy": float(ceiling.mean())})
    return pd.DataFrame(rows)

def ordering_ceiling(order_values, actual_z, cfg, bins=10):
    d = pd.DataFrame({"o": order_values, "z": actual_z})
    d["k"] = pd.qcut(d.o.rank(method="first"), bins, labels=False)
    hits = 0
    for _, g in d.groupby("k"):
        c = best_constant(g.z.to_numpy(float), cfg.lower_ratio)
        hits += int(strict_hits(g.z, np.full(len(g), c), cfg.lower_ratio).sum())
    return hits/len(d)

def random_ordering_ceiling(actual_z, cfg, repetitions=20, bins=10):
    rng = np.random.default_rng(20261031)
    return float(np.mean([ordering_ceiling(rng.permutation(len(actual_z)), actual_z, cfg, bins) for _ in range(repetitions)]))

# ---------------- Confirmed-pivot ATR comparator (V2 rule, unchanged) ----------------

def pivot_state(asset):
    """Uses the V2 causal depth-12 fractal features; a pivot is known only after its right-hand bars close."""
    f = asset.features
    direction = f.swing_direction.to_numpy()
    both = (f.has_swing_high.to_numpy() > 0) & (f.has_swing_low.to_numpy() > 0)
    side = np.where(both & (direction > 0), "low", np.where(both & (direction < 0), "high", "none"))
    age = np.where(side == "low", np.expm1(f.swing_low_age.to_numpy()),
                   np.where(side == "high", np.expm1(f.swing_high_age.to_numpy()), np.nan))
    dist = np.where(side == "low", f.swing_low_distance_atr.to_numpy(),
                    np.where(side == "high", -f.swing_high_distance_atr.to_numpy(), np.nan))
    return pd.DataFrame({"side": side, "age_bars": np.round(age), "distance_atr": dist})

def _bin(values, edges):
    return np.searchsorted(edges, values, side="right")

def assign_pivot_buckets(state, edges):
    out = np.full(len(state), "none", dtype=object)
    for side, e in edges.items():
        m = (state.side == side).to_numpy()
        a = _bin(state.age_bars.to_numpy()[m], np.asarray(e["age"]))
        d = _bin(state.distance_atr.to_numpy()[m], np.asarray(e["distance"]))
        out[m] = [f"{side}|age{i}|dist{j}" for i, j in zip(a, d)]
    return out

def fit_pivot_rule(asset, train_idx, cfg, age_bins=3, distance_bins=3, min_bucket=500):
    st = pivot_state(asset).iloc[train_idx].reset_index(drop=True)
    z = asset.z[train_idx]
    global_c = best_constant(z, cfg.lower_ratio)
    edges, buckets = {}, {}
    for side in ("low", "high"):
        m = (st.side == side).to_numpy()
        if not m.any():
            continue
        edges[side] = {"age": np.unique(np.quantile(st.age_bars[m], np.linspace(0, 1, age_bins+1)[1:-1])).tolist(),
                       "distance": np.unique(np.quantile(st.distance_atr[m], np.linspace(0, 1, distance_bins+1)[1:-1])).tolist()}
    labels = assign_pivot_buckets(st, edges)
    for b in np.unique(labels):
        zb = z[labels == b]
        c = best_constant(zb, cfg.lower_ratio) if len(zb) >= min_bucket else global_c
        buckets[b] = {"train_rows": int(len(zb)), "multiple": float(c), "pooled_fallback": bool(len(zb) < min_bucket),
                      "train_accuracy": float(strict_hits(zb, np.full(len(zb), c), cfg.lower_ratio).mean())}
    return {"edges": edges, "buckets": buckets, "global_multiple": float(global_c),
            "age_bins": age_bins, "distance_bins": distance_bins, "min_bucket": min_bucket, "fitted_on_rows": int(len(train_idx))}

def apply_pivot_rule(rule, asset, idx):
    st = pivot_state(asset).iloc[idx].reset_index(drop=True)
    labels = assign_pivot_buckets(st, rule["edges"])
    mult = np.array([rule["buckets"].get(b, {"multiple": rule["global_multiple"]})["multiple"] for b in labels])
    return mult, labels, st

### Optimizer groups, gradient monitoring and resumable training

| Component | Learning rate |
|---|---|
| Input projection, TCN blocks, Transformer layers | 1e-4 |
| Asset adapter, FiLM conditioner, state fusion, zero/interval/distribution heads, magnitude head | 3e-4 |

AdamW with decay 0.005 on matrices only, 5% warm-up then cosine decay, accumulation ×4, global clipping at 1.0 and EMA 0.995 are kept. Backpropagation is unchanged (`loss.backward()`, then clip, then step). `GradientMonitor` reads gradients **after backward and before clipping**: the global pre-clip norm, the clipping frequency, and per-component gradient norm, parameter norm and gradient-to-parameter ratio. Every `loss_grad_probe_every` updates it measures the gradients of the primary and λ-weighted auxiliary losses on shared parameters, giving their norm ratio and cosine.

Each epoch scores the validation set **and** a fixed training probe with the EMA weights. Early stopping and the saved checkpoint use validation primary loss; strict accuracy, spread and Spearman are logged alongside, so collapse that appears only on unseen data is visible.

In [11]:
class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, value in model.state_dict().items():
            if value.is_floating_point():
                self.shadow[k].lerp_(value.detach(), 1-self.decay)
            else:
                self.shadow[k].copy_(value)
    def apply(self, model):
        model.load_state_dict(self.shadow)

def state_cpu(state):
    return {k: v.detach().cpu().clone() for k, v in state.items()}

def rng_snapshot():
    return {"cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}

def restore_rng(state):
    torch.set_rng_state(state["cpu_rng"].cpu())
    if torch.cuda.is_available() and state.get("cuda_rng"):
        torch.cuda.set_rng_state_all([x.cpu() for x in state["cuda_rng"]])

COMPONENT_LR = {
    "input_projection": "lr_input", "asset_adapter": "lr_adapter", "film_conditioning": "lr_film",
    "state_fusion": "lr_fusion", "zero_head": "lr_heads", "interval_head": "lr_heads",
    "distribution_head": "lr_heads", "magnitude_head": "lr_aux",
}
SHARED_COMPONENT_PREFIXES = ("input_projection", "tcn_block", "transformer", "asset_adapter", "film_conditioning", "state_fusion")

def component_lr(component, cfg):
    if component.startswith("tcn_block"):
        return cfg.lr_tcn
    if component.startswith("transformer"):
        return cfg.lr_transformer
    return getattr(cfg, COMPONENT_LR.get(component, "lr_heads"))

def optimizer_for(model, cfg):
    """AdamW with per-component learning rates; decay on matrices only (no bias/norm/embedding decay)."""
    groups = {}
    comp = parameter_groups(model)
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        lr = component_lr(comp[name], cfg)
        decay = cfg.weight_decay if param.ndim >= 2 and "embedding" not in name else 0.0
        groups.setdefault((comp[name], lr, decay), []).append(param)
    return torch.optim.AdamW([{"params": p, "lr": lr, "weight_decay": d, "component": c}
                              for (c, lr, d), p in groups.items()])

class GradientMonitor:
    """Gradient quality after backward and BEFORE clipping: per-component norms, ratios and clipping frequency."""
    def __init__(self, model, cfg):
        self.cfg = cfg
        self.component = parameter_groups(model)
        self.reset()

    def reset(self):
        self.preclip, self.records, self.loss_probes = [], [], []
        self.clipped = self.nonfinite = 0

    @torch.no_grad()
    def observe(self, model, preclip_norm, update_index, detailed):
        value = float(preclip_norm)
        self.preclip.append(value)
        self.clipped += int(value > self.cfg.gradient_clip)
        self.nonfinite += int(not math.isfinite(value))
        if not detailed:
            return
        grad_sq, param_sq, zero_grads, total = {}, {}, {}, {}
        for name, p in model.named_parameters():
            c = self.component[name]
            param_sq[c] = param_sq.get(c, 0.0)+float(p.detach().float().pow(2).sum())
            total[c] = total.get(c, 0)+1
            if p.grad is None:
                zero_grads[c] = zero_grads.get(c, 0)+1
                continue
            g = float(p.grad.detach().float().pow(2).sum())
            grad_sq[c] = grad_sq.get(c, 0.0)+g
            zero_grads[c] = zero_grads.get(c, 0)+int(g == 0)
        for c in total:
            gn, pn = math.sqrt(grad_sq.get(c, 0.0)), math.sqrt(param_sq[c])
            self.records.append({"update": update_index, "component": c, "grad_norm": gn, "param_norm": pn,
                                 "grad_to_param": gn/pn if pn > 0 else float("nan"),
                                 "zero_grad_tensor_share": zero_grads.get(c, 0)/total[c],
                                 "preclip_global_norm": value, "clipped": value > self.cfg.gradient_clip})

    def probe_losses(self, model, primary, aux, update_index):
        """Relative gradient contribution of primary and lambda-weighted auxiliary losses on shared parameters."""
        if aux is None or self.cfg.aux_lambda == 0:
            return
        shared = [p for n, p in model.named_parameters()
                  if p.requires_grad and self.component[n].startswith(SHARED_COMPONENT_PREFIXES)]
        gp = torch.autograd.grad(primary.mean(), shared, retain_graph=True, allow_unused=True)
        ga = torch.autograd.grad(self.cfg.aux_lambda*aux.mean(), shared, retain_graph=True, allow_unused=True)
        vp = torch.cat([(g if g is not None else torch.zeros_like(p)).float().flatten() for g, p in zip(gp, shared)])
        va = torch.cat([(g if g is not None else torch.zeros_like(p)).float().flatten() for g, p in zip(ga, shared)])
        npn, nan_ = float(vp.norm()), float(va.norm())
        self.loss_probes.append({"update": update_index, "primary_grad_norm": npn, "aux_weighted_grad_norm": nan_,
                                 "aux_to_primary_ratio": nan_/npn if npn > 0 else float("nan"),
                                 "cosine": float(torch.dot(vp, va)/(vp.norm()*va.norm()).clamp_min(1e-30))})

    def summary(self):
        out = {"updates": len(self.preclip),
               "clip_fraction": self.clipped/max(1, len(self.preclip)),
               "nonfinite_updates": self.nonfinite,
               "preclip_norm_median": float(np.median(self.preclip)) if self.preclip else None,
               "preclip_norm_p95": float(np.percentile(self.preclip, 95)) if self.preclip else None}
        if self.records:
            r = pd.DataFrame(self.records).groupby("component")
            out["grad_norm_mean"] = r.grad_norm.mean().to_dict()
            out["grad_to_param_mean"] = r.grad_to_param.mean().to_dict()
        if self.loss_probes:
            lp = pd.DataFrame(self.loss_probes)
            out["aux_to_primary_grad_ratio_mean"] = float(lp.aux_to_primary_ratio.mean())
            out["primary_aux_grad_cosine_mean"] = float(lp.cosine.mean())
        return out

def epoch_diagnostics(prefix, frame, cfg):
    m = summarize_predictions(frame, cfg)
    keep = ("strict_accuracy", "validation_primary_loss", "validation_aux_loss", "predicted_log_z_std",
            "predicted_log_p1_p99_span", "span_in_strict_bands", "spearman_predicted_vs_actual_z",
            "spearman_aux_magnitude_vs_actual_z", "zero_prediction_rate", "mean_selected_probability")
    return {f"{prefix}_{k.replace('validation_', '')}": m.get(k) for k in keep if k in m}

def fixed_probe_indices(prepared, cfg, n):
    idx = np.asarray(prepared.train_idx[cfg.primary_symbol])
    return idx if len(idx) <= n else idx[np.linspace(0, len(idx)-1, n).astype(int)]

def archive_stale_run_dir(output_dir):
    """Move a run directory whose checkpoint no longer matches aside (never deleted) so training can start fresh."""
    output_dir = Path(output_dir)
    stale = output_dir.with_name(f"{output_dir.name}__stale_{time.strftime('%Y%m%d_%H%M%S')}")
    k = 1
    while stale.exists():
        stale = output_dir.with_name(f"{output_dir.name}__stale_{time.strftime('%Y%m%d_%H%M%S')}_{k}"); k += 1
    output_dir.rename(stale)
    output_dir.mkdir(parents=True, exist_ok=True)
    return stale

def train_run(model, prepared, cfg, device, epochs, seed, output_dir, validation=None, tag="v3", on_mismatch=None):
    """Train with AdamW groups, warm-up/cosine, accumulation, clipping and EMA. Resumable.

    With validation: early stopping and checkpoint selection on validation PRIMARY LOSS (hard labels);
    strict accuracy, prediction spread and Spearman are logged every epoch for both validation and a fixed
    training probe, so collapse on unseen data versus training data is visible.

    on_mismatch: what to do when output_dir holds a checkpoint from a different config/data/scaler/bins.
    'raise' stops; 'archive' renames the old directory to <name>__stale_<timestamp> and trains from scratch.
    Default: the notebook-level ON_CHECKPOINT_MISMATCH setting ('raise' if it is not defined).
    """
    on_mismatch = on_mismatch or globals().get("ON_CHECKPOINT_MISMATCH", "raise")
    if on_mismatch not in ("raise", "archive"):
        raise ValueError("on_mismatch must be 'raise' or 'archive'.")
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    latest, best_path = output_dir/"latest.pt", output_dir/"best.pt"
    metadata = {
        "config": asdict(cfg), "epochs": epochs, "seed": seed, "tag": tag,
        "fit_cutoff": prepared.fit_cutoff, "scaler": prepared.scaler.to_dict(),
        "domain": asdict(prepared.domain), "bins": asdict(prepared.bins),
        "feature_version": FEATURE_VERSION, "implementation_version": IMPLEMENTATION_VERSION,
        "sources": {s: a.info["source_sha256"] for s, a in prepared.assets.items()},
        "validation_indices": list(map(int, validation)) if validation is not None else None,
    }
    signature = hashlib.sha256(json.dumps(json_safe(metadata), sort_keys=True).encode()).hexdigest()
    n_samples = len(training_refs(prepared, cfg, seed, 0))
    updates_per_epoch = math.ceil(math.ceil(n_samples/cfg.batch_size)/cfg.gradient_accumulation)
    planned_updates = max(1, epochs*updates_per_epoch)
    warmup = max(1, int(cfg.warmup_fraction*planned_updates))
    def multiplier(step):
        if step < warmup:
            return (step+1)/warmup
        progress = min(1., (step-warmup)/max(1, planned_updates-warmup))
        return .1+.9*.5*(1+math.cos(math.pi*progress))
    optimizer = optimizer_for(model, cfg)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)
    ema = EMA(model, cfg.ema_decay)
    eval_model = copy.deepcopy(model).eval()
    monitor = GradientMonitor(model, cfg)
    log_grid = torch.tensor(prepared.domain.log_grid(), dtype=torch.float64, device=device)
    probe_idx = fixed_probe_indices(prepared, cfg, cfg.train_probe_anchors)
    start_epoch = resume_batch = global_updates = bad_checks = 0
    best_loss, best_epoch, best_accuracy = float("inf"), 0, None
    history, running = [], {"sum": 0.0, "primary": 0.0, "aux": 0.0, "count": 0}
    saved = None
    if cfg.resume and latest.exists():
        saved = torch.load(latest, map_location=device, weights_only=True)
        if saved["signature"] != signature:
            if on_mismatch != "archive":
                raise ValueError(f"Checkpoint/config mismatch in {output_dir}. Use a new run directory, "
                                 "or set ON_CHECKPOINT_MISMATCH = 'archive' in the configuration cell.")
            stale = archive_stale_run_dir(output_dir)
            print(f"[{tag}] checkpoint in {output_dir} was made with a different config/data/scaler/bins; "
                  f"moved it to {stale.name} and training from scratch.")
            saved = None
    if saved is not None:
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        ema.shadow = saved["ema"]
        start_epoch, resume_batch = saved["epoch"], saved["next_batch"]
        global_updates, bad_checks = saved["global_updates"], saved["bad_checks"]
        best_loss, best_epoch, best_accuracy = saved["best_loss"], saved["best_epoch"], saved["best_accuracy"]
        history, running = saved["history"], saved["running"]
        restore_rng(saved)
        if saved.get("complete", False):
            state = torch.load(best_path, map_location=device, weights_only=True)
            return state["model"], state["info"]
        print(f"Resuming {tag}: epoch {start_epoch+1}, batch {resume_batch}")

    def save_progress(epoch, next_batch, complete=False):
        atomic_torch_save({
            "signature": signature, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "ema": ema.shadow, "epoch": epoch, "next_batch": next_batch,
            "global_updates": global_updates, "best_loss": best_loss, "best_epoch": best_epoch,
            "best_accuracy": best_accuracy, "bad_checks": bad_checks, "history": history,
            "running": running, "complete": complete, **rng_snapshot()}, latest)
        write_json(output_dir/"history.json", history)

    for epoch in range(start_epoch, epochs):
        if epoch != start_epoch:
            resume_batch, running = 0, {"sum": 0.0, "primary": 0.0, "aux": 0.0, "count": 0}
        monitor.reset()
        refs = training_refs(prepared, cfg, seed, epoch)
        loader = make_loader(prepared, refs, cfg, train=True, seed=seed+epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        started = time.time()
        for batch_index, (x, cond, side, ids, target, _) in enumerate(tqdm(loader, desc=f"{tag} epoch {epoch+1}/{epochs}", leave=False)):
            if batch_index < resume_batch:
                continue
            x, cond, side, ids, target = x.to(device), cond.to(device), side.to(device), ids.to(device), target.to(device)
            group_start = (batch_index//cfg.gradient_accumulation)*cfg.gradient_accumulation
            group_samples = min(len(refs)-group_start*cfg.batch_size, cfg.gradient_accumulation*cfg.batch_size)
            with amp_context(device, cfg):
                h = model.encode(x, ids, side, cond)
                primary, aux = loss_rows(model, h, target, prepared, log_grid, cfg, training=True)
                loss = total_loss(primary, aux, cfg)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite training loss; no failed batch is silently skipped.")
            first_in_group = batch_index % cfg.gradient_accumulation == 0
            if first_in_group and cfg.loss_grad_probe_every and global_updates % cfg.loss_grad_probe_every == 0:
                monitor.probe_losses(model, primary, aux, global_updates)
            (loss*(len(x)/group_samples)).backward()
            running["sum"] += float(loss.detach())*len(x); running["count"] += len(x)
            running["primary"] += float(primary.detach().mean())*len(x)
            running["aux"] += float(aux.detach().mean())*len(x) if aux is not None else 0.0
            if (batch_index+1) % cfg.gradient_accumulation == 0 or batch_index+1 == len(loader):
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip, error_if_nonfinite=True)
                monitor.observe(model, norm, global_updates, detailed=global_updates % max(1, cfg.grad_log_every) == 0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
                ema.update(model); global_updates += 1
                if cfg.checkpoint_every_updates and global_updates % cfg.checkpoint_every_updates == 0:
                    save_progress(epoch, batch_index+1)
        count = max(1, running["count"])
        record = {"epoch": epoch+1, "train_loss": running["sum"]/count, "train_primary_loss": running["primary"]/count,
                  "train_aux_loss": running["aux"]/count if cfg.magnitude_head else None,
                  "seconds": time.time()-started, "learning_rate_scale": multiplier(global_updates)}
        grad = monitor.summary()
        record.update({f"grad_{k}": v for k, v in grad.items()})
        if monitor.records:
            pd.DataFrame(monitor.records).to_csv(output_dir/f"gradients_epoch_{epoch+1:03d}.csv", index=False)
        if monitor.loss_probes:
            pd.DataFrame(monitor.loss_probes).to_csv(output_dir/f"loss_gradient_probe_epoch_{epoch+1:03d}.csv", index=False)
        ema.apply(eval_model)
        probe = predict_rows([eval_model], prepared, cfg.primary_symbol, probe_idx, cfg, device, with_loss=True)
        record.update(epoch_diagnostics("train_probe", probe, cfg))
        if validation is not None:
            vp = predict_rows([eval_model], prepared, cfg.primary_symbol, validation, cfg, device, with_loss=True)
            vm = summarize_predictions(vp, cfg)
            if not vm["evaluation_complete"] or vm["coverage"] != 1:
                raise ValueError("Validation must have mature labels and a prediction for every selected anchor.")
            record.update(epoch_diagnostics("val", vp, cfg))
            vloss = vm["validation_primary_loss"]
            if vloss < best_loss:
                best_loss, best_epoch, best_accuracy, bad_checks = vloss, epoch+1, vm["strict_accuracy"], 0
                atomic_torch_save({"model": state_cpu(ema.shadow), "info": {
                    "best_epoch": best_epoch, "best_validation_loss": best_loss,
                    "validation_strict_accuracy_at_best": best_accuracy, "tag": tag}}, best_path)
                vp.to_csv(output_dir/"best_validation_predictions.csv", index=False)
            else:
                bad_checks += 1
        else:
            best_epoch = epoch+1
            atomic_torch_save({"model": state_cpu(ema.shadow), "info": {
                "best_epoch": best_epoch, "best_validation_loss": None,
                "validation_strict_accuracy_at_best": None, "tag": tag}}, best_path)
        record = json_safe(record)
        history.append(record)
        print(json.dumps(json_safe({k: record[k] for k in record if not isinstance(record[k], dict)})))
        resume_batch, running = 0, {"sum": 0.0, "primary": 0.0, "aux": 0.0, "count": 0}
        stop = validation is not None and bad_checks >= cfg.patience
        save_progress(epoch+1, 0, complete=stop or epoch+1 == epochs)
        if stop:
            break
    saved = torch.load(best_path, map_location=device, weights_only=True)
    del optimizer, scheduler, ema, eval_model
    return saved["model"], {**saved["info"], "history_path": str(output_dir/"history.json")}

### Development windows, staged selection, refit and model bundles

Each window trains on labels resolved before its start and validates the following 60 days. Validation labels must have matured before the outer period. The defaults are 2025-09-01, 2026-01-01 and 2026-04-01. The selected configuration is refit on all data before `EVAL_START` for the median best epoch across windows.

**Selection rule, declared before training.** (1) Pooled strict accuracy across windows; options within 0.25 pp of the best tie. (2) Among ties, the higher minimum per-window lift over fixed ATR. (3) Then lower mean validation loss, only when the tied options share a head type. (4) Then the incumbent. Validation loss stops training; it never selects across heads. Jul–Oct 2026 stays development history and never picks a configuration.

In [12]:
PREPARATION_FIELDS = ("context", "anchor_gap_bars", "grid_intervals_per_window", "dist_bins", "dist_core_quantiles",
                      "lower_ratio", "target_mode", "scaler_samples_per_asset", "fractal_depth", "use_structure",
                      "direction_source", "side_column", "zigzag_depth", "zigzag_min_deviation_atr",
                      "execution_test_max_train_anchors", "execution_test_max_eval_anchors")

def config_key(cfg):
    return hashlib.sha256(json.dumps(json_safe(asdict(cfg)), sort_keys=True).encode()).hexdigest()[:12]

def config_delta(cfg, reference):
    a, b = json_safe(asdict(cfg)), json_safe(asdict(reference))
    return {k: a[k] for k in a if a[k] != b.get(k)}

@dataclass
class DevWindow:
    start: str
    end: str
    fit: Prepared
    val_idx: np.ndarray
    fixed_atr_multiple: float

def prepare_dev_windows(assets, cfg, starts, outer_start):
    """Chronological development windows: train on labels resolved before each start (purge + embargo),
    validate on the next dev_validation_days, using only labels that matured before the outer period."""
    primary = assets[cfg.primary_symbol]
    windows = []
    for s in starts:
        start = utc(s)
        end = start+pd.Timedelta(days=cfg.dev_validation_days)
        if end > utc(outer_start)-cfg.anchor_gap_bars*BAR:
            raise ValueError(f"Development window {s} overlaps the outer evaluation period.")
        fit = prepare_fit(assets, start, cfg)
        val_idx = eligible_indices(primary, cfg, start=start, end=end-cfg.anchor_gap_bars*BAR, labels_mature_before=outer_start)
        if not len(val_idx):
            raise ValueError(f"No mature validation anchors for window {s}.")
        fixed = best_constant(primary.z[fit.train_idx[cfg.primary_symbol]], cfg.lower_ratio)
        windows.append(DevWindow(str(start), str(end), fit, val_idx, fixed))
    return windows

def windows_data_key(windows):
    """Hash of what each window trains/validates on: source files, fit cutoff, scaler, bins and validation rows."""
    payload = [{"start": w.start, "fit_cutoff": w.fit.fit_cutoff,
                "sources": {s: a.info["source_sha256"] for s, a in w.fit.assets.items()},
                "scaler": w.fit.scaler.to_dict(), "bins": asdict(w.fit.bins), "domain": asdict(w.fit.domain),
                "validation": hashlib.sha256(np.asarray(w.val_idx, np.int64).tobytes()).hexdigest()} for w in windows]
    return hashlib.sha256(json.dumps(json_safe(payload), sort_keys=True).encode()).hexdigest()[:16]

def train_candidate_on_windows(cfg, windows, seed, output_dir, device):
    """Train one configuration on every development window. Cached by configuration hash."""
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    summary_path = out/"candidate_summary.json"
    if summary_path.is_file():
        cached = json.loads(summary_path.read_text())
        if (cached.get("config_key") == config_key(cfg) and cached.get("data_key") == windows_data_key(windows)
                and [w["start"] for w in cached["windows"]] == [w.start for w in windows]):
            return cached
        print(f"Cached summary in {out} is for different config/data; re-checking its window runs.")
    rows = []
    for w_i, w in enumerate(windows):
        wdir = out/f"window_{w_i+1}_{w.start[:10]}"
        seed_all(seed)
        model = build_model(cfg, w.fit).to(device)
        initialize_output_priors(model, w.fit, cfg)
        _, info = train_run(model, w.fit, cfg, device, cfg.max_epochs, seed, wdir, validation=w.val_idx,
                            tag=f"{config_key(cfg)}/w{w_i+1}")
        del model
        vp = pd.read_csv(wdir/"best_validation_predictions.csv")
        history = json.loads((wdir/"history.json").read_text())
        best = next(h for h in history if h["epoch"] == info["best_epoch"])
        asset = w.fit.assets[cfg.primary_symbol]
        fixed_hits = strict_hits(asset.z[w.val_idx], np.full(len(w.val_idx), w.fixed_atr_multiple), cfg.lower_ratio)
        hits = vp.strict_hit.to_numpy(bool)
        rows.append({
            "window": w_i+1, "start": w.start, "end": w.end, "validation_rows": int(len(vp)),
            "best_epoch": int(info["best_epoch"]), "validation_primary_loss": float(info["best_validation_loss"]),
            "strict_hits": int(hits.sum()), "strict_accuracy": float(hits.mean()),
            "fixed_atr_multiple": w.fixed_atr_multiple, "fixed_atr_hits": int(fixed_hits.sum()),
            "fixed_atr_accuracy": float(fixed_hits.mean()),
            "lift_vs_fixed_atr_pp": float(100*(hits.mean()-fixed_hits.mean())),
            "train_probe_strict_accuracy": best.get("train_probe_strict_accuracy"),
            "generalization_gap_pp": (100*(best["train_probe_strict_accuracy"]-hits.mean())
                                      if best.get("train_probe_strict_accuracy") is not None else None),
            "val_predicted_log_z_std": best.get("val_predicted_log_z_std"),
            "train_probe_predicted_log_z_std": best.get("train_probe_predicted_log_z_std"),
            "val_spearman": best.get("val_spearman_predicted_vs_actual_z"),
            "val_aux_spearman": best.get("val_spearman_aux_magnitude_vs_actual_z"),
            "clip_fraction_best_epoch": best.get("grad_clip_fraction"),
        })
    table = pd.DataFrame(rows)
    table.to_csv(out/"windows.csv", index=False)
    gaps = table.generalization_gap_pp.dropna()
    summary = {
        "config_key": config_key(cfg), "data_key": windows_data_key(windows), "config": asdict(cfg),
        "pooled_strict_accuracy": float(table.strict_hits.sum()/table.validation_rows.sum()),
        "pooled_fixed_atr_accuracy": float(table.fixed_atr_hits.sum()/table.validation_rows.sum()),
        "mean_lift_pp": float(table.lift_vs_fixed_atr_pp.mean()),
        "min_lift_pp": float(table.lift_vs_fixed_atr_pp.min()),
        "windows_beating_fixed_atr": int((table.lift_vs_fixed_atr_pp > 0).sum()),
        "mean_validation_primary_loss": float(table.validation_primary_loss.mean()),
        "mean_generalization_gap_pp": float(gaps.mean()) if len(gaps) else None,
        "best_epochs": table.best_epoch.tolist(), "windows": rows,
    }
    write_json(summary_path, summary)
    return summary

def choose_by_rule(candidates, summaries, tie_pp):
    """Predeclared rule: (1) pooled strict accuracy across windows; candidates within tie_pp of the best are tied;
    (2) higher minimum per-window lift over fixed ATR (stability); (3) lower mean validation primary loss, only
    when all tied candidates share a head type; (4) earlier option (the incumbent is listed first)."""
    best = max(s["pooled_strict_accuracy"] for s in summaries)
    tied = [i for i, s in enumerate(summaries) if s["pooled_strict_accuracy"] >= best-tie_pp/100]
    if len(tied) == 1:
        return tied[0], "highest pooled strict accuracy"
    top_lift = max(summaries[i]["min_lift_pp"] for i in tied)
    tied = [i for i in tied if summaries[i]["min_lift_pp"] >= top_lift-1e-12]
    if len(tied) == 1:
        return tied[0], "tie on accuracy; higher minimum window lift"
    if len({candidates[i].head_type for i in tied}) == 1:
        lowest = min(summaries[i]["mean_validation_primary_loss"] for i in tied)
        tied = [i for i in tied if summaries[i]["mean_validation_primary_loss"] <= lowest+1e-12]
        if len(tied) == 1:
            return tied[0], "tie on accuracy and lift; lower validation loss"
    return tied[0], "full tie; earlier (incumbent) option kept"

def run_staged_selection(assets, cfg, base_overrides, stages, windows, output_dir, device, gap_trigger_pp):
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    incumbent = replace(cfg, **base_overrides).validate()
    reference = incumbent
    results, log = {}, []
    seed = cfg.seeds[0]
    def evaluate(c):
        bad = [f for f in PREPARATION_FIELDS if getattr(c, f) != getattr(cfg, f)]
        if bad:
            raise ValueError(f"Selection options may not change data-preparation fields: {bad}")
        key = config_key(c)
        if key not in results:
            print(f"\n=== Candidate {key}: {config_delta(c, reference)}")
            results[key] = train_candidate_on_windows(c, windows, seed, out/f"candidate_{key}", device)
        return results[key]
    evaluate(incumbent)
    for stage_name, options in stages:
        if stage_name.endswith("_if_gap"):
            gap = results[config_key(incumbent)]["mean_generalization_gap_pp"]
            if gap is None or gap < gap_trigger_pp:
                log.append({"stage": stage_name, "skipped": True, "reason":
                            f"mean train-probe minus validation accuracy {gap} pp < trigger {gap_trigger_pp} pp"})
                continue
        candidates = [replace(incumbent, **opt).validate() for opt in options]
        if config_key(incumbent) not in {config_key(c) for c in candidates}:
            candidates = [incumbent]+candidates
        summaries = [evaluate(c) for c in candidates]
        winner, reason = choose_by_rule(candidates, summaries, cfg.selection_tie_pp)
        log.append({"stage": stage_name, "skipped": False, "reason": reason,
                    "winner": config_key(candidates[winner]), "winner_delta": config_delta(candidates[winner], reference),
                    "options": [{"key": config_key(c), "delta": config_delta(c, reference),
                                 "pooled_strict_accuracy": s["pooled_strict_accuracy"],
                                 "pooled_fixed_atr_accuracy": s["pooled_fixed_atr_accuracy"],
                                 "mean_lift_pp": s["mean_lift_pp"], "min_lift_pp": s["min_lift_pp"],
                                 "mean_validation_primary_loss": s["mean_validation_primary_loss"],
                                 "mean_generalization_gap_pp": s["mean_generalization_gap_pp"]}
                                for c, s in zip(candidates, summaries)]})
        incumbent = candidates[winner]
        write_json(out/"selection_log.json", log)
    final = results[config_key(incumbent)]
    plan = {
        "selected_config": asdict(incumbent), "selected_key": config_key(incumbent),
        "refit_epochs": int(max(1, round(float(np.median(final["best_epochs"]))))),
        "refit_epoch_rule": "median best-validation-loss epoch across development windows",
        "selection_rule": choose_by_rule.__doc__, "development_windows": [(w.start, w.end) for w in windows],
        "selection_log": log, "selected_summary": {k: v for k, v in final.items() if k != "config"},
        "created_at": str(pd.Timestamp.now(tz="UTC")),
    }
    write_json(out/"v3_frozen_plan.json", plan)
    return incumbent, plan, results

# ---------------- Refit, bundle and evaluation ----------------

def save_bundle(path, models, prepared, cfg, plan, eval_start, metadata):
    path = Path(path); path.mkdir(parents=True, exist_ok=True)
    manifest = {
        "feature_version": FEATURE_VERSION, "implementation_version": IMPLEMENTATION_VERSION,
        "config": asdict(cfg), "scaler": prepared.scaler.to_dict(),
        "domain": asdict(prepared.domain), "bins": asdict(prepared.bins), "asset_to_id": prepared.asset_to_id,
        "fit_cutoff": prepared.fit_cutoff, "evaluation_start": str(utc(eval_start)),
        "plan": plan, "versions": package_versions(),
        "data": {s: a.info for s, a in prepared.assets.items()},
        "ensemble_member_count": len(models), **metadata,
    }
    for j, model in enumerate(models):
        atomic_torch_save(state_cpu(model.state_dict()), path/f"member_{j}.pt")
    write_json(path/"manifest.json", manifest)
    return path

@dataclass
class BundleContext:
    domain: CandidateDomain
    bins: LogZBins
    asset_to_id: dict

def load_bundle(path, device=None):
    path = Path(path)
    meta = json.loads((path/"manifest.json").read_text())
    if meta["feature_version"] != FEATURE_VERSION or meta.get("implementation_version") != IMPLEMENTATION_VERSION:
        raise ValueError("Bundle version differs from this notebook. Use load_v2_bundle for V2 artifacts.")
    cfg = config_from_dict(meta["config"]).validate()
    ctx = BundleContext(CandidateDomain(**meta["domain"]), LogZBins(**meta["bins"]), meta["asset_to_id"])
    scaler = FeatureScaler.from_dict(meta["scaler"])
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    models = []
    for j in range(meta["ensemble_member_count"]):
        model = EndpointModelV3(cfg, len(meta["asset_to_id"])+1, ctx.domain, ctx.bins).to(device)
        model.load_state_dict(torch.load(path/f"member_{j}.pt", map_location=device, weights_only=True), strict=True)
        models.append(model.eval())
    return models, cfg, ctx, scaler, meta, device

def refit_models(assets, cfg, plan, eval_start, output_dir, device):
    prepared = prepare_fit(assets, eval_start, cfg)
    models = []
    for seed in cfg.seeds:
        seed_all(seed)
        model = build_model(cfg, prepared).to(device)
        initialize_output_priors(model, prepared, cfg)
        state, _ = train_run(model, prepared, cfg, device, plan["refit_epochs"], seed,
                             Path(output_dir)/f"refit_seed_{seed}", validation=None, tag=f"refit/{seed}")
        model.load_state_dict(state); model.eval()
        models.append(model)
    return models, prepared

def side_ablation_prepared(prepared, cfg, mode, seed=20261101):
    """Counterfactual direction input: flip or shuffle the live side (and every side-signed input) for all rows.
    Leg geometry from the causal ZigZag is left unchanged. Used to measure what the side input contributes."""
    sym = cfg.primary_symbol
    a = prepared.assets[sym]
    side = -a.side if mode == "flip" else np.random.default_rng(seed).permutation(a.side)
    feat = a.features.copy()
    s = side.astype(np.float64)
    feat["zz_side"] = s
    feat["dir_momentum_16_atr"] = s*feat["displacement_atr_16"].to_numpy()
    feat["dir_macd_hist_atr"] = s*feat["macd_hist_atr"].to_numpy()
    feat["dir_hma_slope_atr"] = s*feat["hma_slope_atr"].to_numpy()
    asset = replace(a, side=side.astype(np.int8), features=feat)
    return replace(prepared, assets={sym: asset}, scaled={sym: prepared.scaler.transform(feat)})

def evaluate_models(models, prepared, cfg, eval_start, eval_end, output_dir, device, plan=None, tag="v3"):
    """V3.5 report on every eligible candle: metrics, groups, baselines, calibration, side ablation."""
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    sym = cfg.primary_symbol
    asset = prepared.assets[sym]
    idx = eligible_indices(asset, cfg, start=eval_start, end=eval_end)
    if not len(idx):
        raise ValueError("No evaluation anchors at the requested dates.")
    predictions = predict_rows(models, prepared, sym, idx, cfg, device, rowwise=cfg.rowwise_final_decode,
                               with_loss=True, progress=True)
    predictions.to_csv(out/"predictions.csv", index=False)
    metrics = summarize_predictions(predictions, cfg, bootstrap=True)
    verified = all(a.info["maturity_status"] == "verified_by_supplied_provenance" for a in prepared.assets.values())
    metrics.update({
        "evaluation_role": "execution_test" if metrics["execution_test_only"] else cfg.run_mode,
        "label_maturity_verified": verified, "uninspected_test_claim": cfg.run_mode == "final",
        "meets_strict_accuracy_target": bool(metrics["strict_accuracy"] is not None and metrics["strict_accuracy"] >= cfg.minimum_accuracy and metrics["coverage"] == 1),
        "final_acceptance_strict_accuracy": bool(cfg.run_mode == "final" and verified and metrics["evaluation_complete"]
                                                 and metrics["coverage"] == 1 and metrics["strict_accuracy"] >= cfg.minimum_accuracy),
        "execution_reach_gate": "measured separately in the V3.5 reach section",
        "seeds": list(cfg.seeds), "head_type": cfg.head_type,
    })
    for name, report in grouped_reports(predictions, cfg).items():
        report.to_csv(out/f"metrics_by_{name}.csv", index=False)
    baselines = constant_baseline_rows(prepared, sym, idx, cfg)
    write_json(out/"constant_baselines.json", {k: {"fitted_constant": v["fitted_constant"], "metrics": v["metrics"]} for k, v in baselines.items()})
    fixed = baselines["constant_atr_units"]
    fixed_hits = fixed["predictions"].strict_hit.to_numpy(bool)
    legs = asset.raw.leg_id.to_numpy()[idx]
    metrics["paired_vs_fixed_atr"] = paired_report(tag, predictions.strict_hit, "fixed_atr", fixed_hits,
                                                   predictions.time, cfg, legs=legs)
    metrics["prediction_spread"] = prediction_spread_report(predictions, fixed["fitted_constant"], cfg)
    if cfg.conditioned or (cfg.use_state_features and cfg.state_in_sequence):
        batched = predict_rows(models, prepared, sym, idx, cfg, device, rowwise=False)
        ablation = {"batched_reference_accuracy": float(batched.strict_hit.mean())}
        for mode in ("flip", "shuffle"):
            ap = predict_rows(models, side_ablation_prepared(prepared, cfg, mode), sym, idx, cfg, device, rowwise=False)
            ablation[f"{mode}_side_accuracy"] = float(ap.strict_hit.mean())
            ablation[f"{mode}_side_accuracy_drop_pp"] = 100*(ablation["batched_reference_accuracy"]-ablation[f"{mode}_side_accuracy"])
            ablation[f"{mode}_side_prediction_change_rate"] = float(np.mean(~np.isclose(ap.predicted_z, batched.predicted_z, rtol=1e-6)))
        metrics["side_ablation"] = ablation
    if plan is not None:
        metrics["plan_key"] = plan.get("selected_key")
    write_json(out/"metrics.json", metrics)
    return predictions, metrics, baselines

### Saved-model inference, causality audits and contract tests

`EndpointPredictor` recomputes features, ZigZag state and live side from the full raw prefix and applies the saved scaler and bins; it never fits anything. The audits check prefix replay at fixed and random cut points for all 97 inputs, the side and the leg start; invariance to future-price perturbation and to every hindsight column; and causal masking of the encoder. Contract tests check the strict metric, distribution normalization and tails, one-hot and smoothed labels, point-mass decoding, the exact decoder against dense brute force, a Monte Carlo hit rate and ZigZag prefix replay.

In [13]:
class EndpointPredictor:
    """Keep loaded; pass the full causal raw prefix (from the saved feature origin) for exact indicator/ZigZag state."""
    def __init__(self, bundle_path, device=None):
        self.models, self.cfg, self.ctx, self.scaler, self.meta, self.device = load_bundle(bundle_path, device)

    def predict_latest(self, raw_prefix, symbol):
        raw = validate_raw_frame(raw_prefix)
        if symbol in self.meta["data"]:
            if raw.time.iloc[0] != utc(self.meta["data"][symbol]["start"]):
                raise ValueError("Supply history from the saved feature origin. A short rolling CSV resets ATR/RSI/ZigZag state.")
        features, atr_pct, side, source, leg_start = build_features(raw, self.cfg)
        cfg = self.cfg
        if len(features) < cfg.context:
            raise ValueError("Insufficient sequence context.")
        window = self.scaler.transform(features)[-cfg.context:]
        x = window[:, :n_sequence_features(cfg)]
        cond = window[-1, BASE_FEATURE_COUNT:BASE_FEATURE_COUNT+n_condition_features(cfg)]
        if not np.isfinite(x).all() or not np.isfinite(atr_pct[-1]) or atr_pct[-1] <= 0:
            raise ValueError("Insufficient indicator warm-up or invalid current inputs.")
        d = self.device
        out = infer_batch(self.models, torch.from_numpy(np.ascontiguousarray(x))[None].to(d),
                          torch.from_numpy(np.ascontiguousarray(cond))[None].to(d),
                          torch.tensor([int(side[-1])+1], dtype=torch.long, device=d),
                          torch.tensor([self.meta["asset_to_id"].get(symbol, 0)], dtype=torch.long, device=d),
                          torch.tensor([float("nan")], dtype=torch.float64, device=d), self.ctx, cfg, d)
        z_hat = float(out["selected"][0])
        prediction = z_hat*atr_pct[-1] if cfg.target_mode == "z_atr" else z_hat
        result = {
            "symbol": symbol, "time": str(raw.time.iloc[-1]),
            "prediction_available_at": str(raw.available_at.iloc[-1]),
            "side": int(side[-1]), "side_label": {1: "LONG", -1: "SHORT", 0: "UNKNOWN"}[int(side[-1])],
            "side_source": source, "predicted_z": float(prediction/atr_pct[-1]),
            "atr_pct_points": float(atr_pct[-1]), "predicted_tp_pct": float(prediction),
            "estimated_strict_hit_probability": float(out["score"][0]), "p_zero": float(out["p_zero"][0]),
            "prediction_status": "produced" if np.isfinite(prediction) else "failed",
            "head_type": cfg.head_type, "feature_version": FEATURE_VERSION, "model_version": "V3",
            "model_fit_cutoff": self.meta["fit_cutoff"],
        }
        if "aux_predicted_z" in out:
            result["aux_predicted_z"] = float(out["aux_predicted_z"][0])
        if "dist_median_z" in out:
            result["distribution_median_z"] = float(out["dist_median_z"][0])
        return result

def audit_features(asset, cfg, positions=None, random_cuts=3, seed=20261101):
    """Prefix replay equality (fixed + random cut points), future-price perturbation and label-column exclusion
    for all 97 inputs, the live side and the ZigZag leg start."""
    n = len(asset.raw)
    rng = np.random.default_rng(seed)
    fixed = [min(n-1, 800), n//2, n-1]
    extra = rng.integers(min(n-1, 400), n-1, size=random_cuts).tolist() if n > 401 else []
    positions = sorted(set(positions or fixed+extra))
    columns = ["time", *RAW_COLUMNS] + ([cfg.side_column] if cfg.direction_source == "column" else [])
    base_raw = asset.raw[columns+[c for c in ("tp_pct", "tp_price", "tp_pct_signed", "tp_value", "signal", "signal_live", "zigzag")
                                  if c in asset.raw and c not in columns]].copy()
    for t in positions:
        prefix, atr, side, _, leg = build_features(base_raw.iloc[:t+1], cfg)
        np.testing.assert_allclose(prefix.iloc[-1].to_numpy(), asset.features.iloc[t].to_numpy(), atol=1e-10, rtol=1e-10, equal_nan=True)
        np.testing.assert_allclose(atr[-1], asset.atr_pct[t], rtol=1e-12, atol=1e-12)
        assert side[-1] == asset.side[t] and leg[-1] == asset.leg_start[t], f"Side/leg state differs at prefix {t}"
    t = max(200, min(n//2, n-2))
    changed = base_raw.copy()
    changed.loc[t+1:, ["open", "high", "low", "close"]] *= 1.7
    f, atr, side, _, leg = build_features(changed, cfg)
    np.testing.assert_allclose(f.iloc[:t+1], asset.features.iloc[:t+1], atol=1e-10, rtol=1e-10, equal_nan=True)
    np.testing.assert_allclose(atr[:t+1], asset.atr_pct[:t+1], atol=1e-12, rtol=1e-12)
    assert np.array_equal(side[:t+1], asset.side[:t+1]) and np.array_equal(leg[:t+1], asset.leg_start[:t+1])
    changed = base_raw.copy()
    for c in ("tp_pct", "tp_pct_signed", "tp_price", "tp_value"):
        if c in changed:
            changed[c] = 999999.
    for c in ("signal", "zigzag") + (("signal_live",) if cfg.direction_source != "column" else ()):
        if c in changed:
            changed[c] = "modified_future_label"
    f2, _, side2, _, _ = build_features(changed, cfg)
    np.testing.assert_allclose(f2, asset.features, atol=1e-10, rtol=1e-10, equal_nan=True)
    assert np.array_equal(side2, asset.side)
    return {"prefix_positions_tested": positions, "future_perturbation_passed": True,
            "label_input_exclusion_passed": True, "features_tested": int(asset.features.shape[1]), "side_source": asset.side_source}

def audit_model_causality(model, cfg, device):
    model.eval()
    length = min(cfg.context, 128)
    prefix_len = length//2
    gen = torch.Generator(device="cpu").manual_seed(20261030)
    x = torch.randn(2, length, n_sequence_features(cfg), generator=gen).to(device)
    changed = x.clone()
    changed[:, prefix_len:] += 5
    with torch.no_grad():
        full = model.encode_sequence(x)
        prefix = model.encode_sequence(x[:, :prefix_len])
        altered = model.encode_sequence(changed)
    torch.testing.assert_close(full[:, :prefix_len], prefix, atol=2e-5, rtol=2e-5)
    torch.testing.assert_close(full[:, :prefix_len], altered[:, :prefix_len], atol=2e-5, rtol=2e-5)
    return {"model_prefix_test": True, "model_future_perturbation_test": True, "tested_length": length,
            "conditioning": "uses only the anchor-time side and leg state"}

def conditioning_identity_check(model, cfg, device):
    """FiLM and fusion are initialized at the identity: conditioning must not change h before training."""
    if not cfg.conditioned:
        return {"identity_at_init": "not_applicable"}
    model.eval()
    gen = torch.Generator(device="cpu").manual_seed(7)
    h = torch.randn(4, cfg.width, generator=gen).to(device)
    cond = torch.randn(4, n_condition_features(cfg), generator=gen).to(device)
    side = torch.tensor([0, 1, 2, 0], device=device)
    with torch.no_grad():
        torch.testing.assert_close(model.condition(h, side, cond), h)
    return {"identity_at_init": True}

def replay_audit(bundle_path, asset, cfg, indices, device="cpu"):
    predictor = EndpointPredictor(bundle_path, device)
    prepared = Prepared({asset.symbol: asset}, {asset.symbol: predictor.scaler.transform(asset.features)}, {},
                        predictor.scaler, predictor.ctx.domain, predictor.ctx.bins, predictor.meta["asset_to_id"],
                        predictor.meta["fit_cutoff"])
    batch = predict_rows(predictor.models, prepared, asset.symbol, np.asarray(indices), predictor.cfg, predictor.device, rowwise=True)
    columns = ["time", *RAW_COLUMNS] + ([cfg.side_column] if cfg.direction_source == "column" else [])
    records = []
    for t, expected in zip(indices, batch.itertuples()):
        live = predictor.predict_latest(asset.raw.iloc[:int(t)+1][columns], asset.symbol)
        np.testing.assert_allclose(live["predicted_tp_pct"], expected.predicted_tp_pct, rtol=2e-4, atol=1e-10)
        np.testing.assert_allclose(live["estimated_strict_hit_probability"], expected.estimated_strict_hit_probability, rtol=2e-4, atol=2e-6)
        assert live["side"] == expected.side
        records.append(live)
    return pd.DataFrame(records)

def _brute_force_window_mass(p0, probs, edges, lower_ratio, points=20001):
    delta = math.log(1/lower_ratio)
    cdf_edges = torch.cat([torch.zeros_like(probs[:, :1]), probs.cumsum(1)], dim=1)
    u = torch.linspace(float(edges[0])-delta-0.5, float(edges[-1])+0.5, points, dtype=torch.float64)
    g = cdf_at(cdf_edges, edges, u+delta)-cdf_at(cdf_edges, edges, u)
    return torch.maximum((1-p0)*g.max(dim=1).values, p0)

def run_contract_tests():
    a = np.array([3, 3, .25, .25, .25, 0, 0, 1, np.nan], dtype=float)
    p = np.array([2.25, 2, .1875, .15, .26, 0, 1e-8, np.nan, .8], dtype=float)
    expected = np.array([1, 0, 1, 0, 0, 1, 0, 0, 0], dtype=bool)
    np.testing.assert_array_equal(strict_hits(a, p), expected)
    for scale in (.03, .2, .6, 2.):
        np.testing.assert_array_equal(strict_hits(a/scale, p/scale), expected)
    np.testing.assert_array_equal(strict_hits(np.array([1e-12, 0.]), np.array([0., 1e-20])), [False, False])
    cfg = Config()
    domain = CandidateDomain.fit([0, .00001, .1, 1, 100], cfg)
    assert domain.coverage(np.array([0, .00001, .1, 1, 100])).all()
    values = np.array([0., .1, .11, .12, 2., 3.])
    fitted = best_constant(values)
    assert strict_hits(values, np.full(len(values), fitted)).sum() == max(strict_hits(values, np.full(len(values), v)).sum() for v in np.r_[0, values])
    # ---- G5: distribution normalization, zero mass, tails and exact decoding ----
    rng = np.random.default_rng(5)
    bins = LogZBins.fit(np.exp(rng.normal(0.7, 0.8, 5000)), cfg)
    edges = torch.tensor(bins.edges, dtype=torch.float64)
    K = bins.count
    assert K == cfg.dist_bins+2 and bins.core_width() < STRICT_BAND_LOG
    soft = bins.soft_targets(torch.tensor([-5.0, 0.3, 1.1, 9.0], dtype=torch.float64), 0.75*bins.core_width())
    torch.testing.assert_close(soft.sum(1), torch.ones(4, dtype=torch.float64))
    hard = bins.soft_targets(torch.tensor([0.3], dtype=torch.float64), 0)
    assert int(hard.argmax()) == int(np.searchsorted(bins.edges, 0.3, side="right")-1)
    j = K//2
    one_hot = F.one_hot(torch.tensor([j]), K).double()
    sel, score, _ = decode_distribution(torch.tensor([0.01], dtype=torch.float64), one_hot, edges)
    assert abs(float(score[0])-0.99) < 1e-12, "Point mass in one narrow bin must be fully captured."
    zs = np.exp(np.linspace(bins.edges[j]+1e-9, bins.edges[j+1]-1e-9, 50))
    assert strict_hits(zs, np.full(50, float(sel[0]))).all(), "Selected candidate must hit every target in the mass bin."
    sel0, score0, _ = decode_distribution(torch.tensor([0.6], dtype=torch.float64), one_hot, edges)
    assert float(sel0[0]) == 0.0 and abs(float(score0[0])-0.6) < 1e-12
    logits = torch.tensor(rng.normal(size=(64, K)), dtype=torch.float64)*2
    probs = torch.softmax(logits, -1)
    p0 = torch.tensor(rng.uniform(0, .05, 64), dtype=torch.float64)
    torch.testing.assert_close(probs.sum(1), torch.ones(64, dtype=torch.float64))
    sel, score, cdf_edges = decode_distribution(p0, probs, edges)
    assert torch.allclose(cdf_edges[:, -1], torch.ones(64, dtype=torch.float64))
    brute = _brute_force_window_mass(p0, probs, edges, cfg.lower_ratio)
    assert torch.all(score >= brute-1e-12), "Exact breakpoint decoder must match a dense brute-force search."
    # Monte Carlo: sampled targets hit the selected candidate at the claimed probability.
    row = 3
    cat = torch.distributions.Categorical(probs[row]).sample((200000,), ).numpy()
    lo, hi = np.asarray(bins.edges)[cat], np.asarray(bins.edges)[cat+1]
    samples = np.exp(lo+(hi-lo)*rng.random(len(cat)))
    samples[rng.random(len(cat)) < float(p0[row])] = 0.0
    rate = strict_hits(samples, np.full(len(samples), float(sel[row]))).mean()
    assert abs(rate-float(score[row])) < 5*math.sqrt(.25/len(samples)), (rate, float(score[row]))
    # ---- causal ZigZag unit test on a hand-made path ----
    path = np.r_[np.linspace(100, 110, 40), np.linspace(109.6, 95, 40), np.linspace(95.4, 120, 40), np.linspace(119.5, 100, 40)]
    hi_, lo_ = path+0.1, path-0.1
    st, sd, ls = causal_zigzag_state(hi_, lo_, path, np.full(len(path), 1.0), depth=5)
    assert (sd[50:84] == -1).all() and (sd[85:124] == 1).all() and (sd[125:] == -1).all() and ls[100] == 79
    for cut in (50, 90, 130, len(path)-1):
        st2, sd2, ls2 = causal_zigzag_state(hi_[:cut+1], lo_[:cut+1], path[:cut+1], np.ones(cut+1), depth=5)
        np.testing.assert_array_equal(st2, st[:cut+1]); np.testing.assert_array_equal(sd2, sd[:cut+1])
    return {"metric_cases": len(a), "candidate_domain": "passed", "constant_baseline": "passed",
            "distribution_normalization_and_tails": "passed", "exact_decoder_vs_brute_force": "passed",
            "decoder_monte_carlo_hit_rate": float(rate), "zigzag_prefix_replay": "passed"}

### Execution TP reach (separate from strict accuracy)

A historical endpoint label existing does not mean an order would have filled. The simulator uses the **live side**, an entry at the anchor close (or next open), TP price \(C_t(1 + side \cdot tp/100)\), a spread buffer the TP must clear, an optional ATR stop, a maximum holding period and fees. A bar touching both stop and TP resolves conservatively as stopped. The headline denominator is **all eligible candles**: a zero prediction or unknown side places no order and counts as not reached. Trade-level reach and a hindsight-side touch rate are reported separately; the hindsight rate is diagnostic only.

In [14]:
@dataclass
class ReachSpec:
    entry: str = "signal_close"      # enter at the anchor close, or 'next_open'
    max_hold_bars: int = 192
    stop_atr_multiple: float | None = None
    spread_bps: float = 1.0
    fee_bps_per_side: float = 5.0
    same_bar_policy: str = "stop_first"   # conservative: no favorable intrabar ordering is assumed

def simulate_reach(asset, indices, predicted_tp_pct, sides, spec, nominal=False):
    """Execution path test, separate from strict endpoint accuracy.

    TP price = close_t x (1 + side x tp/100), matching the tp_pct definition. LONG reaches when a later high
    clears TP plus the spread buffer (SHORT symmetric). With a stop, a bar touching both is resolved by
    same_bar_policy ('stop_first' unless finer data proves otherwise). nominal=True drops stop and spread.
    Zero/failed predictions or unknown side create no order and count as not reached in all-candle reach.
    """
    o, h, l, c = (asset.raw[k].to_numpy(float) for k in ("open", "high", "low", "close"))
    atr_price = asset.atr_pct*c/100
    n = len(c)
    spread = 0.0 if nominal else spec.spread_bps/1e4
    rows = []
    for t, tp, s in zip(np.asarray(indices), np.asarray(predicted_tp_pct, float), np.asarray(sides)):
        rec = {"row": int(t), "side": int(s), "predicted_tp_pct": float(tp)}
        if s == 0 or not np.isfinite(tp) or tp <= 0:
            rec.update(status="no_order", reached=False, ambiguous=False, bars_to_outcome=np.nan, net_return_bps=0.0)
            rows.append(rec); continue
        first = t+1
        if first+spec.max_hold_bars-1 >= n:
            rec.update(status="pending_future_data", reached=False, ambiguous=False, bars_to_outcome=np.nan, net_return_bps=np.nan)
            rows.append(rec); continue
        entry = c[t] if spec.entry == "signal_close" else o[first]
        target = c[t]*(1+s*tp/100)
        need = target*(1+s*spread)
        seg = slice(first, first+spec.max_hold_bars)
        hh, ll = h[seg], l[seg]
        tp_hit = hh >= need if s > 0 else ll <= need
        if spec.entry == "next_open" and ((s > 0 and o[first] >= need) or (s < 0 and o[first] <= need)):
            tp_hit[0] = True
        stop_hit = np.zeros_like(tp_hit)
        stop = np.nan
        if spec.stop_atr_multiple is not None and not nominal:
            stop = entry-s*spec.stop_atr_multiple*atr_price[t]
            stop_hit = ll <= stop if s > 0 else hh >= stop
        i_tp = int(np.argmax(tp_hit)) if tp_hit.any() else None
        i_st = int(np.argmax(stop_hit)) if stop_hit.any() else None
        ambiguous = i_tp is not None and i_st is not None and i_tp == i_st
        if i_tp is not None and (i_st is None or i_tp < i_st or (ambiguous and spec.same_bar_policy == "tp_first")):
            status, exit_price, k = "reached", target, i_tp
        elif i_st is not None:
            status, exit_price, k = "stopped", stop, i_st
        else:
            status, exit_price, k = "expired", c[first+spec.max_hold_bars-1], spec.max_hold_bars-1
        gross = s*(exit_price-entry)/entry
        rec.update(status=status, reached=status == "reached", ambiguous=bool(ambiguous), bars_to_outcome=k+1,
                   net_return_bps=1e4*(gross-2*spec.fee_bps_per_side/1e4))
        rows.append(rec)
    return pd.DataFrame(rows)

def reach_summary(sim, label):
    done = sim[sim.status != "pending_future_data"]
    orders = done[done.status != "no_order"]
    return {"rule": label, "eligible_candles_scored": int(len(done)), "pending_future_data": int((sim.status == "pending_future_data").sum()),
            "orders_issued": int(len(orders)), "no_order": int((done.status == "no_order").sum()),
            "reach_all_candles": float(done.reached.mean()) if len(done) else None,
            "reach_issued_orders": float(orders.reached.mean()) if len(orders) else None,
            "stopped_rate_orders": float((orders.status == "stopped").mean()) if len(orders) else None,
            "expired_rate_orders": float((orders.status == "expired").mean()) if len(orders) else None,
            "ambiguous_same_bar": int(orders.ambiguous.sum()) if len(orders) else 0,
            "mean_net_return_bps_per_order": float(orders.net_return_bps.mean()) if len(orders) else None,
            "median_bars_to_reach": float(orders.loc[orders.reached, "bars_to_outcome"].median()) if orders.reached.any() else None}

def run_reach_tests():
    """Synthetic scenario checks, including same-bar stop/TP ambiguity."""
    times = pd.date_range("2026-01-01", periods=8, freq="15min", tz="UTC")
    raw = pd.DataFrame({"time": times, "open": 100., "high": 100.5, "low": 99.5, "close": 100.})
    raw.loc[3, "high"] = 102.5   # TP of 2% touched at bar 3
    raw.loc[5, ["high", "low"]] = [103., 97.]  # both touched on bar 5
    asset = type("A", (), {})()
    asset.raw, asset.atr_pct = raw, np.full(len(raw), 1.0)
    spec = ReachSpec(max_hold_bars=4, stop_atr_multiple=2.0, spread_bps=0.0)
    r = simulate_reach(asset, [0], [2.0], [1], spec)
    assert r.status[0] == "reached" and r.bars_to_outcome[0] == 3
    r = simulate_reach(asset, [3], [2.5], [1], spec)
    assert r.status[0] == "stopped" and bool(r.ambiguous[0]), "Same-bar touch must resolve conservatively."
    r = simulate_reach(asset, [0], [5.0], [1], spec)
    assert r.status[0] == "expired"
    r = simulate_reach(asset, [0], [0.0], [1], spec)
    assert r.status[0] == "no_order" and not r.reached[0]
    r = simulate_reach(asset, [6], [1.0], [1], spec)
    assert r.status[0] == "pending_future_data"
    r = simulate_reach(asset, [0], [2.0], [-1], spec)
    assert r.status[0] != "reached", "A SHORT target below price is not reached by an up-move."
    return {"reach_scenarios": "passed"}

### Plots

Prediction review (with the 80% requirement line and actual vs predicted log-Z histograms) and training history (loss, strict accuracy, and predicted log-Z spread for training probe vs validation).

In [15]:
def plot_results(predictions, output_dir=None, target_accuracy=0.80):
    import matplotlib.pyplot as plt
    p = predictions.copy()
    p["time"] = pd.to_datetime(p.time, utc=True)
    fig, ax = plt.subplots(3, 1, figsize=(13, 11), constrained_layout=True)
    recent = p.tail(min(len(p), 400))
    ax[0].plot(recent.time, recent.actual_tp_pct, label="Actual tp_pct", linewidth=1)
    ax[0].plot(recent.time, recent.predicted_tp_pct, label="Predicted tp_pct", linewidth=1)
    ax[0].fill_between(recent.time, .75*recent.actual_tp_pct, recent.actual_tp_pct, alpha=.15, label="Accepted range")
    ax[0].set_ylabel("Percentage points"); ax[0].legend()
    rolling = p.strict_hit.astype(float).rolling(7*96, min_periods=min(96, len(p))).mean()*100
    ax[1].plot(p.time, rolling)
    ax[1].axhline(100*target_accuracy, color="black", linestyle="--", linewidth=1, label=f"{100*target_accuracy:.0f}% requirement")
    ax[1].set_ylabel("Trailing 7-day strict accuracy (%)"); ax[1].legend()
    pos_a, pos_p = p.actual_z[p.actual_z > 0], p.predicted_z[p.predicted_z > 0]
    edges = np.linspace(np.log(min(pos_a.min(), pos_p.min())), np.log(max(pos_a.max(), pos_p.max())), 80)
    ax[2].hist(np.log(pos_a), bins=edges, alpha=.5, density=True, label="actual log Z")
    ax[2].hist(np.log(pos_p), bins=edges, alpha=.5, density=True, label="predicted log Z")
    ax[2].set_xlabel("log Z (ATR multiples)"); ax[2].legend()
    if output_dir:
        fig.savefig(Path(output_dir)/"prediction_review.png", dpi=150)
    plt.show()
    return fig

def plot_training_history(history_path, output_path=None):
    import matplotlib.pyplot as plt
    h = pd.DataFrame(json.loads(Path(history_path).read_text()))
    fig, ax = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    ax[0].plot(h.epoch, h.train_primary_loss, label="train primary")
    if "val_primary_loss" in h:
        ax[0].plot(h.epoch, h.val_primary_loss, label="validation primary")
    ax[0].set_title("Loss"); ax[0].legend()
    for k, lab in (("train_probe_strict_accuracy", "train probe"), ("val_strict_accuracy", "validation")):
        if k in h:
            ax[1].plot(h.epoch, 100*h[k], label=lab)
    ax[1].set_title("Strict accuracy (%)"); ax[1].legend()
    for k, lab in (("train_probe_predicted_log_z_std", "train probe"), ("val_predicted_log_z_std", "validation")):
        if k in h:
            ax[2].plot(h.epoch, h[k], label=lab)
    ax[2].axhline(STRICT_BAND_LOG/4, color="gray", linestyle=":", label="quarter strict band")
    ax[2].set_title("Std of predicted log Z (collapse check)"); ax[2].legend()
    if output_path:
        fig.savefig(output_path, dpi=130)
    plt.show()
    return fig

## Load the cleaned data and check live-direction provenance

Required raw columns: `time` (UTC opening time), `open`, `high`, `low`, `close`, `volume`, `quote_volume`, `num_trades`, `taker_base_volume`, `taker_quote_volume`. Training also needs `tp_pct` and `tp_confirm_time` (or an audited `label_available_time`); `tp_pivot_time` and `signal` enable leg grouping and direction diagnostics.

In [16]:
def resolve_data_files(mapping):
    if mapping:
        result = {s: str(Path(p).expanduser()) for s, p in mapping.items()}
        missing = [p for p in result.values() if not Path(p).is_file()]
        if missing:
            raise FileNotFoundError(f'CSV paths do not exist: {missing}')
        return result
    roots = list(dict.fromkeys([Path('/content'), Path.cwd()]))
    candidates = list(dict.fromkeys(str(p.resolve()) for r in roots if r.exists() for p in r.glob('*.csv')))
    if len(candidates) == 1:
        return {CFG.primary_symbol: candidates[0]}
    if candidates:
        raise ValueError(f'Multiple CSVs found. Set DATA_FILES explicitly: {candidates}')
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Set DATA_FILES to your cleaned CSV path, then rerun this cell.') from exc
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
    if len(csv_names) != 1:
        raise ValueError('Upload one labeled CSV or set DATA_FILES explicitly after upload.')
    return {CFG.primary_symbol: str(Path(csv_names[0]).resolve())}

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/model_b_runs'

RUN_DIR = Path(OUTPUT_ROOT)/RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
V2_DIR = Path(V2_RUN_DIR) if Path(V2_RUN_DIR).is_absolute() else Path(OUTPUT_ROOT)/V2_RUN_DIR
DATA_FILES = resolve_data_files(DATA_FILES)
ASSETS = prepare_assets(DATA_FILES, DATA_METADATA, CFG)
PRIMARY = ASSETS[CFG.primary_symbol]
write_json(RUN_DIR/'data_audit.json', {s: a.info for s, a in ASSETS.items()})
write_json(RUN_DIR/'runtime_versions.json', package_versions())
display(pd.DataFrame([{'symbol': s, **a.info, 'finite_feature_rows': int(a.finite_features.sum()),
                       'feature_count': a.features.shape[1]} for s, a in ASSETS.items()]).drop(columns=['metadata']))

# V3.0 / cell-10 task: live direction provenance. The model input side must be live; the label side is hindsight.
DIRECTION_PROVENANCE = {
    'training_period': direction_provenance_report(PRIMARY, CFG, end=EVAL_START),
    'development_evaluation_period': direction_provenance_report(PRIMARY, CFG, start=EVAL_START, end=EVAL_END),
}
write_json(RUN_DIR/'direction_provenance.json', DIRECTION_PROVENANCE)
print(json.dumps(json_safe(DIRECTION_PROVENANCE), indent=2))
if any(a.info['maturity_status'] != 'verified_by_supplied_provenance' for a in ASSETS.values()):
    print('Development data: label-confirmation availability is an UNVERIFIED proxy.')
print('Output directory:', RUN_DIR, '| V2 run directory:', V2_DIR, '(found)' if V2_DIR.exists() else '(not found)')

Mounted at /content/drive


,symbol,source_path,source_sha256,rows,start,end,pending_labels,zero_targets,maturity_status,label_rule_id,leg_id_source,unique_legs,side_source,finite_feature_rows,feature_count
0,BTCUSDT,/content/drive/MyDrive/data_raw_with_indicator...,b5819fdb81ac71ff9210ff39d49ee75a944fe973904cca...,99416,2024-01-01 08:15:00+00:00,2026-11-01 22:00:00+00:00,0,3,provided_confirmation_proxy_UNVERIFIED,None,tp_pivot_time + signal (training-only grouping...,5710,"causal_zigzag(depth=12, min_dev_atr=0.0)",99224,97


{
  "training_period": {
    "rows": 87327,
    "side_source": "causal_zigzag(depth=12, min_dev_atr=0.0)",
    "live_side_unknown_rows": 0,
    "live_long_share": 0.49726888591157375,
    "agreement_live_vs_label_direction": 0.49013478076654415,
    "agreement_signal_live_vs_label_direction": 0.526389318309343,
    "agreement_signal_live_vs_causal_zigzag_side": 0.6548833694046515
  },
  "development_evaluation_period": {
    "rows": 10176,
    "side_source": "causal_zigzag(depth=12, min_dev_atr=0.0)",
    "live_side_unknown_rows": 0,
    "live_long_share": 0.5229952830188679,
    "agreement_live_vs_label_direction": 0.4907625786163522,
    "agreement_signal_live_vs_label_direction": 0.5090408805031447,
    "agreement_signal_live_vs_causal_zigzag_side": 0.6723663522012578
  }
}
Development data: label-confirmation availability is an UNVERIFIED proxy.
Output directory: /content/drive/MyDrive/model_b_runs/model_b_v3_directional_multitask | V2 run directory: /content/drive/MyDrive/model_b_

## Contracts, causality and smoke tests before any training

This writes the feature manifest and leg statistics. It runs one-batch forward/backward smoke tests with finite-gradient checks for the V2 reference configuration and both V3 heads, and reports parameter counts.

In [17]:
CONTRACT_CHECKS = run_contract_tests()
CONTRACT_CHECKS['reach_simulator'] = run_reach_tests()
PREFLIGHT = prepare_fit(ASSETS, EVAL_START, CFG)
EVAL_INDICES = eligible_indices(PRIMARY, CFG, start=EVAL_START, end=EVAL_END)
if len(EVAL_INDICES) == 0:
    raise ValueError('No eligible evaluation rows. Check dates and available history.')
TRAIN_IDX = PREFLIGHT.train_idx[CFG.primary_symbol]
LEG_STATS = {'training_anchors': leg_statistics(PRIMARY, TRAIN_IDX, CFG.leg_cap),
             'evaluation_anchors': leg_statistics(PRIMARY, EVAL_INDICES, CFG.leg_cap)}
write_json(RUN_DIR/'leg_statistics.json', LEG_STATS)
print('Training anchors:', len(TRAIN_IDX), '| evaluation anchors:', len(EVAL_INDICES),
      '| interval candidates:', PREFLIGHT.domain.count, '| distribution bins:', PREFLIGHT.bins.count)
for name, s in LEG_STATS.items():
    print(f"{name}: {s['unique_legs']:,} legs, {s['anchors_per_leg_mean']:.1f} anchors/leg (median {s['anchors_per_leg_median']:.0f}, "
          f"max {s['anchors_per_leg_max']}), intra-leg ICC(log Z) {s['intra_leg_icc_logz']:.3f}, "
          f"effective sample size ~{(s['effective_sample_size'] or float('nan')):,.0f}")
write_json(RUN_DIR/'feature_manifest.json', {
    'version': FEATURE_VERSION, 'columns': list(PRIMARY.features.columns),
    'base_columns': list(PRIMARY.features.columns[:BASE_FEATURE_COUNT]), 'state_columns': STATE_COLUMNS,
    'sequence_features_for_CFG': n_sequence_features(CFG), 'condition_features_for_CFG': n_condition_features(CFG),
    'side_source': PRIMARY.side_source, 'feature_origins': {s: a.info['start'] for s, a in ASSETS.items()},
    'target': 'tp_pct / (100 * ATR14 / close)', 'strict_rule': '0.75 * actual <= predicted <= actual',
})

def smoke_test(cfg, prepared, device):
    """One-batch forward/backward: finite loss, finite gradients, exact output shapes."""
    seed_all(1)
    model = build_model(cfg, prepared).to(device)
    initialize_output_priors(model, prepared, cfg)
    refs = training_refs(prepared, cfg, 1, 0)[:2]
    x, cond, side, ids, target, _ = next(iter(make_loader(prepared, refs, cfg, train=True)))
    x, cond, side, ids, target = x.to(device), cond.to(device), side.to(device), ids.to(device), target.to(device)
    model.train()
    log_grid = torch.tensor(prepared.domain.log_grid(), dtype=torch.float64, device=device)
    with amp_context(device, cfg):
        h = model.encode(x, ids, side, cond)
        primary, aux = loss_rows(model, h, target, prepared, log_grid, cfg)
        loss = total_loss(primary, aux, cfg)
    loss.backward()
    grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
    assert h.shape == (2, cfg.width) and primary.shape == (2,) and torch.isfinite(loss)
    assert all(torch.isfinite(g).all() for g in grads), 'Nonfinite gradient in smoke test.'
    result = {'loss': float(loss.detach()), 'parameters': sum(p.numel() for p in model.parameters()),
              'tensors_with_gradients': len(grads), **audit_model_causality(model, cfg, device),
              **conditioning_identity_check(build_model(cfg, prepared).to(device), cfg, device)}
    del model
    return result

if RUN_CAUSALITY_AUDITS:
    CONTRACT_CHECKS['feature_audits'] = {s: audit_features(a, CFG) for s, a in ASSETS.items()}
    CONTRACT_CHECKS['smoke_V2_reference_config'] = smoke_test(v2_reference_config(CFG), PREFLIGHT, DEVICE)
    CONTRACT_CHECKS['smoke_V3_distribution_head'] = smoke_test(replace(CFG, head_type='distribution'), PREFLIGHT, DEVICE)
    CONTRACT_CHECKS['smoke_V3_interval_head'] = smoke_test(replace(CFG, head_type='interval'), PREFLIGHT, DEVICE)
    print('Parameters: V2 reference', CONTRACT_CHECKS['smoke_V2_reference_config']['parameters'],
          '| V3 distribution', CONTRACT_CHECKS['smoke_V3_distribution_head']['parameters'],
          '| V3 interval', CONTRACT_CHECKS['smoke_V3_interval_head']['parameters'])
write_json(RUN_DIR/'preflight_checks.json', CONTRACT_CHECKS)
print(json.dumps(json_safe(CONTRACT_CHECKS), indent=2))

Training anchors: 86175 | evaluation anchors: 10176 | interval candidates: 1539 | distribution bins: 98
training_anchors: 4,930 legs, 17.5 anchors/leg (median 14, max 100), intra-leg ICC(log Z) 0.548, effective sample size ~8,591
evaluation_anchors: 612 legs, 16.6 anchors/leg (median 14, max 77), intra-leg ICC(log Z) 0.586, effective sample size ~1,003
Parameters: V2 reference 3621554 | V3 distribution 3796172 | V3 interval 3816459
{
  "metric_cases": 9,
  "candidate_domain": "passed",
  "constant_baseline": "passed",
  "distribution_normalization_and_tails": "passed",
  "exact_decoder_vs_brute_force": "passed",
  "decoder_monte_carlo_hit_rate": 0.1776,
  "zigzag_prefix_replay": "passed",
  "reach_simulator": {
    "reach_scenarios": "passed"
  },
  "feature_audits": {
    "BTCUSDT": {
      "prefix_positions_tested": [
        800,
        49708,
        54953,
        58974,
        76852,
        99415
      ],
      "future_perturbation_passed": true,
      "label_input_exclusion_p

## V3.0 — Establish the real failure mechanism

- **A. Prediction collapse.** Uses the saved V2 predictions (no GPU): spread, ordering, paired comparison with fixed ATR (week- and leg-block CIs), and accuracy by label direction, live side, side agreement, target size, ATR regime, causal leg age and bars to endpoint. If the V2 `model_bundle` exists, it also scores a training sample, so seen-data spread can be compared with the unseen inner-validation and development predictions.
- **B. Gradient flow.** Per-component gradient norms and ratios before clipping, plus clipping frequency, for the V2 configuration and the V3 configuration from initialization, and as a snapshot at the saved V2 weights when available.
- **C. Memorization.** 64–128 anchors from distinct legs, no dropout, weight decay or EMA. A functional test of the learning system, not evidence of forecasting skill.

**Completion condition:** collapse, gradient behavior, sample dependence and small-subset fitting have all been measured.

In [18]:
# V3.0 diagnostics: definitions only. The next three cells run A, B and C.

def load_run_predictions(run_main):
    path = Path(run_main)/"predictions.csv"
    if not path.is_file():
        raise FileNotFoundError(f"{path} not found. Set V2_RUN_DIR to the finished V2 run.")
    p = pd.read_csv(path)
    p["time"] = pd.to_datetime(p.time, utc=True)
    return p.sort_values("time").reset_index(drop=True)

def rows_for_times(asset, times):
    idx = pd.Index(asset.raw.time).get_indexer(pd.to_datetime(times, utc=True))
    if (idx < 0).any():
        raise ValueError("Saved predictions contain candle times missing from the loaded CSV. Load the same data file.")
    return idx

def enrich_predictions(p, asset):
    """Attach causal side, label direction, leg age and label-derived endpoint distance (diagnostics only)."""
    p = p.copy()
    idx = rows_for_times(asset, p.time)
    p["side"] = asset.side[idx].astype(int)
    p["label_side"] = asset.raw.label_side.to_numpy()[idx].astype(int)
    p["leg_age_bars"] = np.where(asset.leg_start[idx] >= 0, idx-asset.leg_start[idx], -1)
    p["leg_id"] = asset.raw.leg_id.to_numpy()[idx]
    if "tp_pivot_time" in asset.raw:
        p["bars_to_endpoint"] = ((asset.raw.tp_pivot_time.iloc[idx]-asset.raw.time.iloc[idx])/BAR).to_numpy()
    if "actual_z" not in p:
        p["actual_z"] = asset.z[idx]
    if "atr_pct_points" not in p:
        p["atr_pct_points"] = asset.atr_pct[idx]
    if "predicted_z" not in p:
        p["predicted_z"] = p.predicted_tp_pct/p.atr_pct_points
    p["strict_hit"] = strict_hits(p.actual_tp_pct, p.predicted_tp_pct)
    return p, idx

def collapse_report(p, cfg, fixed_multiple, label):
    """Diagnostic A for one prediction set: spread, ordering and where errors occur."""
    rep = prediction_spread_report(p, fixed_multiple, cfg)
    rep["set"] = label
    rep["rows"] = int(len(p))
    rep["strict_accuracy"] = float(p.strict_hit.mean())
    fixed_hits = strict_hits(p.actual_z, np.full(len(p), fixed_multiple))
    rep["fixed_atr_accuracy_same_rows"] = float(fixed_hits.mean())
    return rep

def v2_prepared(asset, scaler, domain, asset_to_id, fit_cutoff, train_idx=None):
    view = replace(asset, features=asset.features.iloc[:, :BASE_FEATURE_COUNT])
    return Prepared({asset.symbol: view}, {asset.symbol: scaler.transform(view.features)},
                    {asset.symbol: np.asarray(train_idx if train_idx is not None else [], dtype=np.int64)},
                    scaler, domain, None, asset_to_id, fit_cutoff)

def interpret_gradient_table(table, summary):
    enc = table[table.component.str.startswith(("input_projection", "tcn_block", "transformer"))]
    head = table[table.component.isin(["zero_head", "interval_head", "distribution_head", "magnitude_head"])]
    enc_r, head_r = float(enc.grad_to_param.median()), float(head.grad_to_param.median())
    notes = []
    if head_r > 0 and enc_r < 0.01*head_r:
        notes.append("ENCODER STARVED: TCN/Transformer gradient-to-parameter ratios are <1% of the heads' -> training problem; "
                     "inspect loss scaling, prior initialization and residual/normalization behaviour.")
    if summary.get("clip_fraction", 0) > 0.9:
        notes.append("Nearly every update clips: check learning rates and objective scaling before anything else.")
    if summary.get("nonfinite_updates", 0):
        notes.append("Nonfinite gradients observed.")
    if not notes:
        notes.append("Gradients reach every component. If unseen accuracy is still poor, treat it as a generalization "
                     "or information problem rather than a backpropagation defect.")
    return {"encoder_median_grad_to_param": enc_r, "head_median_grad_to_param": head_r,
            "encoder_to_head_ratio": enc_r/head_r if head_r > 0 else None, "interpretation": notes}

def gradient_flow_probe(prepared, cfg, device, updates, output_dir, init_state=None, tag="probe"):
    """Diagnostic B. init_state=None: short training run from the prior initialization (optimizer steps taken).
    init_state given (e.g. trained V2 weights): gradient snapshot at that solution, no optimizer steps."""
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    seed_all(cfg.seeds[0])
    model = build_model(cfg, prepared).to(device)
    if init_state is not None:
        model.load_state_dict(init_state, strict=True)
    else:
        initialize_output_priors(model, prepared, cfg)
    optimizer = optimizer_for(model, cfg)
    monitor = GradientMonitor(model, cfg)
    log_grid = torch.tensor(prepared.domain.log_grid(), dtype=torch.float64, device=device)
    refs = training_refs(prepared, cfg, cfg.seeds[0], 0)
    loader = iter(make_loader(prepared, refs, cfg, train=True, seed=cfg.seeds[0]))
    losses = []
    model.train()
    for u in range(updates):
        optimizer.zero_grad(set_to_none=True)
        for micro in range(cfg.gradient_accumulation):
            try:
                x, cond, side, ids, target, _ = next(loader)
            except StopIteration:
                loader = iter(make_loader(prepared, refs, cfg, train=True, seed=cfg.seeds[0]+u))
                x, cond, side, ids, target, _ = next(loader)
            x, cond, side, ids, target = x.to(device), cond.to(device), side.to(device), ids.to(device), target.to(device)
            with amp_context(device, cfg):
                h = model.encode(x, ids, side, cond)
                primary, aux = loss_rows(model, h, target, prepared, log_grid, cfg)
                loss = total_loss(primary, aux, cfg)
            if u % max(1, cfg.loss_grad_probe_every) == 0 and micro == 0:
                monitor.probe_losses(model, primary, aux, u)
            (loss/cfg.gradient_accumulation).backward()
            losses.append(float(loss.detach()))
        clip = float("inf") if init_state is not None else cfg.gradient_clip
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        monitor.observe(model, norm, u, detailed=True)
        if init_state is None:
            optimizer.step()
    records = pd.DataFrame(monitor.records)
    records.to_csv(out/f"{tag}_gradient_records.csv", index=False)
    half = records["update"] >= records["update"].median()
    table = (records.groupby("component")
             .agg(grad_norm=("grad_norm", "mean"), param_norm=("param_norm", "mean"),
                  grad_to_param=("grad_to_param", "mean"), zero_grad_tensor_share=("zero_grad_tensor_share", "mean"))
             .join(records[half].groupby("component").grad_to_param.mean().rename("grad_to_param_late_half"))
             .reset_index())
    table["share_of_squared_grad"] = table.grad_norm**2/(table.grad_norm**2).sum()
    order = {c: i for i, c in enumerate(["input_projection", *[f"tcn_block_{i}" for i in range(len(cfg.dilations))],
                                         *[f"transformer_{i}" for i in range(cfg.transformer_layers)], "asset_adapter",
                                         "film_conditioning", "state_fusion", "zero_head", "interval_head",
                                         "distribution_head", "magnitude_head"])}
    table = table.sort_values("component", key=lambda s: s.map(order).fillna(99)).reset_index(drop=True)
    summary = {**monitor.summary(), "mode": "snapshot_at_saved_weights" if init_state is not None else "training_from_prior_init",
               "updates": updates, "first_loss": losses[0], "last_loss": float(np.mean(losses[-cfg.gradient_accumulation:]))}
    summary.update(interpret_gradient_table(table, summary))
    table.to_csv(out/f"{tag}_gradient_by_component.csv", index=False)
    write_json(out/f"{tag}_gradient_summary.json", summary)
    del model, optimizer
    return table, summary

def choose_memorization_anchors(prepared, cfg, n, seed=11):
    """One anchor per distinct original leg, spread across the training history; positive targets only."""
    sym = cfg.primary_symbol
    asset = prepared.assets[sym]
    idx = np.asarray(prepared.train_idx[sym])
    idx = idx[asset.z[idx] > 0]
    legs = leg_ids_for(asset, idx)
    rng = np.random.default_rng(seed)
    frame = pd.DataFrame({"idx": idx, "leg": legs, "r": rng.random(len(idx))})
    one_per_leg = frame.sort_values("r").groupby("leg").idx.first().sort_values().to_numpy()
    if len(one_per_leg) >= n:
        return one_per_leg[np.linspace(0, len(one_per_leg)-1, n).astype(int)]
    return np.sort(rng.choice(idx, n, replace=False))

def memorization_test(prepared, cfg, device, n_anchors, updates, output_dir, lr=1e-3, pass_accuracy=0.9, tag="memorize"):
    """Diagnostic C: fit 64-128 training anchors with no regularization. A functional test of the learning
    system, NOT evidence of forecasting skill."""
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    sym = cfg.primary_symbol
    chosen = choose_memorization_anchors(prepared, cfg, n_anchors)
    mem_cfg = replace(cfg, dropout=0.0, head_dropout=0.0, weight_decay=0.0, unknown_asset_probability=0.0,
                      sampler="uniform", max_train_anchors_per_epoch=None)
    mem = replace(prepared, train_idx={sym: chosen})
    seed_all(cfg.seeds[0])
    model = build_model(mem_cfg, mem).to(device)
    initialize_output_priors(model, mem, mem_cfg)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    log_grid = torch.tensor(mem.domain.log_grid(), dtype=torch.float64, device=device)
    refs = [(sym, int(i)) for i in chosen]
    history = []
    check_every = max(1, updates//12)
    for u in range(updates):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total = 0.0
        for x, cond, side, ids, target, _ in make_loader(mem, refs, mem_cfg, train=True):
            x, cond, side, ids, target = x.to(device), cond.to(device), side.to(device), ids.to(device), target.to(device)
            with amp_context(device, mem_cfg):
                h = model.encode(x, ids, side, cond)
                primary, aux = loss_rows(model, h, target, mem, log_grid, mem_cfg)
                loss = total_loss(primary, aux, mem_cfg)
            (loss*len(x)/len(refs)).backward()
            total += float(loss.detach())*len(x)/len(refs)
        torch.nn.utils.clip_grad_norm_(model.parameters(), mem_cfg.gradient_clip)
        optimizer.step()
        if (u+1) % check_every == 0 or u+1 == updates:
            p = predict_rows([model], mem, sym, chosen, mem_cfg, device)
            s = spread_summary(p.predicted_z, p.actual_z)
            history.append({"update": u+1, "train_loss": total, "strict_accuracy": float(p.strict_hit.mean()),
                            "predicted_log_z_std": s.get("predicted_log_z_std"), "spearman": s.get("spearman_predicted_vs_actual_z")})
            print(f"[{tag}] update {u+1}: loss {total:.4f}, strict accuracy {history[-1]['strict_accuracy']:.3f}, "
                  f"log-Z std {history[-1]['predicted_log_z_std']}")
    hist = pd.DataFrame(history)
    hist.to_csv(out/f"{tag}_history.csv", index=False)
    final = history[-1]
    summary = {"anchors": int(len(chosen)), "distinct_legs": int(len(np.unique(leg_ids_for(prepared.assets[sym], chosen)))),
               "updates": updates, "lr": lr, "final_strict_accuracy": final["strict_accuracy"],
               "best_strict_accuracy": float(hist.strict_accuracy.max()), "final_predicted_log_z_std": final["predicted_log_z_std"],
               "actual_log_z_std": float(np.log(prepared.assets[sym].z[chosen]).std()),
               "passed": bool(final["strict_accuracy"] >= pass_accuracy), "pass_threshold": pass_accuracy,
               "interpretation": ("Learning system can fit a small set." if final["strict_accuracy"] >= pass_accuracy else
                                  "FAILED to memorize: investigate loss, gradients, target representation or implementation before training at scale.")}
    write_json(out/f"{tag}_summary.json", summary)
    del model, optimizer
    return summary, hist

In [19]:
def group_accuracy_table(p, key, fixed_hits):
    d = add_report_groups(p).assign(_fixed=np.asarray(fixed_hits, bool))
    if key not in d:
        return pd.DataFrame()
    rows = []
    for g, f in d.groupby(key, observed=True):
        rows.append({key: str(g), "rows": len(f), "model_accuracy": float(f.strict_hit.mean()),
                     "fixed_atr_accuracy": float(f._fixed.mean()), "lift_pp": 100*float(f.strict_hit.mean()-f._fixed.mean()),
                     "median_actual_z": float(f.actual_z.median()), "median_predicted_z": float(f.predicted_z.median())})
    return pd.DataFrame(rows)

V30 = {}
if RUN_V30_COLLAPSE_DIAGNOSTICS:
    DIAG_OUT = RUN_DIR/'v3_0_diagnostics'; DIAG_OUT.mkdir(parents=True, exist_ok=True)
    V2_MAIN = V2_DIR/'main'
    P2, IDX2 = enrich_predictions(load_run_predictions(V2_MAIN), PRIMARY)
    if not np.allclose(P2.actual_tp_pct.to_numpy(float), PRIMARY.y[IDX2], rtol=1e-9, atol=1e-12, equal_nan=True):
        raise ValueError('Saved V2 actual_tp_pct does not match the loaded CSV. Use the same data file as the V2 run.')
    V2_CFG = v2_reference_config(CFG)
    V2_TRAIN_IDX = eligible_indices(PRIMARY, V2_CFG, fit_cutoff=EVAL_START)
    FIXED_ATR = best_constant(PRIMARY.z[V2_TRAIN_IDX], CFG.lower_ratio)
    FIXED_TP = best_constant(PRIMARY.y[V2_TRAIN_IDX], CFG.lower_ratio)
    saved_constants = V2_MAIN/'constant_baselines.json'
    if saved_constants.is_file():
        saved = json.loads(saved_constants.read_text())
        for key, value in (('constant_atr_units', FIXED_ATR), ('constant_raw_tp', FIXED_TP)):
            if key in saved and not np.isclose(saved[key]['fitted_constant'], value, rtol=1e-9):
                raise ValueError(f'{key}: refitted {value} differs from the saved V2 value {saved[key]["fitted_constant"]}.')
    times, actual, atr = P2.time, PRIMARY.y[IDX2], PRIMARY.atr_pct[IDX2]
    PIVOT_RULE = fit_pivot_rule(PRIMARY, V2_TRAIN_IDX, CFG, PIVOT_RULE_AGE_BINS, PIVOT_RULE_DISTANCE_BINS, PIVOT_RULE_MIN_BUCKET)
    PIVOT_MULT, _, _ = apply_pivot_rule(PIVOT_RULE, PRIMARY, IDX2)
    fixed_key = f'Fixed {FIXED_ATR:.4f} ATR'
    RULES = {f'Fixed TP {FIXED_TP:.4f}%': np.full(len(IDX2), FIXED_TP), 'V2 TCN + Transformer': P2.predicted_tp_pct.to_numpy(float),
             fixed_key: FIXED_ATR*atr, 'Confirmed-pivot ATR rule': PIVOT_MULT*atr}
    HITS = {k: strict_hits(actual, v, CFG.lower_ratio) for k, v in RULES.items()}
    RULE_TABLE = pd.DataFrame([{'rule': k, **hit_breakdown(actual, v, CFG.lower_ratio),
                                'hits_vs_fixed_atr': int(HITS[k].sum()-HITS[fixed_key].sum())} for k, v in RULES.items()])
    PAIRED_V2 = paired_report('V2 TCN + Transformer', HITS['V2 TCN + Transformer'], fixed_key, HITS[fixed_key], times, CFG,
                              legs=P2.leg_id.to_numpy())

    # ---- Prediction collapse on seen versus unseen data ----
    SETS = {'development_evaluation (V2 refit, unseen)': P2}
    for path in sorted(V2_MAIN.glob('selection_seed_*/inner_validation_predictions.csv')):
        q = pd.read_csv(path); q['time'] = pd.to_datetime(q.time, utc=True)
        SETS[f'inner_validation {path.parent.name} (selection model, unseen)'] = enrich_predictions(q, PRIMARY)[0]
    V2_BUNDLE = V2_MAIN/'model_bundle'
    if (V2_BUNDLE/'manifest.json').is_file():
        v2_models, v2_cfg, v2_domain, v2_scaler, v2_meta, _ = load_v2_bundle(V2_BUNDLE, DEVICE)
        sample = V2_TRAIN_IDX[np.linspace(0, len(V2_TRAIN_IDX)-1, min(COLLAPSE_TRAIN_SAMPLE, len(V2_TRAIN_IDX))).astype(int)]
        v2_prep = v2_prepared(PRIMARY, v2_scaler, v2_domain, v2_meta['asset_to_id'], v2_meta['fit_cutoff'], V2_TRAIN_IDX)
        train_pred = predict_rows(v2_models, v2_prep, CFG.primary_symbol, sample, v2_cfg, DEVICE, progress=True)
        SETS['training_sample (V2 refit, seen)'] = enrich_predictions(train_pred, PRIMARY)[0]
        del v2_models
    else:
        print('No V2 model_bundle found: training-set spread is skipped (unseen sets are still reported).')
    COLLAPSE = pd.DataFrame([{k: v for k, v in collapse_report(p, CFG, FIXED_ATR, name).items()
                              if not isinstance(v, dict)} for name, p in SETS.items()])
    seen = COLLAPSE[COLLAPSE.set.str.contains('seen\\)') & ~COLLAPSE.set.str.contains('unseen')]
    unseen = COLLAPSE[COLLAPSE.set.str.contains('unseen')]
    if len(seen) and len(unseen):
        ratio = float(seen.predicted_log_iqr.iloc[0]/max(1e-12, unseen.predicted_log_iqr.max()))
        COLLAPSE_VERDICT = ('variance appears on training data but disappears on unseen data (generalization)'
                            if ratio > 2 else 'predictions are similarly narrow on seen and unseen data (learning/information)')
    else:
        ratio, COLLAPSE_VERDICT = None, 'training-set spread unavailable'

    # ---- Where the errors are: direction, magnitude, leg progression ----
    BREAKDOWNS = {key: group_accuracy_table(P2, key, HITS[fixed_key])
                  for key in ('label_direction', 'live_side', 'live_side_matches_label', 'tp_size', 'target_band',
                              'volatility_regime', 'causal_leg_age', 'bars_to_endpoint_label_diag')}
    for key, table in BREAKDOWNS.items():
        table.to_csv(DIAG_OUT/f'v2_accuracy_by_{key}.csv', index=False)
    EVAL_LEGS = leg_statistics(PRIMARY, IDX2, CFG.leg_cap)
    RULE_TABLE.to_csv(DIAG_OUT/'rule_comparison.csv', index=False)
    COLLAPSE.to_csv(DIAG_OUT/'prediction_collapse_by_set.csv', index=False)
    V30['A'] = {'rows': len(P2), 'fixed_atr_multiple': FIXED_ATR, 'fixed_tp_pct': FIXED_TP,
                'paired_v2_vs_fixed_atr': PAIRED_V2, 'collapse_verdict': COLLAPSE_VERDICT,
                'seen_to_unseen_log_iqr_ratio': ratio, 'collapse_by_set': COLLAPSE.to_dict('records'),
                'training_leg_statistics': LEG_STATS['training_anchors'], 'evaluation_leg_statistics': EVAL_LEGS,
                'label_maturity': 'tp_confirm_time proxy, UNVERIFIED'}
    write_json(DIAG_OUT/'diagnostic_A_prediction_collapse.json', V30['A'])
    pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
    display(RULE_TABLE)
    display(COLLAPSE[['set', 'rows', 'strict_accuracy', 'fixed_atr_accuracy_same_rows', 'predicted_z_std', 'actual_z_std',
                      'predicted_log_iqr', 'predicted_log_p1_p99_span', 'spearman_predicted_vs_actual_z', 'verdict']])
    print('Collapse verdict:', COLLAPSE_VERDICT)
    for key in ('label_direction', 'live_side_matches_label', 'tp_size', 'causal_leg_age'):
        display(BREAKDOWNS[key])
    print(f"Evaluation period: {EVAL_LEGS['unique_legs']:,} original legs, {EVAL_LEGS['anchors_per_leg_mean']:.1f} anchors per leg, "
          f"ICC {EVAL_LEGS['intra_leg_icc_logz']:.3f}, effective sample ~{(EVAL_LEGS['effective_sample_size'] or float('nan')):,.0f} of {len(P2):,}")

In [20]:
if RUN_V30_GRADIENT_PROBE:
    GRAD_OUT = RUN_DIR/'v3_0_diagnostics'/'gradient_flow'
    GRADIENT_TABLES, GRADIENT_SUMMARIES = {}, {}
    probes = [('v2_config_from_init', PREFLIGHT, v2_reference_config(CFG), None),
              ('v3_config_from_init', PREFLIGHT, CFG, None)]
    v2_bundle = V2_DIR/'main'/'model_bundle'
    if (v2_bundle/'manifest.json').is_file():
        v2_models, v2_cfg, v2_domain, v2_scaler, v2_meta, _ = load_v2_bundle(v2_bundle, DEVICE)
        v2_prep = v2_prepared(PRIMARY, v2_scaler, v2_domain, v2_meta['asset_to_id'], v2_meta['fit_cutoff'],
                              eligible_indices(PRIMARY, v2_cfg, fit_cutoff=EVAL_START))
        probes.append(('v2_saved_weights_snapshot', v2_prep, v2_cfg, state_cpu(v2_models[0].state_dict())))
        del v2_models
    for tag, prep, cfg_probe, init in probes:
        n_updates = GRADIENT_PROBE_UPDATES if init is None else max(1, min(50, GRADIENT_PROBE_UPDATES))
        print(f'\n--- Gradient flow: {tag} ({n_updates} updates) ---')
        GRADIENT_TABLES[tag], GRADIENT_SUMMARIES[tag] = gradient_flow_probe(prep, cfg_probe, DEVICE, n_updates, GRAD_OUT, init, tag)
        s = GRADIENT_SUMMARIES[tag]
        print(f"clip fraction {s['clip_fraction']:.2f} | pre-clip norm median {s['preclip_norm_median']:.3g} | "
              f"encoder/head grad-to-param ratio {s['encoder_to_head_ratio']}")
        for note in s['interpretation']:
            print(' -', note)
    COMPARE = None
    for tag, table in GRADIENT_TABLES.items():
        t = table[['component', 'grad_to_param']].rename(columns={'grad_to_param': tag})
        COMPARE = t if COMPARE is None else COMPARE.merge(t, on='component', how='outer')
    display(COMPARE)
    V30['B'] = GRADIENT_SUMMARIES
    write_json(RUN_DIR/'v3_0_diagnostics'/'diagnostic_B_gradient_flow.json', GRADIENT_SUMMARIES)

In [21]:
if RUN_V30_MEMORIZATION_TEST:
    MEM_OUT = RUN_DIR/'v3_0_diagnostics'/'memorization'
    MEMORIZATION = {}
    for tag, cfg_mem in (('v2_config', v2_reference_config(CFG)), ('v3_config', CFG)):
        print(f'\n--- Memorization test: {tag}, {MEMORIZATION_ANCHORS} anchors, {MEMORIZATION_UPDATES} full-batch updates ---')
        MEMORIZATION[tag], _ = memorization_test(PREFLIGHT, cfg_mem, DEVICE, MEMORIZATION_ANCHORS, MEMORIZATION_UPDATES,
                                                 MEM_OUT, pass_accuracy=MEMORIZATION_PASS_ACCURACY, tag=tag)
        print(json.dumps(json_safe(MEMORIZATION[tag]), indent=2))
    V30['C'] = MEMORIZATION
    write_json(RUN_DIR/'v3_0_diagnostics'/'diagnostic_C_memorization.json', MEMORIZATION)

if V30:
    write_json(RUN_DIR/'v3_0_diagnostics'/'v3_0_summary.json', V30)
    print('\nV3.0 completion check:',
          {'A_collapse_measured': 'A' in V30, 'B_gradients_measured': 'B' in V30, 'C_memorization_measured': 'C' in V30})

## Information tests — run before the expensive selection

V3.0 showed that the learning system works: V3 memorized 128 anchors by update 50. It also showed that the live side agrees with the label direction on only 49% of candles. These cells measure whether the information needed for the 80% target exists at candle close **before** any large training run.

1. **Direction study (CPU).** How well can the label's long/short direction be predicted from the 97 causal features? It is compared with confirmed-pivot ZigZag sides at several depths and with `signal_live`. Accuracy is also broken down by how many candles have passed since the true pivot (hindsight grouping only), which shows the confirmation-lag effect directly.
2. **Tabular information test (CPU).** A distributional gradient-boosted model over log-Z bins, decoded with the **same exact strict-interval decoder** as V3, plus a log-Z regression for ranking. It runs with V2's 81 features, with V3's 97, and with 97 + the **ORACLE** label direction. The oracle variant can never be deployed; it measures how much knowing the direction would be worth.
3. **V3 screen (GPU).** The V3 distribution head with the auxiliary loss on the latest development window, with direction and leg-state conditioning on vs off, on a short budget.

Folds: the three development windows (training only on labels resolved before each window), plus the already-inspected Jul–Oct 2026 period shown for reference only. Verdicts use the development windows. They are rules declared here before the run, not proofs.

In [22]:
# Information tests: definitions only. Hindsight (label) quantities appear ONLY as targets, grouping
# variables or explicitly named ORACLE inputs; none of them is ever used by a V3 model.

def _ns_int(series):
    """tz-aware datetime Series -> int64 nanoseconds (-1 for NaT)."""
    valid = series.notna().to_numpy()
    out = np.full(len(series), -1, dtype=np.int64)
    if valid.any():
        out[valid] = series[valid].dt.tz_convert("UTC").dt.tz_localize(None).to_numpy().astype("datetime64[ns]").astype(np.int64)
    return out

def bars_since_true_pivot(asset):
    """Label-derived (hindsight) position inside the true leg: bars since the previous original endpoint pivot."""
    raw = asset.raw
    if "tp_pivot_time" not in raw:
        return np.full(len(raw), np.nan)
    pv = _ns_int(raw.tp_pivot_time)
    uniq = np.unique(pv[pv >= 0])
    k = np.searchsorted(uniq, pv)
    start = np.where((pv >= 0) & (k > 0), uniq[np.maximum(k-1, 0)], -1)
    t = _ns_int(raw.time)
    return np.where(start >= 0, (t-start)/(15*60*1e9), np.nan)

SINCE_BINS = [-0.5, 3.5, 7.5, 11.5, 15.5, 23.5, 47.5, np.inf]
SINCE_LABELS = ["0-3", "4-7", "8-11", "12-15", "16-23", "24-47", "48+"]

def information_folds(windows, prepared_outer, eval_idx, cfg):
    """Chronological folds: each development window (train before it, test on it) + the inspected outer period."""
    folds = [{"fold": f"dev_{i+1}_{w.start[:10]}", "role": "development_window",
              "train": np.asarray(w.fit.train_idx[cfg.primary_symbol]), "test": np.asarray(w.val_idx)}
             for i, w in enumerate(windows)]
    folds.append({"fold": "outer_jul_oct_2026", "role": "inspected_development_period",
                  "train": np.asarray(prepared_outer.train_idx[cfg.primary_symbol]), "test": np.asarray(eval_idx)})
    return folds

def chronological_split(asset, train_idx, cfg, es_fraction=0.1, max_rows=None):
    """Last es_fraction of training anchors (after an embargo) is the early-stopping set; earlier rows must
    have labels resolved before it starts. Optional even thinning keeps chronology."""
    idx = np.asarray(train_idx)
    cut = int(len(idx)*(1-es_fraction))
    fit, es = idx[:cut], idx[cut:]
    es = es[es > fit[-1]+cfg.anchor_gap_bars] if len(fit) else es
    if len(es):
        fit = fit[(asset.raw.label_available_at.iloc[fit] < asset.raw.time.iloc[es[0]]).to_numpy()]
    if max_rows and len(fit) > max_rows:
        fit = fit[np.linspace(0, len(fit)-1, max_rows).astype(int)]
    return fit, es

def binary_auc(y, score):
    y, score = np.asarray(y, bool), np.asarray(score, float)
    pos, neg = y.sum(), (~y).sum()
    if pos == 0 or neg == 0:
        return float("nan")
    ranks = pd.Series(score).rank().to_numpy()
    return float((ranks[y].sum()-pos*(pos+1)/2)/(pos*neg))

def block_accuracy_ci(correct, blocks, reps=1000, seed=12345):
    d = pd.DataFrame({"b": np.asarray(blocks), "c": np.asarray(correct, int)}).groupby("b").c.agg(["sum", "count"]).to_numpy()
    rng = np.random.default_rng(seed)
    s = d[rng.integers(len(d), size=(reps, len(d)))].sum(axis=1)
    r = s[:, 0]/s[:, 1]
    return float(np.percentile(r, 2.5)), float(np.percentile(r, 97.5))

LGB_COMMON = dict(learning_rate=0.05, num_leaves=31, min_data_in_leaf=200, feature_fraction=0.8,
                  bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, num_threads=0, seed=7)

def lgb_train(params, x_fit, y_fit, x_es, y_es, rounds):
    import lightgbm as lgb
    train = lgb.Dataset(x_fit, y_fit, free_raw_data=True)
    valid = lgb.Dataset(x_es, y_es, reference=train)
    return lgb.train({**LGB_COMMON, **params}, train, num_boost_round=rounds, valid_sets=[valid],
                     callbacks=[lgb.early_stopping(30, verbose=False)])

# ---------------- Step 1: direction predictability ----------------

def zigzag_side_for_depth(asset, depth):
    c = asset.raw.close.to_numpy(float)
    atr = asset.atr_pct*c/100
    _, side, _ = causal_zigzag_state(asset.raw.high.to_numpy(float), asset.raw.low.to_numpy(float), c, atr, depth, 0.0)
    return side

def direction_study(asset, folds, cfg, depths, rounds, max_rows, output_dir):
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    label = asset.raw.label_side.to_numpy()
    legs = asset.raw.leg_id.to_numpy()
    since = bars_since_true_pivot(asset)
    rules = {f"causal_zigzag_depth_{d}": zigzag_side_for_depth(asset, d) for d in depths}
    if "signal_live" in asset.raw:
        rules["signal_live_column"] = parse_side(asset.raw.signal_live)
    x_all = asset.features.to_numpy(np.float32)
    rows, by_since = [], []
    for f in folds:
        test = f["test"][label[f["test"]] != 0]
        fit, es = chronological_split(asset, f["train"][label[f["train"]] != 0], cfg, max_rows=max_rows)
        model = lgb_train(dict(objective="binary", metric="binary_logloss"), x_all[fit], (label[fit] > 0).astype(int),
                          x_all[es], (label[es] > 0).astype(int), rounds)
        p_long = model.predict(x_all[test], num_iteration=model.best_iteration)
        pred = np.where(p_long >= 0.5, 1, -1)
        y = label[test]
        correct = pred == y
        lo, hi = block_accuracy_ci(correct, legs[test])
        train_long = float(np.mean(label[fit] > 0))
        row = {"fold": f["fold"], "role": f["role"], "test_rows": int(len(test)), "trees": int(model.best_iteration),
               "classifier_accuracy": float(correct.mean()), "classifier_leg_ci95_low": lo, "classifier_leg_ci95_high": hi,
               "classifier_auc": binary_auc(y > 0, p_long),
               "classifier_logloss": float(-np.mean(np.where(y > 0, np.log(np.clip(p_long, 1e-9, 1)), np.log(np.clip(1-p_long, 1e-9, 1))))),
               "majority_baseline_accuracy": float(np.mean(y == (1 if train_long >= .5 else -1)))}
        for name, side in rules.items():
            s = side[test]
            row[f"{name}_accuracy"] = float(np.mean(s == y))
        rows.append(row)
        g = pd.DataFrame({"since": pd.cut(since[test], SINCE_BINS, labels=SINCE_LABELS), "clf": correct,
                          **{name: side[test] == y for name, side in rules.items()}})
        t = g.groupby("since", observed=True).mean().reset_index()
        t.insert(1, "rows", g.groupby("since", observed=True).size().to_numpy())
        t.insert(0, "fold", f["fold"])
        by_since.append(t)
        imp = pd.Series(model.feature_importance("gain"), index=asset.features.columns).sort_values(ascending=False)
        imp.head(25).to_csv(out/f"direction_importance_{f['fold']}.csv")
    table = pd.DataFrame(rows)
    since_table = pd.concat(by_since, ignore_index=True).rename(columns={"clf": "classifier"})
    table.to_csv(out/"direction_predictability.csv", index=False)
    since_table.to_csv(out/"direction_accuracy_by_bars_since_true_pivot.csv", index=False)
    dev = table[table.role == "development_window"]
    pooled = float((dev.classifier_accuracy*dev.test_rows).sum()/dev.test_rows.sum())
    verdict = ("NOT predictable at candle close" if pooled < 0.55 else "weakly predictable" if pooled < 0.65 else "usable")
    summary = {"pooled_dev_classifier_accuracy": pooled, "verdict": verdict,
               "verdict_rule": "pooled development-window accuracy: <0.55 not predictable, 0.55-0.65 weak, >=0.65 usable",
               "best_rule_accuracies": {c: float((dev[c]*dev.test_rows).sum()/dev.test_rows.sum())
                                        for c in dev.columns if c.endswith("_accuracy") and c != "classifier_accuracy"}}
    write_json(out/"direction_summary.json", summary)
    return table, since_table, summary

# ---------------- Step 2: tabular information test for the endpoint magnitude ----------------

def gbdt_information_test(asset, folds, cfg, n_bins, rounds, max_rows, output_dir):
    """Distributional GBDT on current-row features, decoded with the SAME exact strict-interval decoder.
    Variants: V2 81 features; all 97 V3 features; all 97 + ORACLE label direction (hindsight; never deployable)."""
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    x97 = asset.features.to_numpy(np.float32)
    oracle = asset.raw.label_side.to_numpy().astype(np.float32)[:, None]
    variants = {"v2_81_features": x97[:, :BASE_FEATURE_COUNT], "v3_97_features": x97,
                "v3_97_plus_ORACLE_direction": np.concatenate([x97, oracle], axis=1)}
    z, legs = asset.z, asset.raw.leg_id.to_numpy()
    bcfg = replace(cfg, dist_bins=n_bins)
    rows, preds = [], []
    for f in folds:
        fit, es = chronological_split(asset, f["train"], cfg, max_rows=max_rows)
        test = f["test"]
        bins = LogZBins.fit(z[fit], bcfg)
        edges = np.asarray(bins.edges)
        def cls(i):
            u = np.log(np.clip(z[i], 1e-300, None))
            return np.clip(np.searchsorted(edges, np.clip(u, edges[0], edges[-1]), side="right")-1, 0, bins.count-1)
        pos_fit, pos_es = fit[z[fit] > 0], es[z[es] > 0]
        p0 = float(np.mean(z[fit] == 0))
        fixed = best_constant(z[f["train"]], cfg.lower_ratio)
        fixed_hits = strict_hits(z[test], np.full(len(test), fixed), cfg.lower_ratio)
        for name, x in variants.items():
            mc = lgb_train(dict(objective="multiclass", num_class=bins.count, metric="multi_logloss"),
                           x[pos_fit], cls(pos_fit), x[pos_es], cls(pos_es), rounds)
            probs = mc.predict(x[test], num_iteration=mc.best_iteration)
            sel, score, cdf = decode_distribution(torch.full((len(test),), p0, dtype=torch.float64),
                                                  torch.tensor(probs, dtype=torch.float64),
                                                  torch.tensor(edges, dtype=torch.float64), cfg.lower_ratio)
            median, _ = distribution_summaries(cdf, torch.tensor(edges, dtype=torch.float64), np.full(len(test), np.nan))
            reg = lgb_train(dict(objective="regression", metric="l2"), x[pos_fit], np.log(z[pos_fit]),
                            x[pos_es], np.log(z[pos_es]), rounds)
            reg_pred = reg.predict(x[test], num_iteration=reg.best_iteration)
            zhat = sel.numpy()
            hits = strict_hits(z[test], zhat, cfg.lower_ratio)
            paired = paired_report(name, hits, "fixed_atr", fixed_hits, asset.raw.time.iloc[test], cfg, legs=legs[test])
            lz = np.log(zhat[zhat > 0])
            pos = z[test] > 0
            r2 = 1-np.mean((np.log(z[test][pos])-reg_pred[pos])**2)/np.var(np.log(z[test][pos]))
            rows.append({"fold": f["fold"], "role": f["role"], "variant": name, "test_rows": int(len(test)),
                         "train_rows_used": int(len(fit)), "trees_dist": int(mc.best_iteration), "trees_reg": int(reg.best_iteration),
                         "strict_accuracy": float(hits.mean()), "fixed_atr_accuracy": float(fixed_hits.mean()),
                         "lift_pp": paired["difference_pp"], "lift_leg_ci95_low": paired["leg_block_ci95_low_pp"],
                         "lift_leg_ci95_high": paired["leg_block_ci95_high_pp"],
                         "spearman_regression_logz": spearman(reg_pred[pos], z[test][pos]),
                         "spearman_distribution_median": spearman(median.numpy()[pos], z[test][pos]),
                         "r2_log_z_out_of_sample": float(r2),
                         "decoded_log_iqr": float(np.subtract(*np.percentile(lz, [75, 25]))) if len(lz) > 1 else None,
                         "decoded_span_in_strict_bands": float(np.subtract(*np.percentile(lz, [99, 1]))/STRICT_BAND_LOG) if len(lz) > 1 else None,
                         "mean_selected_probability": float(score.mean())})
            preds.append(pd.DataFrame({"fold": f["fold"], "variant": name, "time": asset.raw.time.iloc[test].to_numpy(),
                                       "actual_z": z[test], "predicted_z": zhat, "regression_z": np.exp(reg_pred),
                                       "strict_hit": hits, "fixed_atr_hit": fixed_hits}))
            print(f"{f['fold']:>24} | {name:<28} strict {hits.mean():.4f} vs fixed ATR {fixed_hits.mean():.4f} | "
                  f"Spearman {rows[-1]['spearman_regression_logz']:.3f} | R2 {r2:.3f}")
    table = pd.DataFrame(rows)
    table.to_csv(out/"gbdt_information_test.csv", index=False)
    pd.concat(preds, ignore_index=True).to_csv(out/"gbdt_predictions.csv", index=False)
    dev = table[table.role == "development_window"]
    def pooled(v, col):
        d = dev[dev.variant == v]
        return float((d[col]*d.test_rows).sum()/d.test_rows.sum())
    summary = {v: {"pooled_strict_accuracy": pooled(v, "strict_accuracy"), "pooled_fixed_atr": pooled(v, "fixed_atr_accuracy"),
                   "mean_spearman": float(dev[dev.variant == v].spearman_regression_logz.mean()),
                   "mean_r2_log_z": float(dev[dev.variant == v].r2_log_z_out_of_sample.mean()),
                   "windows_beating_fixed_atr": int((dev[dev.variant == v].lift_pp > 0).sum())}
               for v in variants}
    deployable = summary["v3_97_features"]
    summary["verdict"] = ("information-limited: full selection is unlikely to approach 80%"
                          if deployable["mean_spearman"] < 0.2 or deployable["pooled_strict_accuracy"]-deployable["pooled_fixed_atr"] < 0.01
                          else "usable tabular signal: proceed to V3 selection")
    summary["verdict_rule"] = "deployable 97-feature variant: mean Spearman >= 0.2 AND pooled lift over fixed ATR >= 1 pp"
    summary["oracle_direction_gain_pp"] = 100*(summary["v3_97_plus_ORACLE_direction"]["pooled_strict_accuracy"]-deployable["pooled_strict_accuracy"])
    write_json(out/"gbdt_summary.json", summary)
    return table, summary

In [23]:
INFO_DIR = RUN_DIR/'information_tests'
INFO = {}
if RUN_DIRECTION_STUDY or RUN_INFORMATION_TEST:
    INFO_WINDOWS = prepare_dev_windows(ASSETS, CFG, DEV_WINDOW_STARTS, EVAL_START)
    INFO_FOLDS = information_folds(INFO_WINDOWS, PREFLIGHT, EVAL_INDICES, CFG)
    display(pd.DataFrame([{'fold': f['fold'], 'role': f['role'], 'train_anchors': len(f['train']), 'test_anchors': len(f['test'])}
                          for f in INFO_FOLDS]))
if RUN_DIRECTION_STUDY:
    DIR_TABLE, DIR_SINCE, DIR_SUMMARY = direction_study(PRIMARY, INFO_FOLDS, CFG, INFO_ZIGZAG_DEPTHS,
                                                        INFO_GBDT_ROUNDS, INFO_MAX_TRAIN_ROWS, INFO_DIR/'direction')
    INFO['direction'] = DIR_SUMMARY
    pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
    display(DIR_TABLE)
    dev_since = DIR_SINCE[~DIR_SINCE.fold.str.startswith('outer')]
    rate_cols = [c for c in dev_since.columns if c not in ('fold', 'since', 'rows')]
    since_key = dev_since.since.astype(str)
    pooled_since = (dev_since[rate_cols].astype(float).mul(dev_since.rows, axis=0).groupby(since_key).sum()
                    .div(dev_since.groupby(since_key).rows.sum(), axis=0))
    pooled_since['rows'] = dev_since.groupby(since_key).rows.sum()
    pooled_since = pooled_since.reindex([l for l in SINCE_LABELS if l in pooled_since.index])
    print('Accuracy by bars since the TRUE pivot (development windows, hindsight grouping only):')
    display(pooled_since)
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
        for c in [c for c in pooled_since.columns if c != 'rows']:
            ax.plot(pooled_since.index.astype(str), 100*pooled_since[c], marker='o', label=c)
        ax.axhline(50, color='black', linestyle=':', linewidth=1)
        ax.set_xlabel('Bars since the true (hindsight) pivot'); ax.set_ylabel('Direction accuracy (%)'); ax.legend(fontsize=8)
        fig.savefig(INFO_DIR/'direction'/'direction_by_bars_since_pivot.png', dpi=130); plt.show()
    except Exception as exc:
        print('Plot skipped:', exc)
    print('Direction verdict:', DIR_SUMMARY['verdict'], f"(pooled classifier accuracy {DIR_SUMMARY['pooled_dev_classifier_accuracy']:.4f})")

In [24]:
if RUN_INFORMATION_TEST:
    GBDT_TABLE, GBDT_SUMMARY = gbdt_information_test(PRIMARY, INFO_FOLDS, CFG, INFO_GBDT_BINS, INFO_GBDT_ROUNDS,
                                                     INFO_MAX_TRAIN_ROWS, INFO_DIR/'gbdt')
    INFO['gbdt'] = GBDT_SUMMARY
    display(GBDT_TABLE[['fold', 'variant', 'strict_accuracy', 'fixed_atr_accuracy', 'lift_pp', 'lift_leg_ci95_low',
                        'lift_leg_ci95_high', 'spearman_regression_logz', 'r2_log_z_out_of_sample', 'decoded_span_in_strict_bands']])
    display(pd.DataFrame({k: v for k, v in GBDT_SUMMARY.items() if isinstance(v, dict)}).T)
    print('Information verdict:', GBDT_SUMMARY['verdict'])
    print(f"Value of knowing the label direction (ORACLE, not deployable): {GBDT_SUMMARY['oracle_direction_gain_pp']:+.2f} pp strict accuracy")

In [25]:
if RUN_V3_SCREEN:
    screen_windows = prepare_dev_windows(ASSETS, CFG, DEV_WINDOW_STARTS[-1:], EVAL_START)
    screen_base = replace(CFG, head_type='distribution', aux_lambda=0.10, magnitude_head=True, max_epochs=INFO_V3_EPOCHS,
                          patience=3, max_train_anchors_per_epoch=INFO_V3_ANCHORS_PER_EPOCH)
    screen_variants = {
        'conditioning_on': screen_base,
        'conditioning_off': replace(screen_base, use_state_features=False, use_film=False, use_state_fusion=False, state_in_sequence=False),
    }
    rows = []
    for name, c in screen_variants.items():
        print(f'\n=== V3 screen: {name} on window {screen_windows[0].start[:10]} ===')
        s = train_candidate_on_windows(c.validate(), screen_windows, c.seeds[0], INFO_DIR/'v3_screen'/name, DEVICE)
        w = s['windows'][0]
        rows.append({'variant': name, 'best_epoch': w['best_epoch'], 'validation_rows': w['validation_rows'],
                     'strict_accuracy': w['strict_accuracy'], 'fixed_atr_accuracy': w['fixed_atr_accuracy'],
                     'lift_pp': w['lift_vs_fixed_atr_pp'], 'val_spearman': w['val_spearman'], 'val_aux_spearman': w['val_aux_spearman'],
                     'val_log_z_std': w['val_predicted_log_z_std'], 'train_probe_log_z_std': w['train_probe_predicted_log_z_std'],
                     'generalization_gap_pp': w['generalization_gap_pp']})
    SCREEN = pd.DataFrame(rows)
    SCREEN.to_csv(INFO_DIR/'v3_screen_summary.csv', index=False)
    INFO['v3_screen'] = SCREEN.to_dict('records')
    display(SCREEN)
    for name in screen_variants:
        hist = sorted((INFO_DIR/'v3_screen'/name).glob('candidate_*/window_*/history.json')) or \
               sorted((INFO_DIR/'v3_screen'/name).glob('window_*/history.json'))
        if hist:
            plot_training_history(hist[0], INFO_DIR/f'v3_screen_{name}_history.png')

if INFO:
    previous = json.loads((INFO_DIR/'information_summary.json').read_text()) if (INFO_DIR/'information_summary.json').is_file() else {}
    previous.update(INFO)
    write_json(INFO_DIR/'information_summary.json', previous)
    print('\n================ INFORMATION TEST SUMMARY ================')
    if 'direction' in INFO:
        print('Direction at candle close:', INFO['direction']['verdict'])
    if 'gbdt' in INFO:
        g = INFO['gbdt']
        print('Endpoint magnitude (tabular):', g['verdict'])
        print(f"  97 features: pooled strict {g['v3_97_features']['pooled_strict_accuracy']:.4f} vs fixed ATR "
              f"{g['v3_97_features']['pooled_fixed_atr']:.4f}; mean Spearman {g['v3_97_features']['mean_spearman']:.3f}")
        print(f"  with ORACLE direction: pooled strict {g['v3_97_plus_ORACLE_direction']['pooled_strict_accuracy']:.4f}")
    if 'v3_screen' in INFO:
        for r in INFO['v3_screen']:
            print(f"V3 screen {r['variant']}: strict {r['strict_accuracy']:.4f} vs fixed ATR {r['fixed_atr_accuracy']:.4f}, "
                  f"Spearman {r['val_spearman']}")
    print('Proceed to RUN_V3_SELECTION only if the tabular or V3-screen result shows a real out-of-sample lift.')

## Cross-asset information, the combined system and the strict–reach frontier

The contract is unchanged: a TP for **every** candle, judged on **both** strict endpoint accuracy and TP reach. No extra data types are available, so this section adds information from **other symbols**: their returns, volatility, volume, taker flow and causal swing state at the same candle close, plus alt-market aggregates. These are BTC model inputs only. Other symbols cannot add training labels without your original ZigZag generator.

Data: Binance spot 15-minute klines from `data.binance.vision`, downloaded in Colab and cached under `OUTPUT_ROOT/cross_assets/`, or your own CSVs via `CROSS_ASSET_FILES`. Each symbol is aligned to the BTC candle grid. A missing bar repeats the last close with zero volume and is counted; bars before a listing stay NaN. A prefix-replay audit checks causality.

**Combined system (fast tree models, same chronological folds):**
- Direction: a classifier gives LONG/SHORT for every candle.
- TP: a log-Z distribution, decoded several ways — the exact strict decoder, distribution quantiles 0.05–0.60 (more conservative TPs), and fixed ATR multiples.
- Each decision is scored on strict accuracy (all candles) and on nominal reach in the **predicted** direction within `REACH_SETTINGS['max_hold_bars']`, after the spread buffer and without a stop (all candles with enough future data). Reach in the ZigZag direction and in the hindsight **ORACLE** direction are reported for reference only.

The frontier plot puts every decision family on the same axes as the requirement (98% reach, 80% strict). The table shows the best strict accuracy achievable at each reach level. The verdicts use the development windows only.

In [26]:
# Cross-asset information and the strict-accuracy / reach frontier: definitions only.
import io
import urllib.request
import urllib.error
import zipfile

KLINE_COLUMNS = ["open_time", "open", "high", "low", "close", "volume", "close_time", "quote_volume",
                 "num_trades", "taker_base_volume", "taker_quote_volume", "ignore"]

def parse_kline_csv(data):
    """Binance kline CSV (spot files have no header; some have one). Open time may be ms or us (spot since 2025)."""
    frame = pd.read_csv(io.BytesIO(data), header=None)
    if not np.issubdtype(frame.iloc[:, 0].dtype, np.number):
        frame = frame.iloc[1:].reset_index(drop=True)
    frame = frame.iloc[:, :12]
    frame.columns = KLINE_COLUMNS
    t = pd.to_numeric(frame.open_time)
    unit = np.where(t > 1e14, "us", "ms")
    frame["time"] = [pd.Timestamp(int(v), unit=u, tz="UTC") for v, u in zip(t, unit)] if len(set(unit)) > 1 else \
        pd.to_datetime(t, unit=unit[0], utc=True)
    out = frame[["time", *RAW_COLUMNS]].copy()
    for c in RAW_COLUMNS:
        out[c] = pd.to_numeric(out[c], errors="coerce").astype(np.float64)
    return out

def _fetch(url, path):
    if path.is_file():
        return path.read_bytes()
    with urllib.request.urlopen(url, timeout=60) as r:
        data = r.read()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
    return data

def _read_zip(data):
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        return parse_kline_csv(z.read(z.namelist()[0]))

def load_binance_klines(symbol, start, end, cache_dir, base="https://data.binance.vision/data/spot", interval="15m"):
    """Monthly archives, falling back to daily archives for months not yet published. Cached as zip files."""
    cache = Path(cache_dir)/symbol
    frames, missing = [], []
    for month in pd.period_range(utc(start).tz_localize(None), utc(end).tz_localize(None), freq="M"):
        name = f"{symbol}-{interval}-{month.year}-{month.month:02d}"
        try:
            frames.append(_read_zip(_fetch(f"{base}/monthly/klines/{symbol}/{interval}/{name}.zip", cache/f"{name}.zip")))
            continue
        except (urllib.error.HTTPError, urllib.error.URLError, FileNotFoundError, OSError):
            pass
        for day in pd.date_range(month.start_time, month.end_time.normalize(), freq="D"):
            if day > utc(end).tz_localize(None):
                break
            dname = f"{symbol}-{interval}-{day:%Y-%m-%d}"
            try:
                frames.append(_read_zip(_fetch(f"{base}/daily/klines/{symbol}/{interval}/{dname}.zip", cache/f"{dname}.zip")))
            except (urllib.error.HTTPError, urllib.error.URLError, FileNotFoundError, OSError):
                missing.append(dname)
    if not frames:
        raise FileNotFoundError(f"No kline data for {symbol}. Supply CROSS_ASSET_FILES or check internet access.")
    d = pd.concat(frames, ignore_index=True).drop_duplicates("time").sort_values("time").reset_index(drop=True)
    return d, missing

def align_to_primary(other, primary_times):
    """Reindex another symbol to the primary's candle grid. A missing bar repeats the last close (causal) with zero
    volume and is flagged; bars before the symbol existed stay NaN (GBDTs handle NaN)."""
    o = other.set_index("time").reindex(pd.DatetimeIndex(primary_times))
    present = o.close.notna().to_numpy()
    started = np.maximum.accumulate(present)
    close = o.close.ffill()
    for c in ("open", "high", "low"):
        o[c] = o[c].where(present, close)
    o["close"] = close
    for c in ("volume", "quote_volume", "num_trades", "taker_base_volume", "taker_quote_volume"):
        o[c] = o[c].where(present, 0.0)
    o.loc[~started, list(RAW_COLUMNS)] = np.nan
    o = o.reset_index(names="time")
    return o, {"bars": int(len(o)), "missing_filled": int((started & ~present).sum()), "before_listing": int((~started).sum())}

def cross_asset_features(primary_raw, primary_atr_pct, aligned):
    """Causal features from other symbols' candles that close at the same instant as the BTC anchor."""
    c0 = primary_raw.close.astype(float).reset_index(drop=True)
    lr0 = {k: np.log(c0/c0.shift(k)) for k in (1, 4, 16, 64, 96)}
    f, alt_ret = {}, {k: [] for k in (1, 16, 64)}
    alt_side = []
    for sym, o in aligned.items():
        s = sym.replace("USDT", "").lower()
        c, h, l = (o[k].astype(float) for k in ("close", "high", "low"))
        tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
        atr = tr.ewm(alpha=1/14, adjust=False).mean()
        lr = {k: np.log(c/c.shift(k)) for k in (1, 4, 16, 64, 96)}
        for k in (1, 4, 16, 64, 96):
            f[f"x_{s}_logret_{k}"] = lr[k]
        for k in (16, 64):
            f[f"x_{s}_displacement_atr_{k}"] = (c-c.shift(k))/atr
        f[f"x_{s}_atr_ratio_vs_primary"] = safe_ratio(100*atr/c, primary_atr_pct, 1)
        for k in (4, 16, 64):
            f[f"x_{s}_relative_ret_{k}"] = lr[k]-lr0[k]
        lv = np.log1p(o.volume.astype(float))
        f[f"x_{s}_volume_z96"] = safe_ratio(lv-lv.rolling(96).mean(), lv.rolling(96).std(ddof=0), 0)
        imb = pd.Series(safe_ratio(2*o.taker_base_volume.astype(float)-o.volume.astype(float), o.volume.astype(float), 0))
        f[f"x_{s}_taker_imbalance_16"] = imb.rolling(16).mean()
        ok = c.notna().to_numpy()
        state = np.full((len(o), 12), np.nan); side = np.zeros(len(o))
        if ok.any():
            first = int(np.argmax(ok))
            st, sd, _ = causal_zigzag_state(h.to_numpy()[first:], l.to_numpy()[first:], c.to_numpy()[first:],
                                            atr.to_numpy()[first:], 12, 0.0)
            state[first:], side[first:] = st, sd
        f[f"x_{s}_zz_side"] = side
        f[f"x_{s}_zz_leg_age_log"] = state[:, 3]
        f[f"x_{s}_zz_leg_extension_atr"] = state[:, 4]
        alt_side.append(side)
        for k in alt_ret:
            alt_ret[k].append(lr[k].to_numpy())
        f[f"x_{s}_corr96_with_primary"] = lr[1].rolling(96).corr(lr0[1])
    import warnings
    warnings.filterwarnings("ignore", message="Mean of empty slice")
    warnings.filterwarnings("ignore", message="Degrees of freedom <= 0")
    for k, v in alt_ret.items():
        m = np.vstack(v)
        f[f"x_alt_mean_logret_{k}"] = np.nanmean(m, axis=0)
        f[f"x_alt_dispersion_{k}"] = np.nanstd(m, axis=0)
        f[f"x_primary_minus_alt_mean_{k}"] = lr0[k].to_numpy()-np.nanmean(m, axis=0)
    f["x_alt_long_side_share"] = np.mean(np.vstack(alt_side) > 0, axis=0)
    return pd.DataFrame(f, index=primary_raw.index).astype(np.float64)

def audit_cross_features(primary_raw, primary_atr_pct, aligned, full, cuts):
    """Prefix replay: features at row t computed from data <= t must equal the full-history features."""
    for t in cuts:
        pre = cross_asset_features(primary_raw.iloc[:t+1], primary_atr_pct[:t+1], {s: o.iloc[:t+1] for s, o in aligned.items()})
        np.testing.assert_allclose(pre.iloc[-1].to_numpy(), full.iloc[t].to_numpy(), rtol=1e-9, atol=1e-9, equal_nan=True)
    return {"prefix_positions_tested": list(map(int, cuts)), "passed": True}

# ---------------- Combined direction + TP system and the frontier ----------------

def future_extremes(asset, horizon):
    from numpy.lib.stride_tricks import sliding_window_view
    h, l = asset.raw.high.to_numpy(float), asset.raw.low.to_numpy(float)
    n = len(h)
    fmax, fmin = np.full(n, np.nan), np.full(n, np.nan)
    if n > horizon:
        fmax[:n-horizon] = sliding_window_view(h[1:], horizon).max(axis=1)[:n-horizon]
        fmin[:n-horizon] = sliding_window_view(l[1:], horizon).min(axis=1)[:n-horizon]
    return fmax, fmin

def reach_vector(asset, idx, tp_pct, side, fmax, fmin, spread_bps):
    """TP touched within the horizon from the anchor close, in the given direction, clearing the spread buffer.
    Equals simulate_reach without a stop. Zero TP or unknown side = no order = not reached. NaN = pending."""
    c = asset.raw.close.to_numpy(float)[idx]
    tp, s = np.asarray(tp_pct, float), np.asarray(side, float)
    need = c*(1+s*tp/100)*(1+s*spread_bps/1e4)
    hit = np.where(s > 0, fmax[idx] >= need, fmin[idx] <= need) & (tp > 0) & (s != 0)
    pending = ~np.isfinite(fmax[idx])
    return np.where(pending, np.nan, hit.astype(float))

def cdf_quantile(cdf_edges, edges, q):
    """Inverse of the piecewise-linear log-Z CDF at level q -> Z."""
    cdf = cdf_edges.numpy() if hasattr(cdf_edges, "numpy") else np.asarray(cdf_edges)
    e = np.asarray(edges, float)
    j = np.clip((cdf < q).sum(axis=1)-1, 0, len(e)-2)
    c0, c1 = cdf[np.arange(len(cdf)), j], cdf[np.arange(len(cdf)), j+1]
    frac = np.clip((q-c0)/np.maximum(c1-c0, 1e-300), 0, 1)
    return np.exp(e[j]+(e[j+1]-e[j])*frac)

def combined_system_frontier(asset, folds, cfg, feature_sets, n_bins, rounds, max_rows, horizon, spread_bps,
                             quantiles, atr_multiples, output_dir):
    """Every candle gets a direction (GBDT classifier) and a TP (GBDT log-Z distribution). Decision families:
    the exact strict decoder, distribution quantiles, and fixed ATR multiples. Each is scored on BOTH strict endpoint
    accuracy (all candles) and nominal reach in the predicted direction (all non-pending candles)."""
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    fmax, fmin = future_extremes(asset, horizon)
    label = asset.raw.label_side.to_numpy()
    z, y, atr = asset.z, asset.y, asset.atr_pct
    rows, info = [], []
    for fs_name, X in feature_sets.items():
        bcfg = replace(cfg, dist_bins=n_bins)
        for f in folds:
            fit, es = chronological_split(asset, f["train"], cfg, max_rows=max_rows)
            test = f["test"]
            dfit, des = fit[label[fit] != 0], es[label[es] != 0]
            clf = lgb_train(dict(objective="binary", metric="binary_logloss"), X[dfit], (label[dfit] > 0).astype(int),
                            X[des], (label[des] > 0).astype(int), rounds)
            p_long = clf.predict(X[test], num_iteration=clf.best_iteration)
            side_pred = np.where(p_long >= 0.5, 1, -1)
            bins = LogZBins.fit(z[fit], bcfg)
            edges = np.asarray(bins.edges)
            def cls(i):
                u = np.log(np.clip(z[i], 1e-300, None))
                return np.clip(np.searchsorted(edges, np.clip(u, edges[0], edges[-1]), side="right")-1, 0, bins.count-1)
            pf, pe = fit[z[fit] > 0], es[z[es] > 0]
            mc = lgb_train(dict(objective="multiclass", num_class=bins.count, metric="multi_logloss"),
                           X[pf], cls(pf), X[pe], cls(pe), rounds)
            probs = mc.predict(X[test], num_iteration=mc.best_iteration)
            p0 = float(np.mean(z[fit] == 0))
            et = torch.tensor(edges, dtype=torch.float64)
            sel, _, cdf = decode_distribution(torch.full((len(test),), p0, dtype=torch.float64),
                                              torch.tensor(probs, dtype=torch.float64), et, cfg.lower_ratio)
            median, _ = distribution_summaries(cdf, et, np.full(len(test), np.nan))
            fixed = best_constant(z[f["train"]], cfg.lower_ratio)
            decisions = {"strict_decoder": sel.numpy()}
            decisions.update({f"quantile_{q:.2f}": cdf_quantile(cdf, edges, q) for q in quantiles})
            decisions.update({f"fixed_atr_{k:.3f}": np.full(len(test), k) for k in atr_multiples})
            decisions["fixed_atr_train_best"] = np.full(len(test), fixed)
            zz_side, truth = asset.side[test], label[test]
            dir_correct = side_pred == truth
            info.append({"feature_set": fs_name, "fold": f["fold"], "role": f["role"], "test_rows": int(len(test)),
                         "direction_accuracy": float(dir_correct[truth != 0].mean()),
                         "direction_auc": binary_auc(truth[truth != 0] > 0, p_long[truth != 0]),
                         "spearman_distribution_median": spearman(median.numpy()[z[test] > 0], z[test][z[test] > 0])})
            for name, zhat in decisions.items():
                tp = zhat*atr[test]
                hits = strict_hits(y[test], tp, cfg.lower_ratio)
                rows.append(pd.DataFrame({
                    "feature_set": fs_name, "fold": f["fold"], "role": f["role"], "decision": name,
                    "family": "fixed_train_best" if name == "fixed_atr_train_best" else name.split("_")[0],
                    "row": test, "strict_hit": hits,
                    "reach_predicted_side": reach_vector(asset, test, tp, side_pred, fmax, fmin, spread_bps),
                    "reach_zigzag_side": reach_vector(asset, test, tp, zz_side, fmax, fmin, spread_bps),
                    "reach_ORACLE_label_side": reach_vector(asset, test, tp, truth, fmax, fmin, spread_bps),
                    "direction_correct": dir_correct}))
            print(f"{fs_name:>22} | {f['fold']:>20} | direction {info[-1]['direction_accuracy']:.4f} | "
                  f"strict decoder {strict_hits(y[test], sel.numpy()*atr[test]).mean():.4f} "
                  f"vs fixed ATR {strict_hits(y[test], fixed*atr[test]).mean():.4f}")
    detail = pd.concat(rows, ignore_index=True)
    info = pd.DataFrame(info)
    info.to_csv(out/"combined_direction_info.csv", index=False)
    def summarize(d):
        return (d.groupby(["feature_set", "decision", "family"])
                 .agg(rows=("strict_hit", "size"), strict_accuracy=("strict_hit", "mean"),
                      reach_predicted_side=("reach_predicted_side", "mean"), reach_zigzag_side=("reach_zigzag_side", "mean"),
                      reach_ORACLE_label_side=("reach_ORACLE_label_side", "mean"))
                 .reset_index())
    dev = summarize(detail[detail.role == "development_window"])
    outer = summarize(detail[detail.role != "development_window"])
    dev.to_csv(out/"frontier_development_windows.csv", index=False)
    outer.to_csv(out/"frontier_outer_reference.csv", index=False)
    per_fold = (detail.groupby(["feature_set", "fold", "decision"])
                .agg(strict_accuracy=("strict_hit", "mean"), reach_predicted_side=("reach_predicted_side", "mean")).reset_index())
    per_fold.to_csv(out/"frontier_per_fold.csv", index=False)
    best = []
    for fs in dev.feature_set.unique():
        d = dev[dev.feature_set == fs]
        for r in (0.80, 0.85, 0.90, 0.95, 0.98):
            ok = d[d.reach_predicted_side >= r]
            top = ok.sort_values("strict_accuracy", ascending=False).head(1)
            best.append({"feature_set": fs, "reach_at_least": r,
                         "best_strict_accuracy": float(top.strict_accuracy.iloc[0]) if len(top) else None,
                         "decision": top.decision.iloc[0] if len(top) else "none reaches this level",
                         "reach_achieved": float(top.reach_predicted_side.iloc[0]) if len(top) else None})
        top = d.sort_values("strict_accuracy", ascending=False).head(1)
        best.append({"feature_set": fs, "reach_at_least": "unconstrained", "best_strict_accuracy": float(top.strict_accuracy.iloc[0]),
                     "decision": top.decision.iloc[0], "reach_achieved": float(top.reach_predicted_side.iloc[0])})
    best = pd.DataFrame(best)
    best.to_csv(out/"best_strict_accuracy_at_reach.csv", index=False)
    return dev, outer, info, best

def plot_frontier(dev, cfg, path=None):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
    for (fs, fam), d in dev.groupby(["feature_set", "family"]):
        d = d.sort_values("reach_predicted_side")
        style = {"quantile": "-o", "fixed": "--", "strict": "*", "fixed_train_best": "s"}.get(fam, "-")
        ax.plot(100*d.reach_predicted_side, 100*d.strict_accuracy, style, markersize=6 if fam != "strict" else 14,
                label=f"{fs} / {fam}")
    ax.plot([100*cfg.minimum_execution_reach], [100*cfg.minimum_accuracy], "r*", markersize=18, label="required (98% reach, 80% strict)")
    ax.set_xlabel("TP reach in predicted direction, all candles (%)"); ax.set_ylabel("Strict endpoint accuracy, all candles (%)")
    ax.set_title("Strict accuracy vs reach (development windows, pooled)"); ax.grid(alpha=.3); ax.legend(fontsize=7)
    if path:
        fig.savefig(path, dpi=130)
    plt.show()
    return fig

In [27]:
CROSS = None
if RUN_CROSS_ASSET_STUDY:
    CROSS_DIR = RUN_DIR/'cross_asset_study'; CROSS_DIR.mkdir(parents=True, exist_ok=True)
    cache = Path(OUTPUT_ROOT)/'cross_assets'
    start, end = PRIMARY.raw.time.iloc[0], PRIMARY.raw.time.iloc[-1]
    aligned, align_report = {}, {}
    for sym in CROSS_SYMBOLS:
        try:
            if sym in CROSS_ASSET_FILES:
                src = Path(CROSS_ASSET_FILES[sym])
                other = validate_raw_frame(pd.read_csv(src)) if src.suffix == '.csv' else None
                missing = []
            else:
                other, missing = load_binance_klines(sym, start, end, cache, base=CROSS_ASSET_BASE_URL)
            aligned[sym], rep = align_to_primary(other, PRIMARY.raw.time)
            align_report[sym] = {**rep, 'archives_missing': len(missing)}
        except Exception as exc:
            align_report[sym] = {'error': f'{type(exc).__name__}: {exc}'}
            print(f'{sym}: skipped ({exc})')
    display(pd.DataFrame(align_report).T)
    if not aligned:
        raise RuntimeError('No cross-asset data loaded. Check internet access in Colab or set CROSS_ASSET_FILES.')
    CROSS = cross_asset_features(PRIMARY.raw, PRIMARY.atr_pct, aligned)
    n = len(PRIMARY.raw)
    cuts = sorted({min(n-1, 2000), n//2, n-1})
    CROSS_AUDIT = audit_cross_features(PRIMARY.raw, PRIMARY.atr_pct, aligned, CROSS, cuts)
    write_json(CROSS_DIR/'cross_asset_data.json', {'symbols': align_report, 'features': list(CROSS.columns), 'audit': CROSS_AUDIT})
    print(f'{CROSS.shape[1]} cross-asset features from {len(aligned)} symbols; prefix-replay audit passed at {cuts}')

    if 'INFO_FOLDS' not in globals():
        INFO_WINDOWS = prepare_dev_windows(ASSETS, CFG, DEV_WINDOW_STARTS, EVAL_START)
        INFO_FOLDS = information_folds(INFO_WINDOWS, PREFLIGHT, EVAL_INDICES, CFG)
    X97 = PRIMARY.features.to_numpy(np.float32)
    FEATURE_SETS = {'btc_97': X97, 'btc_97_plus_cross': np.concatenate([X97, CROSS.to_numpy(np.float32)], axis=1)}
    FRONTIER_DEV, FRONTIER_OUTER, COMBINED_INFO, FRONTIER_BEST = combined_system_frontier(
        PRIMARY, INFO_FOLDS, CFG, FEATURE_SETS, INFO_GBDT_BINS, INFO_GBDT_ROUNDS, INFO_MAX_TRAIN_ROWS,
        REACH_SETTINGS['max_hold_bars'], REACH_SETTINGS['spread_bps'], FRONTIER_QUANTILES, FRONTIER_ATR_MULTIPLES, CROSS_DIR)
    pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
    print('\nDirection and ranking by feature set and fold:')
    display(COMBINED_INFO)
    print('\nKey decisions (development windows pooled):')
    key = FRONTIER_DEV[FRONTIER_DEV.family.isin(['strict', 'fixed_train_best']) |
                       FRONTIER_DEV.decision.isin(['quantile_0.10', 'quantile_0.25', 'quantile_0.40'])]
    display(key)
    print('\nBest strict accuracy subject to a reach floor (development windows pooled):')
    display(FRONTIER_BEST)
    plot_frontier(FRONTIER_DEV, CFG, CROSS_DIR/'frontier.png')
    dev_info = COMBINED_INFO[COMBINED_INFO.role == 'development_window']
    gain = dev_info.groupby('feature_set')[['direction_accuracy', 'spearman_distribution_median']].mean()
    strict_dec = FRONTIER_DEV[FRONTIER_DEV.family == 'strict'].set_index('feature_set')
    CROSS_SUMMARY = {
        'direction_accuracy_mean': gain.direction_accuracy.to_dict(),
        'spearman_mean': gain.spearman_distribution_median.to_dict(),
        'strict_decoder_accuracy': strict_dec.strict_accuracy.to_dict(),
        'strict_decoder_reach_predicted_side': strict_dec.reach_predicted_side.to_dict(),
        'best_strict_at_98pct_reach': FRONTIER_BEST[FRONTIER_BEST.reach_at_least == 0.98].set_index('feature_set').best_strict_accuracy.to_dict(),
        'requirement': {'strict': CFG.minimum_accuracy, 'reach': CFG.minimum_execution_reach},
    }
    best98 = [v for v in CROSS_SUMMARY['best_strict_at_98pct_reach'].values() if v is not None]
    CROSS_SUMMARY['verdict'] = ('requirement met on development windows' if best98 and max(best98) >= CFG.minimum_accuracy
                                else 'requirement not met: no decision reaches 80% strict at 98% reach')
    write_json(CROSS_DIR/'cross_asset_summary.json', CROSS_SUMMARY)
    INFO['cross_asset'] = CROSS_SUMMARY
    summary_path = RUN_DIR/'information_tests'/'information_summary.json'
    merged = json.loads(summary_path.read_text()) if summary_path.is_file() else {}
    merged.update(INFO)
    write_json(summary_path, merged)
    print('\nCross-asset effect (development windows):', json.dumps(json_safe(CROSS_SUMMARY), indent=2))

## Option A — Safe TP: a TP for every candle that price reaches

**Definition.** The model outputs only `tp_pct` for every candle. A prediction is **correct** when \(0 < \widehat{tp} \le tp\_pct\), meaning the TP is at or before the real ZigZag endpoint, so price reaches it in the label direction. A zero target is correct only with a zero TP (0.15% of candles). The goal is **98%** (`SAFE_TP_GOAL`); 90% and 95% are reported alongside.

**How accuracy is controlled.** Any model can hit 98% by predicting tiny TPs, so accuracy is set by **calibration** and models compete on **TP size at that accuracy**. Each model gives a conditional lower quantile of \(Z = tp/ATR\). One multiplicative shift is then fitted on the **most recent 10% of each training period** (a conformal calibration slice that is never used for fitting) so that the target share of calibration candles is safe. The shift is applied unchanged to the unseen window, and the realized accuracy is reported with leg-block confidence intervals.

**Models, with identical calibration:** a fixed ATR multiple, a fixed TP %, a gradient-boosted quantile regression on the 97 causal features, and (with `RUN_SAFE_TP_NEURAL`) the V3 network's distribution head read at the same quantile.

**Reported per model:**
- safe accuracy and its worst window;
- mean and median TP, and the share of TPs below the trading-cost floor;
- reach within 192 candles (spread included) in the predicted, ZigZag and hindsight **ORACLE** directions;
- the old strict-band accuracy for reference.

The efficiency chart sweeps the shift on the test windows to show each model's best possible TP size at every accuracy level. That shows the value of the model's ranking; the calibrated table is the honest result.

The outer fold (trained before 1 Jul 2026, scored on the inspected Jul–Oct period) is saved as a development artifact with `SafeTPPredictor`.

In [28]:
# Option A — Safe TP: definitions only.
# A TP is correct when 0 < TP <= actual endpoint tp_pct (price reaches it before the endpoint in the label
# direction); a zero target is correct only with a zero TP. Accuracy is set by calibration; models compete on TP size.

def safe_hits(actual, predicted):
    a, p = np.asarray(actual, float), np.asarray(predicted, float)
    tol = 8*np.finfo(float).eps*np.maximum(np.abs(a), np.abs(p))
    return np.isfinite(a) & np.isfinite(p) & (((p > 0) & (p <= a+tol)) | ((a == 0) & (p == 0)))

def safe_metrics(actual, tp, atr_pct, cost_floor_pct):
    a, tp = np.asarray(actual, float), np.asarray(tp, float)
    hits = safe_hits(a, tp)
    pos = a > 0
    return {"rows": int(len(a)), "safe_accuracy": float(hits.mean()),
            "mean_tp_pct": float(np.mean(tp)), "median_tp_pct": float(np.median(tp)),
            "mean_tp_atr": float(np.mean(tp/atr_pct)), "median_tp_atr": float(np.median(tp/atr_pct)),
            "median_capture_of_endpoint": float(np.median(tp[pos & hits]/a[pos & hits])) if (pos & hits).any() else None,
            "mean_captured_pct_all_candles": float(np.mean(np.where(hits, tp, 0.0))),
            "share_tp_below_cost_floor": float(np.mean(tp < cost_floor_pct)),
            "overshoot_rate": float(np.mean(tp > a))}

def three_way_split(asset, idx, cfg, es_fraction=0.05, cal_fraction=0.10, max_rows=None):
    """[core | embargo | early-stopping | embargo | calibration], chronological; labels in earlier parts resolve
    before the later part starts. Calibration = the most recent training data."""
    rest, cal = chronological_split(asset, idx, cfg, es_fraction=cal_fraction)
    core, es = chronological_split(asset, rest, cfg, es_fraction=es_fraction/(1-cal_fraction), max_rows=max_rows)
    return core, es, cal

def conformal_log_shift(z_cal, base_cal, target):
    """Multiplicative shift so that a fraction `target` of calibration targets lie at or above shift*base."""
    z, b = np.asarray(z_cal, float), np.asarray(base_cal, float)
    r = np.where(z > 0, np.log(np.where(z > 0, z, 1.0))-np.log(b), -np.inf)
    n = len(r)
    k = int(np.floor((1-target)*(n+1)))-1     # finite-sample conservative order statistic
    finite = r[np.isfinite(r)]
    value = float(np.sort(r)[max(k, 0)]) if k >= 0 else float(finite.min())
    return value if np.isfinite(value) else float(finite.min())

def efficiency_curve(z_test, base_test, atr_test, y_test, points=60):
    """Accuracy vs mean TP when the shift is swept ON THE TEST SET (diagnostic: the ranking value at equal accuracy)."""
    shifts = np.linspace(-6, 2, points*4)
    rows = []
    for s in shifts:
        tp = base_test*np.exp(s)*atr_test
        rows.append({"shift": s, "safe_accuracy": float(safe_hits(y_test, tp).mean()), "mean_tp_pct": float(tp.mean())})
    return pd.DataFrame(rows)

def tp_at_accuracy(curve, target):
    """Largest mean TP achieved with accuracy >= target on an efficiency curve."""
    ok = curve[curve.safe_accuracy >= target]
    return float(ok.mean_tp_pct.max()) if len(ok) else float("nan")

@torch.no_grad()
def distribution_quantiles(models, prepared, idx, cfg, device, levels):
    """Conditional quantiles of Z from the V3 distribution head (piecewise-linear log CDF)."""
    sym = cfg.primary_symbol
    loader = make_loader(prepared, [(sym, int(i)) for i in idx], cfg, train=False)
    out = {q: [] for q in levels}
    for x, cond, side, ids, target, _ in loader:
        x, cond, side, ids = x.to(device), cond.to(device), side.to(device), ids.to(device)
        joint, p0s = [], []
        for m in models:
            m.eval()
            with amp_context(device, cfg):
                h = m.encode(x, ids, side, cond)
            h = h.float()
            p0 = torch.sigmoid(m.zero_head(h).squeeze(-1).float()).double()
            joint.append((1-p0)[:, None]*torch.softmax(m.dist_head(h).double(), -1)); p0s.append(p0)
        p0 = torch.stack(p0s).mean(0)
        probs = torch.stack(joint).mean(0)/(1-p0)[:, None].clamp_min(1e-300)
        cdf = torch.cat([torch.zeros_like(probs[:, :1]), probs.cumsum(1)], 1)
        cdf = (cdf/cdf[:, -1:]).clamp(0, 1).cpu()
        edges = models[0].bin_edges.cpu().numpy()
        for q in levels:
            out[q].append(cdf_quantile(cdf, edges, q))
    return {q: np.concatenate(v) for q, v in out.items()}

def safe_tp_study(asset, folds, cfg, targets, rounds, max_rows, cost_floor_pct, horizon, spread_bps, output_dir,
                  neural_cfg=None, neural_folds=(), device=None, seed=7):
    """For every fold and accuracy target: fixed ATR, fixed TP%, GBDT quantile regression and (optionally) the
    V3 distribution network; each calibrated on the same most-recent training slice, then scored on the fold."""
    import lightgbm as lgb
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    X = asset.features.to_numpy(np.float32)
    z, y, atr, label = asset.z, asset.y, asset.atr_pct, asset.raw.label_side.to_numpy()
    fmax, fmin = future_extremes(asset, horizon)
    results, curves, saved = [], [], {}
    for f in folds:
        core, es, cal = three_way_split(asset, f["train"], cfg, max_rows=max_rows)
        test = f["test"]
        dc, de = core[label[core] != 0], es[label[es] != 0]
        clf = lgb_train(dict(objective="binary", metric="binary_logloss"), X[dc], (label[dc] > 0).astype(int),
                        X[de], (label[de] > 0).astype(int), rounds)
        side_pred = np.where(clf.predict(X[test], num_iteration=clf.best_iteration) >= .5, 1, -1)
        bases = {}  # model -> {target: (base_cal, base_test)} in Z units
        for t in targets:
            bases.setdefault("fixed_atr", {})[t] = (np.ones(len(cal)), np.ones(len(test)))
            bases.setdefault("fixed_tp_pct", {})[t] = (1/atr[cal], 1/atr[test])
            pc, pe = core[z[core] > 0], es[z[es] > 0]
            qm = lgb_train(dict(objective="quantile", alpha=1-t, metric="quantile"), X[pc], np.log(z[pc]),
                           X[pe], np.log(z[pe]), rounds)
            bases.setdefault("gbdt_quantile", {})[t] = (np.exp(qm.predict(X[cal], num_iteration=qm.best_iteration)),
                                                       np.exp(qm.predict(X[test], num_iteration=qm.best_iteration)))
            if f["role"] != "development_window":
                saved[t] = qm
        if neural_cfg is not None and f["fold"] in neural_folds:
            n_core, n_es, n_cal = three_way_split(asset, f["train"], cfg)   # full (unthinned) core for the network
            fit = replace(f["prepared"], train_idx={cfg.primary_symbol: n_core},
                          bins=LogZBins.fit(z[n_core], neural_cfg), domain=CandidateDomain.fit(z[n_core], neural_cfg))
            seed_all(seed)
            model = build_model(neural_cfg, fit).to(device)
            initialize_output_priors(model, fit, neural_cfg)
            state, info = train_run(model, fit, neural_cfg, device, neural_cfg.max_epochs, seed,
                                    out/"neural"/f["fold"], validation=n_es, tag=f"safe_tp/{f['fold']}")
            model.load_state_dict(state)
            levels = sorted({round(1-t, 4) for t in targets})
            qc = distribution_quantiles([model], fit, cal, neural_cfg, device, levels)
            qt = distribution_quantiles([model], fit, test, neural_cfg, device, levels)
            for t in targets:
                bases.setdefault("v3_network_quantile", {})[t] = (qc[round(1-t, 4)], qt[round(1-t, 4)])
            del model
        for name, per_t in bases.items():
            for t, (b_cal, b_test) in per_t.items():
                shift = conformal_log_shift(z[cal], b_cal, t)
                tp = b_test*np.exp(shift)*atr[test]
                m = safe_metrics(y[test], tp, atr[test], cost_floor_pct)
                hits = safe_hits(y[test], tp)
                m.update({"fold": f["fold"], "role": f["role"], "model": name, "target": t, "shift_log": shift,
                          "calibration_rows": int(len(cal)),
                          "reach_predicted_direction": float(np.nanmean(reach_vector(asset, test, tp, side_pred, fmax, fmin, spread_bps))),
                          "reach_zigzag_direction": float(np.nanmean(reach_vector(asset, test, tp, asset.side[test], fmax, fmin, spread_bps))),
                          "reach_ORACLE_label_direction": float(np.nanmean(reach_vector(asset, test, tp, label[test], fmax, fmin, spread_bps))),
                          "strict_75_100_accuracy": float(strict_hits(y[test], tp).mean())})
                lo, hi = block_accuracy_ci(hits, asset.raw.leg_id.to_numpy()[test])
                m["safe_accuracy_leg_ci95_low"], m["safe_accuracy_leg_ci95_high"] = lo, hi
                results.append(m)
                if t == max(targets):
                    c = efficiency_curve(z[test], b_test, atr[test], y[test]).assign(fold=f["fold"], role=f["role"], model=name)
                    curves.append(c)
        print(f"{f['fold']:>20}: " + " | ".join(
            f"{r['model']}@{r['target']:.2f} acc {r['safe_accuracy']:.3f} meanTP {r['mean_tp_pct']:.3f}%"
            for r in results if r["fold"] == f["fold"] and r["target"] == max(targets)))
    table = pd.DataFrame(results)
    table.to_csv(out/"safe_tp_results.csv", index=False)
    curves = pd.concat(curves, ignore_index=True)
    curves.to_csv(out/"efficiency_curves.csv", index=False)
    dev = table[table.role == "development_window"]
    pooled = (dev.assign(w=dev.rows).groupby(["model", "target"])
              .apply(lambda g: pd.Series({c: np.average(g[c], weights=g.w) for c in
                     ["safe_accuracy", "mean_tp_pct", "median_tp_pct", "mean_tp_atr", "mean_captured_pct_all_candles",
                      "share_tp_below_cost_floor", "reach_predicted_direction", "reach_zigzag_direction",
                      "reach_ORACLE_label_direction", "strict_75_100_accuracy"]} |
                     {"min_window_accuracy": g.safe_accuracy.min(), "windows": len(g)}), include_groups=False)
              .reset_index())
    base = pooled[pooled.model == "fixed_atr"].set_index("target").mean_tp_pct
    pooled["mean_tp_vs_fixed_atr_pct"] = 100*(pooled.mean_tp_pct/pooled.target.map(base)-1)
    pooled.to_csv(out/"safe_tp_pooled_development.csv", index=False)
    return table, pooled, curves, saved

def plot_efficiency(curves, target, path=None):
    import matplotlib.pyplot as plt
    dev = curves[curves.role == "development_window"]
    fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
    for name, g in dev.groupby("model"):
        c = g.groupby("shift")[["safe_accuracy", "mean_tp_pct"]].mean().sort_values("safe_accuracy")
        ax.plot(100*c.safe_accuracy, c.mean_tp_pct, label=name)
    ax.axvline(100*target, color="red", linestyle=":", label=f"goal {100*target:.0f}%")
    ax.set_xlim(80, 100); ax.set_xlabel("Safe-TP accuracy (%)"); ax.set_ylabel("Mean TP (% of price)")
    ax.set_title("TP size at each accuracy (development windows; shift swept on test = ranking value)")
    ax.grid(alpha=.3); ax.legend()
    if path:
        fig.savefig(path, dpi=130)
    plt.show()
    return fig

class SafeTPPredictor:
    """Development artifact: GBDT quantile model + conformal shift for one accuracy target, 97 causal features."""
    def __init__(self, directory, cfg):
        import lightgbm as lgb
        self.dir, self.cfg = Path(directory), cfg
        self.meta = json.loads((self.dir/"safe_tp_manifest.json").read_text())
        self.models = {float(t): lgb.Booster(model_file=str(self.dir/f"gbdt_quantile_{t}.txt")) for t in self.meta["targets"]}

    def predict_latest(self, raw_prefix, target):
        raw = validate_raw_frame(raw_prefix)
        if raw.time.iloc[0] != utc(self.meta["feature_origin"]):
            raise ValueError("Supply history from the saved feature origin so indicator/ZigZag state matches training.")
        features, atr_pct, side, _, _ = build_features(raw, self.cfg)
        if list(features.columns) != self.meta["feature_columns"]:
            raise ValueError("Feature schema differs from the saved artifact.")
        x = features.iloc[[-1]].to_numpy(np.float32)
        base = float(np.exp(self.models[float(target)].predict(x)[0]))
        z_tp = base*np.exp(self.meta["shifts"][str(float(target))])
        return {"time": str(raw.time.iloc[-1]), "target_accuracy": float(target), "tp_pct": float(z_tp*atr_pct[-1]),
                "tp_atr_multiple": float(z_tp), "atr_pct_points": float(atr_pct[-1]),
                "model": "gbdt_quantile + conformal shift", "fit_cutoff": self.meta["fit_cutoff"]}

In [ ]:
SAFE = None
if RUN_SAFE_TP_STUDY:
    SAFE_DIR = RUN_DIR/'safe_tp'
    windows = prepare_dev_windows(ASSETS, CFG, DEV_WINDOW_STARTS, EVAL_START)
    safe_folds = [{'fold': f'dev_{i+1}_{w.start[:10]}', 'role': 'development_window',
                   'train': np.asarray(w.fit.train_idx[CFG.primary_symbol]), 'test': np.asarray(w.val_idx), 'prepared': w.fit}
                  for i, w in enumerate(windows)]
    safe_folds.append({'fold': 'outer_jul_oct_2026', 'role': 'inspected_development_period',
                       'train': np.asarray(PREFLIGHT.train_idx[CFG.primary_symbol]), 'test': np.asarray(EVAL_INDICES), 'prepared': PREFLIGHT})
    neural_cfg = None
    if RUN_SAFE_TP_NEURAL:
        neural_cfg = replace(CFG, head_type='distribution', aux_lambda=0.10, magnitude_head=True,
                             max_epochs=SAFE_TP_NEURAL_EPOCHS, patience=3,
                             max_train_anchors_per_epoch=SAFE_TP_NEURAL_ANCHORS_PER_EPOCH).validate()
    cost_floor = (2*REACH_SETTINGS['fee_bps_per_side']+REACH_SETTINGS['spread_bps'])/100
    SAFE_TABLE, SAFE_POOLED, SAFE_CURVES, SAFE_MODELS = safe_tp_study(
        PRIMARY, safe_folds, CFG, SAFE_TP_TARGETS, INFO_GBDT_ROUNDS, INFO_MAX_TRAIN_ROWS, cost_floor,
        REACH_SETTINGS['max_hold_bars'], REACH_SETTINGS['spread_bps'], SAFE_DIR, neural_cfg=neural_cfg,
        neural_folds=[f['fold'] for f in safe_folds], device=DEVICE)
    pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
    show = ['model', 'target', 'safe_accuracy', 'min_window_accuracy', 'mean_tp_pct', 'median_tp_pct', 'mean_tp_atr',
            'mean_tp_vs_fixed_atr_pct', 'mean_captured_pct_all_candles', 'share_tp_below_cost_floor',
            'reach_predicted_direction', 'reach_ORACLE_label_direction', 'strict_75_100_accuracy']
    print(f'Development windows pooled (calibrated on each window\'s most recent training slice; cost floor {cost_floor:.2f}%):')
    display(SAFE_POOLED[show].sort_values(['target', 'mean_tp_pct'], ascending=[True, False]))
    print('Per window at the goal:')
    display(SAFE_TABLE[SAFE_TABLE.target == SAFE_TP_GOAL][['fold', 'model', 'safe_accuracy', 'safe_accuracy_leg_ci95_low',
                                                          'safe_accuracy_leg_ci95_high', 'mean_tp_pct', 'reach_predicted_direction']])
    plot_efficiency(SAFE_CURVES, SAFE_TP_GOAL, SAFE_DIR/'efficiency.png')
    dev_curves = SAFE_CURVES[SAFE_CURVES.role == 'development_window']
    EFFICIENCY = {m: tp_at_accuracy(g.groupby('shift')[['safe_accuracy', 'mean_tp_pct']].mean(), SAFE_TP_GOAL)
                  for m, g in dev_curves.groupby('model')}
    print(f'Best achievable mean TP at {SAFE_TP_GOAL:.0%} accuracy if calibration were perfect (ranking value):',
          {k: round(v, 4) for k, v in EFFICIENCY.items()})

    # Development artifact: GBDT quantile models trained before 1 Jul 2026 with their calibrated shifts.
    outer = SAFE_TABLE[(SAFE_TABLE.role != 'development_window') & (SAFE_TABLE.model == 'gbdt_quantile')]
    art = SAFE_DIR/'artifact'; art.mkdir(parents=True, exist_ok=True)
    for t, booster in SAFE_MODELS.items():
        booster.save_model(str(art/f'gbdt_quantile_{float(t)}.txt'), num_iteration=booster.best_iteration)
    write_json(art/'safe_tp_manifest.json', {
        'targets': [float(t) for t in SAFE_MODELS], 'shifts': {str(float(r.target)): float(r.shift_log) for r in outer.itertuples()},
        'feature_columns': list(PRIMARY.features.columns), 'feature_version': FEATURE_VERSION,
        'feature_origin': str(PRIMARY.raw.time.iloc[0]), 'fit_cutoff': str(utc(EVAL_START)),
        'definition': '0 < tp_hat <= actual tp_pct', 'evaluation_role': 'development (Jul-Oct 2026 already inspected)'})
    SAFE_PREDICTOR = SafeTPPredictor(art, CFG)
    LATEST_SAFE_TP = {t: SAFE_PREDICTOR.predict_latest(PRIMARY.raw[['time', *RAW_COLUMNS]], t) for t in SAFE_TP_TARGETS}
    print('Latest-candle safe TPs:', json.dumps(json_safe(LATEST_SAFE_TP), indent=2))

    g = SAFE_POOLED[(SAFE_POOLED.target == SAFE_TP_GOAL)].set_index('model')
    best_model = g.loc[g.min_window_accuracy >= SAFE_TP_GOAL-0.01, 'mean_tp_pct'].idxmax() if (g.min_window_accuracy >= SAFE_TP_GOAL-0.01).any() else None
    SAFE = {'goal': SAFE_TP_GOAL, 'pooled_at_goal': g.reset_index().to_dict('records'), 'efficiency_at_goal': EFFICIENCY,
            'best_model_meeting_goal_within_1pp_every_window': best_model,
            'verdict': (f'{best_model} meets the {SAFE_TP_GOAL:.0%} goal (within 1 pp in every window) with mean TP '
                        f"{g.loc[best_model, 'mean_tp_pct']:.4f}% ({g.loc[best_model, 'mean_tp_vs_fixed_atr_pct']:+.1f}% vs fixed ATR)")
                       if best_model else f'no model holds {SAFE_TP_GOAL:.0%} in every window: lower the goal or lengthen calibration'}
    write_json(SAFE_DIR/'safe_tp_summary.json', SAFE)
    summary_path = RUN_DIR/'information_tests'/'information_summary.json'
    merged = json.loads(summary_path.read_text()) if summary_path.is_file() else {}
    merged['safe_tp'] = {'verdict': SAFE['verdict']}
    write_json(summary_path, merged)
    print('\nSafe-TP verdict:', SAFE['verdict'])

[safe_tp/dev_1_2025-09-01] checkpoint in /content/drive/MyDrive/model_b_runs/model_b_v3_directional_multitask/safe_tp/neural/dev_1_2025-09-01 was made with a different config/data/scaler/bins; moved it to dev_1_2025-09-01__stale_20261104_183904 and training from scratch.


safe_tp/dev_1_2025-09-01 epoch 1/15:   0%|          | 0/1250 [00:00<?, ?it/s]

{"epoch": 1, "train_loss": 4.297857887458801, "train_primary_loss": 4.2772117559432985, "train_aux_loss": 0.2064613117814064, "seconds": 22.797030687332153, "learning_rate_scale": 0.9993037583856677, "grad_updates": 313, "grad_clip_fraction": 0.6261980830670927, "grad_nonfinite_updates": 0, "grad_preclip_norm_median": 1.4375940561294556, "grad_preclip_norm_p95": 4.569322299957276, "grad_aux_to_primary_grad_ratio_mean": 0.14304924696178156, "grad_primary_aux_grad_cosine_mean": 0.01762405689805746, "train_probe_strict_accuracy": 0.1474609375, "train_probe_primary_loss": 4.244820248684846, "train_probe_aux_loss": 0.18965608637348264, "train_probe_predicted_log_z_std": 0.0971109168022594, "train_probe_predicted_log_p1_p99_span": 0.3195854340252948, "train_probe_span_in_strict_bands": 1.11089798297689, "train_probe_spearman_predicted_vs_actual_z": 0.016041444843587282, "train_probe_spearman_aux_magnitude_vs_actual_z": 0.08693374886199834, "train_probe_zero_prediction_rate": 0.0, "train_prob

safe_tp/dev_1_2025-09-01 epoch 2/15:   0%|          | 0/1250 [00:00<?, ?it/s]

{"epoch": 2, "train_loss": 4.254662628936767, "train_primary_loss": 4.237235293006897, "train_aux_loss": 0.17427337619662284, "seconds": 22.557547330856323, "learning_rate_scale": 0.9829615496049519, "grad_updates": 313, "grad_clip_fraction": 0.0, "grad_nonfinite_updates": 0, "grad_preclip_norm_median": 0.37531158328056335, "grad_preclip_norm_p95": 0.6028759717941282, "grad_aux_to_primary_grad_ratio_mean": 0.3486266456172199, "grad_primary_aux_grad_cosine_mean": -0.015753302723169327, "train_probe_strict_accuracy": 0.13232421875, "train_probe_primary_loss": 4.243172419839539, "train_probe_aux_loss": 0.16839302385497731, "train_probe_predicted_log_z_std": 0.000982542359670198, "train_probe_predicted_log_p1_p99_span": 0.0, "train_probe_span_in_strict_bands": 0.0, "train_probe_spearman_predicted_vs_actual_z": -0.02073019329650365, "train_probe_spearman_aux_magnitude_vs_actual_z": 0.17600840102708964, "train_probe_zero_prediction_rate": 0.0, "train_probe_mean_selected_probability": 0.14182

safe_tp/dev_1_2025-09-01 epoch 3/15:   0%|          | 0/1250 [00:00<?, ?it/s]

... (12 outputs left out)

{"epoch": 9, "train_loss": 4.11519295501709, "train_primary_loss": 4.1022319974899295, "train_aux_loss": 0.12960957151949407, "seconds": 22.359790086746216, "learning_rate_scale": 0.4394345202930252, "grad_updates": 313, "grad_clip_fraction": 1.0, "grad_nonfinite_updates": 0, "grad_preclip_norm_median": 2.3181989192962646, "grad_preclip_norm_p95": 3.2555722713470456, "grad_aux_to_primary_grad_ratio_mean": 0.027206219249619912, "grad_primary_aux_grad_cosine_mean": 0.8840256929397583, "train_probe_strict_accuracy": 0.19189453125, "train_probe_primary_loss": 4.056453264784068, "train_probe_aux_loss": 0.11411466733365919, "train_probe_predicted_log_z_std": 0.4537066437072666, "train_probe_predicted_log_p1_p99_span": 1.1673908228730687, "train_probe_span_in_strict_bands": 4.0579199563043264, "train_probe_spearman_predicted_vs_actual_z": 0.5177965099503057, "train_probe_spearman_aux_magnitude_vs_actual_z": 0.5210722042576302, "train_probe_zero_prediction_rate": 0.0, "train_probe_mean_selecte

safe_tp/dev_2_2026-01-01 epoch 1/15:   0%|          | 0/1250 [00:00<?, ?it/s]

{"epoch": 1, "train_loss": 4.299397062301636, "train_primary_loss": 4.278085647773743, "train_aux_loss": 0.2131141651764512, "seconds": 22.325008392333984, "learning_rate_scale": 0.9993037583856677, "grad_updates": 313, "grad_clip_fraction": 0.6549520766773163, "grad_nonfinite_updates": 0, "grad_preclip_norm_median": 1.4441508054733276, "grad_preclip_norm_p95": 4.295006179809569, "grad_aux_to_primary_grad_ratio_mean": 0.1602789968209274, "grad_primary_aux_grad_cosine_mean": 0.016219557262957096, "train_probe_strict_accuracy": 0.140625, "train_probe_primary_loss": 4.259537234087475, "train_probe_aux_loss": 0.18163881428743883, "train_probe_predicted_log_z_std": 0.10896224624524058, "train_probe_predicted_log_p1_p99_span": 0.39083196147002974, "train_probe_span_in_strict_bands": 1.358555151313915, "train_probe_spearman_predicted_vs_actual_z": 0.033081359929830176, "train_probe_spearman_aux_magnitude_vs_actual_z": 0.11473216411141546, "train_probe_zero_prediction_rate": 0.0, "train_probe_

safe_tp/dev_2_2026-01-01 epoch 2/15:   0%|          | 0/1250 [00:00<?, ?it/s]

## V3.1–V3.4 — Staged selection on chronological development windows

Expensive: run in its own session. Re-running resumes unfinished windows and reuses completed candidates.

In [ ]:
SELECTED_CFG = SELECTED_PLAN = None
PLAN_FILE = RUN_DIR/'selection'/'v3_frozen_plan.json'
if FROZEN_PLAN_PATH:
    SELECTED_PLAN = json.loads(Path(FROZEN_PLAN_PATH).read_text())
    SELECTED_CFG = config_from_dict(SELECTED_PLAN['selected_config'])
    if CFG.run_mode == 'final':
        allowed = {'run_mode', 'protocol_frozen_at', 'resume'}
        frozen, current = json_safe(SELECTED_PLAN['selected_config']), json_safe(asdict(CFG))
        SELECTED_CFG = replace(SELECTED_CFG, run_mode='final', protocol_frozen_at=CFG.protocol_frozen_at)
        changed = [k for k in frozen if k not in allowed and k in current and current[k] != frozen[k]
                   and k in PREPARATION_FIELDS]
        if changed:
            raise ValueError(f'Final data preparation differs from the frozen plan: {changed}')
    print('Using frozen plan', SELECTED_PLAN['selected_key'])
elif RUN_V3_SELECTION:
    if any(utc(s) >= utc(EVAL_START) for s in DEV_WINDOW_STARTS):
        raise ValueError('Development windows must start before the outer evaluation period.')
    WINDOWS = prepare_dev_windows(ASSETS, CFG, DEV_WINDOW_STARTS, EVAL_START)
    display(pd.DataFrame([{'window': i+1, 'validation_start': w.start, 'validation_end': w.end,
                           'training_anchors': len(w.fit.train_idx[CFG.primary_symbol]), 'validation_anchors': len(w.val_idx),
                           'fixed_atr_multiple_train_fit': w.fixed_atr_multiple} for i, w in enumerate(WINDOWS)]))
    SELECTED_CFG, SELECTED_PLAN, SELECTION_RESULTS = run_staged_selection(
        ASSETS, CFG, SELECTION_BASE, SELECTION_STAGES, WINDOWS, RUN_DIR/'selection', DEVICE, REG_GAP_TRIGGER_PP)
    rows = []
    for entry in SELECTED_PLAN['selection_log']:
        if entry['skipped']:
            rows.append({'stage': entry['stage'], 'change_vs_selection_base': 'skipped', 'reason': entry['reason']}); continue
        for o in entry['options']:
            rows.append({'stage': entry['stage'], 'change_vs_selection_base': json.dumps(o['delta']),
                         'winner': o['key'] == entry['winner'], **{k: o[k] for k in o if k not in ('key', 'delta')}})
    SELECTION_TABLE = pd.DataFrame(rows)
    SELECTION_TABLE.to_csv(RUN_DIR/'selection'/'selection_table.csv', index=False)
    display(SELECTION_TABLE)
    print('Selected configuration change vs CFG:', json.dumps(json_safe(config_delta(SELECTED_CFG, CFG))))
    print('Refit epochs:', SELECTED_PLAN['refit_epochs'])
    first = RUN_DIR/'selection'/f"candidate_{SELECTED_PLAN['selected_key']}"
    histories = sorted(first.glob('window_*/history.json'))
    if histories:
        plot_training_history(histories[0], first/'window_1_history.png')
elif PLAN_FILE.is_file():
    SELECTED_PLAN = json.loads(PLAN_FILE.read_text())
    SELECTED_CFG = config_from_dict(SELECTED_PLAN['selected_config'])
    print('Loaded existing selection plan', SELECTED_PLAN['selected_key'], 'from', PLAN_FILE)
else:
    print('Selection not run. Set RUN_V3_SELECTION=True after reviewing the V3.0 diagnostics.')

## V3.5 — Refit the selected plan and evaluate every eligible candle

The evaluation period (Jul–Oct 2026) was inspected during V2, so in development mode these numbers are engineering evidence, not an untouched test. The comparison uses identical candles for V3, the training-fitted fixed ATR and fixed TP rules, the causal-pivot rule and the frozen V2 predictions. The side ablation flips or shuffles the live-side inputs to measure what direction contributes.

In [ ]:
RESULT = None
if RUN_V3_REFIT_EVALUATION:
    if SELECTED_PLAN is None:
        raise ValueError('Run the selection cell first, or set FROZEN_PLAN_PATH.')
    if SELECTED_CFG.run_mode == 'final':
        if utc(EVAL_START) < utc(SELECTED_CFG.protocol_frozen_at):
            raise ValueError('Final evaluation must start after the recorded protocol freeze.')
        if any(a.info['maturity_status'] != 'verified_by_supplied_provenance' for a in ASSETS.values()):
            raise ValueError('Final claims require audited label maturity provenance.')
    MAIN = RUN_DIR/'main'
    MODELS, PREPARED = refit_models(ASSETS, SELECTED_CFG, SELECTED_PLAN, EVAL_START, MAIN, DEVICE)
    PRED, METRICS, BASELINES = evaluate_models(MODELS, PREPARED, SELECTED_CFG, EVAL_START, EVAL_END, MAIN, DEVICE,
                                               SELECTED_PLAN, tag='V3')
    verified = all(a.info['maturity_status'] == 'verified_by_supplied_provenance' for a in ASSETS.values())
    BUNDLE_PATH = save_bundle(MAIN/'model_bundle', MODELS, PREPARED, SELECTED_CFG, SELECTED_PLAN, EVAL_START,
                              {'evaluation_role': METRICS['evaluation_role'], 'label_maturity_verified': verified})

    # ---- Same-candle comparison: V3 vs fixed ATR, fixed TP, causal-pivot rule and V2 ----
    IDX = rows_for_times(PRIMARY, PRED.time)
    atr = PRIMARY.atr_pct[IDX]
    pivot_rule = fit_pivot_rule(PRIMARY, PREPARED.train_idx[CFG.primary_symbol], SELECTED_CFG,
                                PIVOT_RULE_AGE_BINS, PIVOT_RULE_DISTANCE_BINS, PIVOT_RULE_MIN_BUCKET)
    pivot_mult, _, _ = apply_pivot_rule(pivot_rule, PRIMARY, IDX)
    fixed_atr = BASELINES['constant_atr_units']['fitted_constant']
    COMPARE_RULES = {'V3': PRED.predicted_tp_pct.to_numpy(float),
                     f'Fixed {fixed_atr:.4f} ATR': fixed_atr*atr,
                     f"Fixed TP {BASELINES['constant_raw_tp']['fitted_constant']:.4f}%": np.full(len(IDX), BASELINES['constant_raw_tp']['fitted_constant']),
                     'Confirmed-pivot ATR rule': pivot_mult*atr}
    v2_pred_path = V2_DIR/'main'/'predictions.csv'
    if v2_pred_path.is_file():
        v2p = pd.read_csv(v2_pred_path); v2p['time'] = pd.to_datetime(v2p.time, utc=True)
        merged = PRED[['time']].merge(v2p[['time', 'predicted_tp_pct']], on='time', how='left')
        if merged.predicted_tp_pct.notna().all():
            COMPARE_RULES['V2 (saved, frozen)'] = merged.predicted_tp_pct.to_numpy(float)
        else:
            print(f'V2 predictions cover {merged.predicted_tp_pct.notna().mean():.1%} of V3 candles; V2 omitted from the same-candle table.')
    actual = PRIMARY.y[IDX]
    fixed_key = f'Fixed {fixed_atr:.4f} ATR'
    CMP_HITS = {k: strict_hits(actual, v, SELECTED_CFG.lower_ratio) for k, v in COMPARE_RULES.items()}
    lo_hi = {k: accuracy_block_ci(PRED.time, h, SELECTED_CFG) for k, h in CMP_HITS.items()}
    COMPARISON = pd.DataFrame([{'rule': k, **hit_breakdown(actual, v, SELECTED_CFG.lower_ratio),
                                'ci95_low': lo_hi[k][0], 'ci95_high': lo_hi[k][1],
                                'hits_vs_fixed_atr': int(CMP_HITS[k].sum()-CMP_HITS[fixed_key].sum())}
                               for k, v in COMPARE_RULES.items()])
    legs = PRIMARY.raw.leg_id.to_numpy()[IDX]
    PAIRED = pd.DataFrame([paired_report(k, CMP_HITS[k], fixed_key, CMP_HITS[fixed_key], PRED.time, SELECTED_CFG, legs=legs)
                           for k in COMPARE_RULES if k != fixed_key])
    COMPARISON.to_csv(MAIN/'same_candle_comparison.csv', index=False)
    PAIRED.to_csv(MAIN/'paired_vs_fixed_atr.csv', index=False)
    write_json(MAIN/'pivot_rule.json', pivot_rule)
    pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
    display(pd.DataFrame([{k: v for k, v in METRICS.items() if not isinstance(v, dict)}]).T.rename(columns={0: 'value'}))
    display(COMPARISON)
    display(PAIRED[['model', 'only_model_hit', 'only_reference_hit', 'difference_pp', 'ci95_low_pp', 'ci95_high_pp',
                    'leg_block_ci95_low_pp', 'leg_block_ci95_high_pp', 'mcnemar_exact_p_iid']])
    print('Prediction spread:', {k: METRICS['prediction_spread'][k] for k in ('verdict', 'predicted_log_iqr', 'spearman_predicted_vs_actual_z')})
    if 'side_ablation' in METRICS:
        print('Side ablation:', json.dumps(json_safe(METRICS['side_ablation'])))
    for key in ('live_side', 'live_side_matches_label', 'tp_size', 'volatility_regime', 'causal_leg_age'):
        f = MAIN/f'metrics_by_{key}.csv'
        if f.is_file():
            display(pd.read_csv(f)[[key, 'eligible_rows', 'strict_accuracy', 'predicted_log_z_std', 'spearman_predicted_vs_actual_z']])
    plot_results(PRED, MAIN, SELECTED_CFG.minimum_accuracy)
    RESULT = {'metrics': METRICS, 'predictions': PRED, 'bundle_path': BUNDLE_PATH, 'models': MODELS,
              'prepared': PREPARED, 'comparison': COMPARISON, 'paired': PAIRED, 'cfg': SELECTED_CFG}

## V3.5 — Execution TP reach

Edit `REACH_SETTINGS` to match the real strategy (entry, stop, horizon, costs) **before** looking at results. The 98% gate refers to the execution-adjusted, live-side row on all eligible candles.

In [ ]:
REACH = None
if RUN_REACH_SIMULATION:
    SPEC = ReachSpec(**REACH_SETTINGS)
    sources = {}
    if RESULT is not None:
        idx = rows_for_times(PRIMARY, RESULT['predictions'].time)
        sources['V3'] = (idx, RESULT['predictions'].predicted_tp_pct.to_numpy(float))
        fixed_atr = BASELINES['constant_atr_units']['fitted_constant']
        sources[f'Fixed {fixed_atr:.4f} ATR'] = (idx, fixed_atr*PRIMARY.atr_pct[idx])
    if (V2_DIR/'main'/'predictions.csv').is_file():
        v2p = load_run_predictions(V2_DIR/'main')
        idx2 = rows_for_times(PRIMARY, v2p.time)
        sources['V2 (saved)'] = (idx2, v2p.predicted_tp_pct.to_numpy(float))
        if RESULT is None:
            fa = best_constant(PRIMARY.z[eligible_indices(PRIMARY, CFG, fit_cutoff=EVAL_START)], CFG.lower_ratio)
            sources[f'Fixed {fa:.4f} ATR'] = (idx2, fa*PRIMARY.atr_pct[idx2])
    rows, sims = [], {}
    for name, (idx, tp) in sources.items():
        live = PRIMARY.side[idx]
        sims[name] = simulate_reach(PRIMARY, idx, tp, live, SPEC)
        rows.append({**reach_summary(sims[name], name), 'mode': 'execution-adjusted, live side'})
        rows.append({**reach_summary(simulate_reach(PRIMARY, idx, tp, live, SPEC, nominal=True), name), 'mode': 'nominal touch, live side'})
        hindsight = PRIMARY.raw.label_side.to_numpy()[idx]
        rows.append({**reach_summary(simulate_reach(PRIMARY, idx, tp, hindsight, SPEC, nominal=True), name),
                     'mode': 'DIAGNOSTIC ONLY: nominal touch with hindsight label side'})
    if rows:
        REACH = pd.DataFrame(rows)
        REACH_DIR = RUN_DIR/'reach'; REACH_DIR.mkdir(parents=True, exist_ok=True)
        REACH.to_csv(REACH_DIR/'reach_summary.csv', index=False)
        for name, sim in sims.items():
            sim.to_csv(REACH_DIR/f"reach_rows_{name.replace(' ', '_').replace('(', '').replace(')', '')}.csv", index=False)
        write_json(REACH_DIR/'reach_spec.json', asdict(SPEC))
        display(REACH[['rule', 'mode', 'eligible_candles_scored', 'orders_issued', 'reach_all_candles', 'reach_issued_orders',
                       'stopped_rate_orders', 'ambiguous_same_bar', 'mean_net_return_bps_per_order', 'pending_future_data']])
        if 'V3' in sims:
            v3_exec = REACH[(REACH.rule == 'V3') & (REACH['mode'] == 'execution-adjusted, live side')].iloc[0]
            print(f"V3 execution-adjusted reach on all eligible candles: {v3_exec.reach_all_candles:.4f} "
                  f"(requirement {CFG.minimum_execution_reach:.2f}); evaluation role: {RESULT['metrics']['evaluation_role']}")
    else:
        print('No predictions available for reach simulation yet.')

## Predict from the saved model and replay complete prefixes

This predicts the final completed candle of the supplied CSV, then replays three evaluation anchors from their raw prefixes on CPU. For new candles, keep all raw history from the saved feature origin and append completed contiguous bars.

In [ ]:
BUNDLE_PATH = RESULT['bundle_path'] if RESULT is not None else None
# To use a previous artifact: BUNDLE_PATH = Path('/content/extracted_model_bundle')
if BUNDLE_PATH is not None:
    PREDICTOR = EndpointPredictor(BUNDLE_PATH, device=DEVICE)
    columns = ['time', *RAW_COLUMNS] + ([PREDICTOR.cfg.side_column] if PREDICTOR.cfg.direction_source == 'column' else [])
    LATEST_PREDICTION = PREDICTOR.predict_latest(PRIMARY.raw[columns].copy(), CFG.primary_symbol)
    print(json.dumps(json_safe(LATEST_PREDICTION), indent=2))
    write_json(RUN_DIR/'latest_prediction.json', LATEST_PREDICTION)
    if RUN_CAUSALITY_AUDITS and RESULT is not None:
        idx = rows_for_times(PRIMARY, RESULT['predictions'].time)
        replay_indices = idx[np.unique(np.linspace(0, len(idx)-1, min(3, len(idx))).astype(int))]
        REPLAY = replay_audit(BUNDLE_PATH, PRIMARY, RESULT['cfg'], replay_indices, device='cpu')
        REPLAY.to_csv(RUN_DIR/'saved_model_prefix_replay.csv', index=False)
        print('Saved-model prefix replay passed for', len(REPLAY), 'anchors (features, side, ZigZag state and decision).')

## Export

In [ ]:
if RESULT is not None and EXPORT_MODEL_ZIP:
    import shutil
    MODEL_ZIP = Path(shutil.make_archive(str(RESULT['bundle_path']), 'zip', RESULT['bundle_path']))
    print('Inference ZIP:', MODEL_ZIP)
    if DOWNLOAD_MODEL_ZIP:
        try:
            from google.colab import files
            files.download(str(MODEL_ZIP))
        except ImportError:
            print('Download the ZIP using your Jupyter file browser.')
print('Run directory:', RUN_DIR)
for name in ('v3_0_diagnostics', 'selection/v3_frozen_plan.json', 'selection/selection_table.csv', 'main/predictions.csv',
             'main/metrics.json', 'main/same_candle_comparison.csv', 'reach/reach_summary.csv'):
    path = RUN_DIR/name
    print(('  [x] ' if path.exists() else '  [ ] ')+str(path))

## Milestone gates (computed from saved artifacts)

| Gate | Evidence used |
|---|---|
| M1 Small-batch learning works and gradients are healthy | V3 memorization ≥ 90%; no starved encoder or nonfinite gradients |
| M2 Predictions respond to market state on unseen candles | Development spread verdict `varies` and Spearman > 0 (or validation spread in every window) |
| M3 Strict accuracy beats fixed ATR across chronological windows | Every development window beats its training-fitted fixed ATR |
| M4 Head and generalization choices survive validation | Head and sampler stages were evaluated, and the lift on the later period has a positive week- and leg-block CI |
| M5 Frozen full-coverage evaluation meets the contract | Final mode, verified maturity, ≥ 80% strict accuracy at 100% coverage, ≥ 98% execution reach |

In [ ]:
def read_json_if(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.is_file() else None

def milestone_report(run_dir, cfg):
    """Evidence-based gates M1-M5, computed only from saved artifacts. 'not_run' is never treated as a pass."""
    run_dir = Path(run_dir)
    diag = run_dir/'v3_0_diagnostics'
    mem, grad = read_json_if(diag/'diagnostic_C_memorization.json'), read_json_if(diag/'diagnostic_B_gradient_flow.json')
    plan = read_json_if(run_dir/'selection'/'v3_frozen_plan.json')
    metrics = read_json_if(run_dir/'main'/'metrics.json')
    reach_csv = run_dir/'reach'/'reach_summary.csv'
    gates = []
    # M1
    if mem and grad and 'v3_config' in mem and 'v3_config_from_init' in grad:
        g = grad['v3_config_from_init']
        healthy = not any(n.startswith(('ENCODER STARVED', 'Nonfinite')) for n in g['interpretation'])
        ok = mem['v3_config']['passed'] and healthy
        gates.append(('M1', 'Small-batch learning works and gradients are healthy', 'passed' if ok else 'failed',
                      f"memorization {mem['v3_config']['final_strict_accuracy']:.3f} (needs {mem['v3_config']['pass_threshold']}); "
                      f"encoder/head grad ratio {g.get('encoder_to_head_ratio')}; clip fraction {g['clip_fraction']:.2f}"))
    else:
        gates.append(('M1', 'Small-batch learning works and gradients are healthy', 'not_run', 'run V3.0 B and C'))
    # M2
    windows = plan['selected_summary']['windows'] if plan else None
    if metrics is not None:
        sp = metrics['prediction_spread']
        ok = sp['verdict'] == 'varies' and (sp['spearman_predicted_vs_actual_z'] or 0) > 0
        gates.append(('M2', 'Predictions respond to market state on unseen candles', 'passed' if ok else 'failed',
                      f"development evaluation spread verdict {sp['verdict']}, log IQR {sp['predicted_log_iqr']:.4f}, "
                      f"Spearman {sp['spearman_predicted_vs_actual_z']:.3f}"))
    elif windows:
        ok = all((w['val_predicted_log_z_std'] or 0) >= STRICT_BAND_LOG/4 and (w['val_spearman'] or 0) > 0 for w in windows)
        gates.append(('M2', 'Predictions respond to market state on unseen candles', 'passed' if ok else 'failed',
                      'validation log-Z std per window: '+', '.join(f"{w['val_predicted_log_z_std']}" for w in windows)))
    else:
        gates.append(('M2', 'Predictions respond to market state on unseen candles', 'not_run', 'run selection or refit'))
    # M3
    if plan:
        s = plan['selected_summary']
        n = len(s['windows'])
        ok = s['windows_beating_fixed_atr'] == n and s['pooled_strict_accuracy'] > s['pooled_fixed_atr_accuracy']
        gates.append(('M3', 'Strict accuracy beats fixed ATR across chronological windows', 'passed' if ok else 'failed',
                      f"{s['windows_beating_fixed_atr']}/{n} windows beat fixed ATR; pooled {s['pooled_strict_accuracy']:.4f} vs "
                      f"{s['pooled_fixed_atr_accuracy']:.4f}; min lift {s['min_lift_pp']:+.2f} pp"))
    else:
        gates.append(('M3', 'Strict accuracy beats fixed ATR across chronological windows', 'not_run', 'run selection'))
    # M4
    if plan and metrics is not None:
        stages = {e['stage'] for e in plan['selection_log'] if not e['skipped']}
        paired = metrics['paired_vs_fixed_atr']
        ok = {'V3.3_head', 'V3.4_sampler'} <= stages and paired['ci95_low_pp'] > 0 and paired.get('leg_block_ci95_low_pp', -1) > 0
        gates.append(('M4', 'Head and generalization choices survive later validation', 'passed' if ok else 'failed',
                      f"stages evaluated {sorted(stages)}; development-period lift {paired['difference_pp']:+.2f} pp "
                      f"(week CI {paired['ci95_low_pp']:+.2f} to {paired['ci95_high_pp']:+.2f}; "
                      f"leg CI {paired.get('leg_block_ci95_low_pp', float('nan')):+.2f} to {paired.get('leg_block_ci95_high_pp', float('nan')):+.2f})"))
    else:
        gates.append(('M4', 'Head and generalization choices survive later validation', 'not_run', 'run selection and refit evaluation'))
    # M5
    if metrics is not None and reach_csv.is_file():
        r = pd.read_csv(reach_csv)
        v3 = r[(r.rule == 'V3') & (r['mode'] == 'execution-adjusted, live side')]
        reach = float(v3.reach_all_candles.iloc[0]) if len(v3) else float('nan')
        final = metrics['evaluation_role'] == 'final' and metrics['label_maturity_verified']
        acc_ok = metrics['coverage'] == 1 and (metrics['strict_accuracy'] or 0) >= cfg.minimum_accuracy
        reach_ok = reach >= cfg.minimum_execution_reach
        status = 'passed' if (final and acc_ok and reach_ok) else ('failed' if final else 'development_only')
        gates.append(('M5', 'Frozen full-coverage evaluation meets strict accuracy and reach', status,
                      f"role {metrics['evaluation_role']}; strict accuracy {metrics['strict_accuracy']:.4f} (needs {cfg.minimum_accuracy}); "
                      f"coverage {metrics['coverage']:.3f}; execution reach {reach:.4f} (needs {cfg.minimum_execution_reach})"))
    else:
        gates.append(('M5', 'Frozen full-coverage evaluation meets strict accuracy and reach', 'not_run', 'needs refit evaluation and reach'))
    table = pd.DataFrame(gates, columns=['milestone', 'gate', 'status', 'evidence'])
    advice = None
    status = dict(zip(table.milestone, table.status))
    if status.get('M2') == 'passed' and status.get('M3') == 'failed':
        advice = ('SAFEGUARD: the network now varies its predictions but does not beat fixed ATR on unseen windows. '
                  'Investigate the information available at prediction time (live side quality, label definition, '
                  'pivot-state reconstruction) before adding neural complexity.')
    elif status.get('M1') == 'failed':
        advice = 'Fix the learning system (loss, gradients, target representation) before any large training run.'
    return table, advice

INFO_FILE = RUN_DIR/'information_tests'/'information_summary.json'
if INFO_FILE.is_file():
    info = json.loads(INFO_FILE.read_text())
    print('Information tests:', {k: v.get('verdict') for k, v in info.items() if isinstance(v, dict)})
GATES, GATE_ADVICE = milestone_report(RUN_DIR, CFG)
GATES.to_csv(RUN_DIR/'milestone_gates.csv', index=False)
done = int((GATES.status == 'passed').sum())
print(f'{done} of 5 milestones passed')
display(GATES)
if GATE_ADVICE:
    print(GATE_ADVICE)

## Label finality and the later final test

The original ZigZag generator is not available here, so `tp_confirm_time + 15 minutes` remains a development proxy and the V3 ZigZag state is a causal reconstruction.

1. Generate `tp_pct`, the endpoint time and the first instant each label is permanently fixed (`label_available_time`) with the original generator; audit it with `audit_label_prefixes(raw, original_generator, cut_positions)`, including reversal, zero and boundary cases.
2. If the generator's live state can be reproduced, compare its confirmed pivots with `causal_zigzag_state` and set `zigzag_depth` / `zigzag_min_deviation_atr` to match. Re-run the feature audits.
3. Record the rule/version in `DATA_METADATA`; only then set `maturity_verified=True`.
4. Freeze `selection/v3_frozen_plan.json`, the source hash and `REACH_SETTINGS`. Record the freeze instant in `protocol_frozen_at`, set `run_mode='final'` and `FROZEN_PLAN_PATH`, and evaluate a later untouched period once every label has matured.
5. Do not replace a disappointing final result with a re-tuned model and still call that period untouched.

**Not established by this notebook:** V3 training results, 80% strict accuracy, 98% reach, original label finality, or that the causal ZigZag matches the original generator.